In [ ]:
import os, json, math, shutil, time
from pathlib import Path
import numpy as np
import pandas as pd

# -----------------------------
# 0. CONFIG
# -----------------------------
FULL_RUN = True
SEED = 42

N_CANDIDATES = 5000 if FULL_RUN else 300
N_EXAMS = 20
ITEMS_PER_EXAM = 50
N_ITEMS = N_EXAMS * ITEMS_PER_EXAM
N_OPTIONS = 4
N_REVIEWERS = 12
DOUBLE_REVIEW_RATE = 0.10
DATA_DIR = Path.cwd() / "integrityinsight_data"

if DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)
for d in ["raw", "staging", "rejected", "ground_truth", "validation"]:
    (DATA_DIR / d).mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(SEED)

# -----------------------------
# 1. CONTRACTS
# -----------------------------
ALLOWED_OPTIONS = set(list("ABCD"))
ALLOWED_RECOMMENDATIONS = {
    "NO_ACTION", "ROUTINE_REVIEW", "HUMAN_REVIEW", "PRIORITY_REVIEW"
}
ALLOWED_VERDICTS = {
    "CONFIRMED", "OVERTURNED", "INCONCLUSIVE",
    "TECHNICAL_ISSUE", "ESCALATED"
}
FLAG_CONFIG = {
    "TAB_SWITCH":        {"base_rate": 0.12, "precision": 0.95},
    "SECOND_VOICE":      {"base_rate": 0.04, "precision": 0.85},
    "IDENTITY_MISMATCH": {"base_rate": 0.01, "precision": 0.90},
    "PHONE_DETECTED":    {"base_rate": 0.07, "precision": 0.55},
    "GAZE_OFF_SCREEN":   {"base_rate": 0.22, "precision": 0.25},
    "MULTI_PERSON":       {"base_rate": 0.03, "precision": 0.75},
    "MULTI_MONITOR":     {"base_rate": 0.08, "precision": 0.92},
    "REMOTE_TOOL":       {"base_rate": 0.005, "precision": 0.97},
}

PROFILES = [
    "clean", "colluding_pair", "colluding_ring", "item_leak_group",
    "answer_key_holder", "low_effort", "test_anxiety"
]

# -----------------------------
# 2. CANDIDATES + ITEMS
# -----------------------------
candidate_ids = np.array([f"cand_{i:05d}" for i in range(1, N_CANDIDATES + 1)])
exam_ids = np.array([f"exam_{i:03d}" for i in range(1, N_EXAMS + 1)])
item_ids = np.array([
    f"exam_{e:03d}_item_{j:03d}"
    for e in range(1, N_EXAMS + 1)
    for j in range(1, ITEMS_PER_EXAM + 1)
])

theta = rng.normal(0, 1, N_CANDIDATES)

candidates = pd.DataFrame({
    "candidate_id": candidate_ids,
    "ability_theta": np.round(theta, 4),
})

items = pd.DataFrame({
    "item_id": item_ids,
    "exam_id": np.repeat(exam_ids, ITEMS_PER_EXAM),
    "item_number": np.tile(np.arange(1, ITEMS_PER_EXAM + 1), N_EXAMS),
    "difficulty_b": np.round(rng.normal(0, 1, N_ITEMS), 4),
    "discrimination_a": np.round(rng.uniform(0.6, 2.0, N_ITEMS), 4),
    "guessing_c": np.round(rng.uniform(0.15, 0.25, N_ITEMS), 4),
})
# Stable correct answer per item, used by the simulator.
items["correct_option"] = rng.choice(np.array(list("ABCD")), N_ITEMS)

# -----------------------------
# 3. ASSIGN 7 PROFILES
# -----------------------------
# Keep the original project's approximate distribution while making every
# non-clean profile operational.
profile_probs = np.array([
    0.8862, 0.0298, 0.0260, 0.0210, 0.0102, 0.0082, 0.0186
])
profile_probs = profile_probs / profile_probs.sum()

profiles = rng.choice(PROFILES, size=N_CANDIDATES, p=profile_probs)
candidate_profile_df = pd.DataFrame({
    "candidate_id": candidate_ids,
    "profile": profiles
})

# -----------------------------
# 4. GROUND TRUTH STRUCTURES
# -----------------------------
# Ground truth is kept out of analytical tables.
gt_rows = []
pair_groups = []
ring_groups = []
leak_groups = []

def unassigned_candidates(profile):
    return set(candidate_profile_df.loc[
        candidate_profile_df["profile"].eq(profile), "candidate_id"
    ])

# Colluding pairs: exactly 2 candidates per pair.
pair_candidates = list(unassigned_candidates("colluding_pair"))
rng.shuffle(pair_candidates)
for i in range(0, len(pair_candidates) - 1, 2):
    a, b = pair_candidates[i:i+2]
    copy_rate = float(rng.uniform(0.30, 0.60))
    pair_groups.append((a, b, copy_rate))

# Colluding rings: 3–6 candidates with one shared source.
ring_candidates = list(unassigned_candidates("colluding_ring"))
rng.shuffle(ring_candidates)
pos = 0
while pos + 3 <= len(ring_candidates):
    remaining = len(ring_candidates) - pos
    if remaining < 3:
        break
    size = int(rng.integers(3, min(6, remaining) + 1))
    group = ring_candidates[pos:pos+size]
    source = group[0]
    copy_rate = float(rng.uniform(0.35, 0.65))
    ring_groups.append((group, source, copy_rate))
    pos += size

# Item leak groups: choose 8–15 items per group, by exam.
leak_candidates = list(unassigned_candidates("item_leak_group"))
rng.shuffle(leak_candidates)
for cand in leak_candidates:
    n_leaks = int(rng.integers(8, 16))
    leaked_items = rng.choice(item_ids, size=n_leaks, replace=False).tolist()
    leak_groups.append((cand, leaked_items))

pair_map = {}
for a, b, rate in pair_groups:
    pair_map[a] = {"partners": [b], "copy_rate": rate}
    pair_map[b] = {"partners": [a], "copy_rate": rate}

ring_map = {}
for group, source, rate in ring_groups:
    for cand in group:
        ring_map[cand] = {
            "partners": [x for x in group if x != cand],
            "source": source,
            "copy_rate": rate,
        }

leak_map = {cand: leaked for cand, leaked in leak_groups}

for cand, profile in zip(candidate_ids, profiles):
    gt_rows.append({
        "candidate_id": cand,
        "profile": profile,
        "violation": int(profile in {
            "colluding_pair", "colluding_ring",
            "item_leak_group", "answer_key_holder"
        }),
        "partner_ids": json.dumps(
            pair_map.get(cand, ring_map.get(cand, {})).get("partners", [])
            if cand in pair_map or cand in ring_map else []
        ),
        "leaked_items": json.dumps(leak_map.get(cand, [])),
        "copy_rate": (
            pair_map.get(cand, ring_map.get(cand, {})).get("copy_rate", 0.0)
            if cand in pair_map or cand in ring_map else 0.0
        ),
    })

ground_truth_candidates = pd.DataFrame(gt_rows)

# -----------------------------
# 5. EXAMS + SESSIONS
# -----------------------------
sessions = pd.DataFrame({
    "session_id": [
        f"sess_{i:06d}"
        for i in range(1, N_CANDIDATES * N_EXAMS + 1)
    ],
    "candidate_id": np.repeat(candidate_ids, N_EXAMS),
    "exam_id": np.tile(exam_ids, N_CANDIDATES),
})
sessions["session_number"] = sessions.groupby("candidate_id").cumcount() + 1

candidate_index = pd.Series(np.arange(N_CANDIDATES), index=candidate_ids)
session_candidate_idx = sessions["candidate_id"].map(candidate_index).to_numpy()
profile_by_candidate = pd.Series(profiles, index=candidate_ids)
session_profiles = sessions["candidate_id"].map(profile_by_candidate).to_numpy()

# -----------------------------
# 6. RESPONSE SIMULATOR — IRT 3PL
# -----------------------------
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))

def generate_responses(candidates_df, items_df, sessions_df, rng):
    n_c = len(candidates_df)
    n_i = len(items_df)
    total = n_c * n_i

    cand_idx = np.repeat(np.arange(n_c), n_i)
    item_idx = np.tile(np.arange(n_i), n_c)

    theta_v = candidates_df["ability_theta"].to_numpy()[cand_idx]
    a = items_df["discrimination_a"].to_numpy()[item_idx]
    b = items_df["difficulty_b"].to_numpy()[item_idx]
    c = items_df["guessing_c"].to_numpy()[item_idx]

    p_correct = c + (1 - c) * sigmoid(a * (theta_v - b))

    profile_v = candidates_df["candidate_id"].map(
        profile_by_candidate
    ).to_numpy()[cand_idx]

    # Confounds and injected behavior.
    low_effort = profile_v == "low_effort"
    anxiety = profile_v == "test_anxiety"
    key_holder = profile_v == "answer_key_holder"

    p_correct[low_effort] = 0.25
    p_correct[anxiety] *= rng.uniform(0.65, 0.95, anxiety.sum())
    p_correct[key_holder] = np.maximum(
        p_correct[key_holder] + 0.45, 0.97
    )

    is_correct = rng.random(total) < np.clip(p_correct, 0.01, 0.995)

    correct_options = items_df["correct_option"].to_numpy()[item_idx]
    selected = correct_options.copy()

    wrong_mask = ~is_correct
    wrong_idx = np.where(wrong_mask)[0]
    if len(wrong_idx):
        # Plausible, non-uniform distractor choice.
        for idx in wrong_idx:
            correct = correct_options[idx]
            wrong = np.array([x for x in "ABCD" if x != correct])
            selected[idx] = rng.choice(wrong, p=[0.50, 0.30, 0.20])

    # Response time: ability + difficulty + noise.
    base_time = 8000.0
    difficulty_effect = 3000.0 * np.maximum(b, -1)
    ability_effect = -1200.0 * theta_v
    noise = rng.lognormal(mean=0.0, sigma=0.35, size=total)
    rt = (base_time + difficulty_effect + ability_effect) * noise
    rt[low_effort] *= 0.35
    rt[anxiety] *= 1.60

    # Item-leak behavior: correct + implausibly fast on leaked items only.
    candidate_v = candidates_df["candidate_id"].to_numpy()[cand_idx]
    item_v = items_df["item_id"].to_numpy()[item_idx]
    leak_mask = np.zeros(total, dtype=bool)

    for cand, leaked in leak_map.items():
        m = (candidate_v == cand) & np.isin(item_v, leaked)
        leak_mask |= m
        selected[m] = correct_options[m]
        is_correct[m] = True
        rt[m] *= rng.uniform(0.20, 0.45, m.sum())

    rt = np.maximum(rt, 1000)

    out = pd.DataFrame({
        "response_id": [f"resp_{i:09d}" for i in range(1, total + 1)],
        "session_id": sessions_df["session_id"].to_numpy()[np.repeat(
            np.arange(len(sessions_df)), ITEMS_PER_EXAM
        )],
        "candidate_id": candidate_v,
        "exam_id": items_df["exam_id"].to_numpy()[item_idx],
        "item_id": item_v,
        "selected_option": selected,
        "is_correct": is_correct.astype(np.int8),
        "response_time_ms": np.round(rt, 2),
        "sequence_position": np.tile(np.arange(1, ITEMS_PER_EXAM + 1), n_c * N_EXAMS),
        "revisions": rng.poisson(0.15, total).astype(np.int8),
    })

    return out

responses = generate_responses(candidates, items, sessions, rng)

# -----------------------------
# 7. INJECT COLLUDING PAIRS + RINGS
# -----------------------------
def copy_answers(responses_df, source, targets, copy_rate, rng):
    responses_df = responses_df.copy()

    source_rows = responses_df[
        responses_df["candidate_id"].eq(source)
    ].sort_values(["exam_id", "sequence_position"])

    for target in targets:
        target_rows = responses_df[
            responses_df["candidate_id"].eq(target)
        ].sort_values(["exam_id", "sequence_position"])

        n = min(len(source_rows), len(target_rows))
        n_copy = int(n * copy_rate)
        if n_copy <= 0:
            continue

        chosen = rng.choice(n, size=n_copy, replace=False)
        src_idx = source_rows.index.to_numpy()[chosen]
        tgt_idx = target_rows.index.to_numpy()[chosen]

        responses_df.loc[tgt_idx, "selected_option"] = (
            responses_df.loc[src_idx, "selected_option"].to_numpy()
        )
        responses_df.loc[tgt_idx, "is_correct"] = (
            responses_df.loc[tgt_idx, "selected_option"].to_numpy()
            == responses_df.loc[tgt_idx, "selected_option"].to_numpy()
        ).astype(np.int8)

        # Correlated timing for copied answers.
        src_rt = responses_df.loc[src_idx, "response_time_ms"].to_numpy()
        responses_df.loc[tgt_idx, "response_time_ms"] = np.round(
            src_rt * rng.lognormal(0, 0.08, n_copy), 2
        )
    return responses_df

# Fix correctness after copying by joining item answer keys.
answer_map = items.set_index("item_id")["correct_option"].to_dict()

for a, b, rate in pair_groups:
    responses = copy_answers(responses, a, [b], rate, rng)
    # Candidate B copies from A; correctness is based on item key.
    m = responses["candidate_id"].eq(b)
    responses.loc[m, "is_correct"] = (
        responses.loc[m, "selected_option"]
        == responses.loc[m, "item_id"].map(answer_map)
    ).astype(np.int8)

for group, source, rate in ring_groups:
    targets = [x for x in group if x != source]
    responses = copy_answers(responses, source, targets, rate, rng)
    for target in targets:
        m = responses["candidate_id"].eq(target)
        responses.loc[m, "is_correct"] = (
            responses.loc[m, "selected_option"]
            == responses.loc[m, "item_id"].map(answer_map)
        ).astype(np.int8)

# -----------------------------
# 8. PROCTORING SIMULATOR
# -----------------------------
def generate_proctoring(sessions_df, profile_map, rng):
    rows = []
    flag_rows = []
    flag_id = 1

    n = len(sessions_df)
    profiles_v = sessions_df["candidate_id"].map(profile_map).to_numpy()
    true_violation = np.isin(
        profiles_v,
        ["colluding_pair", "colluding_ring", "item_leak_group", "answer_key_holder"]
    ).astype(np.int8)

    # Session-level risk: correlated with truth, but noisy and affected by nuisance.
    risk = rng.beta(2, 8, n)
    risk += true_violation * rng.uniform(0.12, 0.30, n)
    risk += np.where(
        np.isin(profiles_v, ["low_effort", "test_anxiety"]),
        rng.uniform(0.03, 0.10, n), 0.0
    )

    lighting = rng.choice(["good", "poor"], size=n, p=[0.80, 0.20])
    webcam = rng.choice(["standard", "low"], size=n, p=[0.75, 0.25])
    noise = rng.choice(["low", "high"], size=n, p=[0.80, 0.20])
    connection = rng.choice(["stable", "unstable"], size=n, p=[0.85, 0.15])

    risk += 0.08 * (lighting == "poor")
    risk += 0.05 * (webcam == "low")
    risk += 0.04 * (noise == "high")
    risk += 0.05 * (connection == "unstable")
    risk = np.clip(risk, 0, 1)

    nuisance_weight = (
        1.0
        + 2.0 * (lighting == "poor")
        + 1.5 * (webcam == "low")
        + 1.5 * (noise == "high")
        + 1.2 * (connection == "unstable")
    )

    session_flags = [[] for _ in range(n)]

    # Direct calibration: choose the number of flags and the number of
    # true-positive flags so realized base rate and precision match DR-4.
    for flag_type, cfg in FLAG_CONFIG.items():
        n_flags = int(round(cfg["base_rate"] * n))
        n_true = int(round(n_flags * cfg["precision"]))
        n_false = n_flags - n_true

        # Operational precision is defined against a flag-level latent truth,
        # not against the candidate's overall violation profile. This avoids
        # an impossible constraint when a 95% precision flag has a 12% base rate.
        all_pool = np.arange(n)
        true_weights = (
            0.25
            + risk
            + 1.5 * true_violation
        )
        true_weights = true_weights / true_weights.sum()

        selected_true = rng.choice(
            all_pool, size=n_true, replace=False, p=true_weights
        ) if n_true else np.array([], dtype=int)

        remaining = np.setdiff1d(all_pool, selected_true, assume_unique=False)
        false_weights = (
            0.25
            + nuisance_weight[remaining]
            + 0.4 * (1.0 - risk[remaining])
        )
        false_weights = false_weights / false_weights.sum()

        selected_false = rng.choice(
            remaining, size=n_false, replace=False, p=false_weights
        ) if n_false else np.array([], dtype=int)

        selected = np.concatenate([selected_true, selected_false])
        rng.shuffle(selected)

        for idx in selected:
            is_true_flag = int(idx in set(selected_true))
            confidence = float(np.clip(
                rng.normal(cfg["precision"], 0.06), 0.05, 0.99
            ))
            start_ms = int(rng.integers(0, 60 * 60 * 1000))
            end_ms = start_ms + int(rng.integers(2_000, 20_000))
            fid = f"flg_{flag_id:07d}"
            flag_id += 1

            obj = {
                "flag_id": fid,
                "type": flag_type,
                "confidence": round(confidence, 3),
                "t_start_ms": start_ms,
                "t_end_ms": end_ms,
            }
            session_flags[idx].append(obj)

            flag_rows.append({
                "flag_id": fid,
                "session_id": sessions_df.iloc[idx]["session_id"],
                "candidate_id": sessions_df.iloc[idx]["candidate_id"],
                "exam_id": sessions_df.iloc[idx]["exam_id"],
                "flag_type": flag_type,
                "confidence": round(confidence, 3),
                "t_start_ms": start_ms,
                "t_end_ms": end_ms,
                "ground_truth_flag": is_true_flag,
                "candidate_violation": int(true_violation[idx]),
            })

    for i, row in enumerate(sessions_df.itertuples(index=False)):
        if risk[i] < 0.30:
            recommendation = "NO_ACTION"
        elif risk[i] < 0.50:
            recommendation = "ROUTINE_REVIEW"
        elif risk[i] < 0.75:
            recommendation = "HUMAN_REVIEW"
        else:
            recommendation = "PRIORITY_REVIEW"

        missing = []
        if rng.random() < 0.10:
            missing.append(rng.choice(["attention", "audio", "video", "screen"]))

        nuisance = {
            "lighting": lighting[i],
            "webcam_class": webcam[i],
            "noise_level": noise[i],
            "connection": connection[i],
        }

        rows.append({
            "schema": "session_risk.v1",
            "session_id": row.session_id,
            "candidate_id": row.candidate_id,
            "exam_id": row.exam_id,
            "overall_risk": round(float(risk[i]), 4),
            "recommendation": recommendation,
            "flags": json.dumps(session_flags[i]),
            "flag_count": len(session_flags[i]),
            "channels_missing": json.dumps(missing),
            "nuisance": json.dumps(nuisance),
            "model_version": "sim_v2.0",
        })

    return pd.DataFrame(rows), pd.DataFrame(flag_rows)

profile_map = dict(zip(candidate_ids, profiles))
session_risk, flag_events = generate_proctoring(sessions, profile_map, rng)

# -----------------------------
# 9. REVIEWER SIMULATOR
# -----------------------------
reviewer_ids = [f"reviewer_{i:02d}" for i in range(1, N_REVIEWERS + 1)]

# 3 reviewers are deliberately drifting, but this information is NOT
# exposed in analytical reviewer data.
drift_reviewer_ids = set(rng.choice(reviewer_ids, size=3, replace=False).tolist())

reviewer_base_accuracy = {
    rid: float(rng.uniform(0.82, 0.92))
    for rid in reviewer_ids
}

reviewers = pd.DataFrame({
    "schema": "reviewer.v1",
    "reviewer_id": reviewer_ids,
    "agreement_rate_configured": [
        round(reviewer_base_accuracy[r], 3) for r in reviewer_ids
    ],
})

# Reviewer assignment: round-robin over reviewers, giving each reviewer a
# real position_in_shift rather than an independently sampled fake position.
review_rows = []
review_id = 1
review_counts = {rid: 0 for rid in reviewer_ids}

# Public review cases: all flags.
for flag in flag_events.itertuples(index=False):
    reviewer = reviewer_ids[(review_id - 1) % N_REVIEWERS]
    review_counts[reviewer] += 1
    position = review_counts[reviewer]

    base_acc = reviewer_base_accuracy[reviewer]

    # Fatigue: ~8% relative decline after 30 consecutive reviews.
    fatigue_factor = 0.92 if position > 30 else 1.0

    # Slow strictness/accuracy drift for hidden subset.
    drift_factor = 1.0
    if reviewer in drift_reviewer_ids:
        drift_factor = max(0.96, 1.0 - 0.04 * (position / 6000))

    accuracy = float(np.clip(base_acc * fatigue_factor * drift_factor, 0.55, 0.97))

    truth = bool(flag.ground_truth_flag)

    if rng.random() < accuracy:
        verdict = "CONFIRMED" if truth else "OVERTURNED"
    else:
        verdict = "OVERTURNED" if truth else "CONFIRMED"

    review_seconds = float(rng.lognormal(mean=np.log(240), sigma=0.55))

    review_rows.append({
        "schema": "review_verdict.v1",
        "review_id": f"rev_{review_id:07d}",
        "session_id": flag.session_id,
        "candidate_id": flag.candidate_id,
        "exam_id": flag.exam_id,
        "flag_id": flag.flag_id,
        "reviewer_id": reviewer,
        "verdict": verdict,
        "reviewer_confidence": rng.choice(
            ["low", "medium", "high"], p=[0.15, 0.45, 0.40]
        ),
        "review_seconds": round(review_seconds, 2),
        "position_in_shift": position,
        "is_double_review": False,
    })
    review_id += 1

reviews = pd.DataFrame(review_rows)

# 10% of cases receive a SECOND, independent reviewer verdict.
n_double = int(round(len(reviews) * DOUBLE_REVIEW_RATE))
double_base_idx = rng.choice(reviews.index, size=n_double, replace=False)

second_rows = []
for idx in double_base_idx:
    base = reviews.loc[idx]
    available = [r for r in reviewer_ids if r != base["reviewer_id"]]
    reviewer = rng.choice(available)
    review_counts[reviewer] += 1
    position = review_counts[reviewer]

    base_acc = reviewer_base_accuracy[reviewer]
    fatigue_factor = 0.92 if position > 30 else 1.0
    drift_factor = (
        max(0.96, 1.0 - 0.04 * (position / 6000))
        if reviewer in drift_reviewer_ids else 1.0
    )
    accuracy = float(np.clip(base_acc * fatigue_factor * drift_factor, 0.55, 0.97))
    truth = bool(
        flag_events.loc[
            flag_events["flag_id"].eq(base["flag_id"]),
            "ground_truth_flag"
        ].iloc[0]
    )
    if rng.random() < accuracy:
        verdict = "CONFIRMED" if truth else "OVERTURNED"
    else:
        verdict = "OVERTURNED" if truth else "CONFIRMED"

    # Shared case interpretation creates realistic inter-rater correlation
    # on double-reviewed cases without making reviewers identical.
    if rng.random() < 0.35:
        verdict = base["verdict"]

    second_rows.append({
        "schema": "review_verdict.v1",
        "review_id": f"rev_{review_id:07d}",
        "session_id": base["session_id"],
        "candidate_id": base["candidate_id"],
        "exam_id": base["exam_id"],
        "flag_id": base["flag_id"],
        "reviewer_id": reviewer,
        "verdict": verdict,
        "reviewer_confidence": rng.choice(
            ["low", "medium", "high"], p=[0.15, 0.45, 0.40]
        ),
        "review_seconds": round(
            float(rng.lognormal(np.log(240), 0.55)), 2
        ),
        "position_in_shift": position,
        "is_double_review": True,
    })
    reviews.loc[idx, "is_double_review"] = True
    review_id += 1

if second_rows:
    reviews = pd.concat([reviews, pd.DataFrame(second_rows)], ignore_index=True)

# Hidden reviewer ground truth for validation only.
reviewer_ground_truth = pd.DataFrame({
    "reviewer_id": reviewer_ids,
    "is_drifter": [r in drift_reviewer_ids for r in reviewer_ids],
    "configured_agreement": [
        round(reviewer_base_accuracy[r], 4) for r in reviewer_ids
    ],
})

# -----------------------------
# 10. COST SIMULATOR
# -----------------------------
def generate_cost_data(sessions_df, reviews_df, rng):
    n = len(sessions_df)
    compute = rng.uniform(0.02, 0.10, n)
    storage = rng.uniform(0.005, 0.03, n)
    egress = rng.uniform(0.001, 0.02, n)

    review_counts = (
        reviews_df.groupby("session_id").size()
        .reindex(sessions_df["session_id"], fill_value=0)
        .to_numpy()
    )
    # Labour cost is linked to actual review volume and review duration.
    mean_review_cost = 0.12  # synthetic currency unit per review-minute
    labor = review_counts * mean_review_cost * (
        reviews_df["review_seconds"].mean() / 60.0
        if len(reviews_df) else 4.0
    ) / max(1, review_counts.max())

    # Keep the scale non-negative and realistic.
    labor = np.maximum(labor, 0.0)

    return pd.DataFrame({
        "schema": "infrastructure_cost.v1",
        "session_id": sessions_df["session_id"],
        "compute_cost": np.round(compute, 4),
        "storage_cost": np.round(storage, 4),
        "egress_cost": np.round(egress, 4),
        "review_labor_cost": np.round(labor, 4),
        "total_cost": np.round(compute + storage + egress + labor, 4),
    })

costs = generate_cost_data(sessions, reviews, rng)

# -----------------------------
# 11. SAVE RAW DATA
# -----------------------------
raw = DATA_DIR / "raw"
candidates.to_csv(raw / "candidates.csv", index=False)
items.to_csv(raw / "items.csv", index=False)
sessions.to_csv(raw / "sessions.csv", index=False)
responses.to_csv(raw / "exam_responses.csv", index=False)
session_risk.to_csv(raw / "session_risk.csv", index=False)
flag_events.drop(columns=["ground_truth_flag", "candidate_violation"]).to_csv(
    raw / "proctoring_flags.csv", index=False
)
reviews.to_csv(raw / "review_verdicts.csv", index=False)
reviewers.to_csv(raw / "reviewers.csv", index=False)
costs.to_csv(raw / "infrastructure_cost.csv", index=False)

# Validation-only schema.
ground_truth = ground_truth_candidates.copy()
ground_truth.to_csv(
    DATA_DIR / "ground_truth" / "ground_truth_candidates.csv", index=False
)
reviewer_ground_truth.to_csv(
    DATA_DIR / "ground_truth" / "ground_truth_reviewers.csv", index=False
)
# Ground truth for flag validation.
flag_gt = flag_events[[
    "flag_id", "session_id", "flag_type", "ground_truth_flag",
    "candidate_violation"
]].copy()
flag_gt.to_csv(
    DATA_DIR / "ground_truth" / "ground_truth_flags.csv", index=False
)

# -----------------------------
# 12. VALIDATION
# -----------------------------
def assert_true(condition, message):
    if not condition:
        raise AssertionError("FAIL: " + message)
    print("PASS:", message)

def wilson_half_width(phat, n, z=1.96):
    if n == 0:
        return np.nan
    denom = 1 + z*z/n
    center = (phat + z*z/(2*n)) / denom
    half = z*np.sqrt(phat*(1-phat)/n + z*z/(4*n*n)) / denom
    return half

validation = {}

assert_true(len(candidates) == N_CANDIDATES, "FR-1: 5,000 candidates")
assert_true(len(exams) if False else N_EXAMS == 20, "FR-1: 20 exams")
assert_true(len(items) == N_ITEMS, "FR-1: 1,000 items")
assert_true(len(responses) == N_CANDIDATES * N_EXAMS * ITEMS_PER_EXAM,
            "FR-1: 5,000 x 20 x 50 responses")
assert_true(set(ground_truth["profile"]) == set(PROFILES),
            "FR-1: all 7 injected profiles present")
assert_true(set(responses.columns) >= {
    "response_id", "session_id", "candidate_id", "exam_id", "item_id",
    "selected_option", "is_correct", "response_time_ms",
    "sequence_position", "revisions"
}, "Contract: exam_response.v1 fields")
assert_true(set(session_risk["recommendation"]).issubset(ALLOWED_RECOMMENDATIONS),
            "Contract: session_risk recommendations")
assert_true(set(reviews["verdict"]).issubset(ALLOWED_VERDICTS),
            "Contract: reviewer verdict values")
double_case_rate = (
    reviews.groupby("flag_id")["is_double_review"].max().mean()
)
assert_true(
    0.095 <= double_case_rate <= 0.105,
    "FR-3: ~10% of cases are double-reviewed"
)

# Validate realized flag base rates and precision against DR-4.
precision_rows = []
for flag_type, cfg in FLAG_CONFIG.items():
    fe = flag_events[flag_events["flag_type"].eq(flag_type)]
    n = len(fe)
    rate = n / len(sessions)
    precision = fe["ground_truth_flag"].mean() if n else np.nan
    precision_rows.append({
        "flag_type": flag_type,
        "target_base_rate": cfg["base_rate"],
        "realized_base_rate": rate,
        "base_rate_abs_error": abs(rate - cfg["base_rate"]),
        "target_precision": cfg["precision"],
        "realized_precision": precision,
        "precision_abs_error": abs(precision - cfg["precision"])
        if n else np.nan,
        "n_flags": n,
        "precision_wilson_half_width": wilson_half_width(
            precision, n
        ) if n else np.nan
    })
precision_report = pd.DataFrame(precision_rows)
precision_report.to_csv(
    DATA_DIR / "validation" / "flag_precision_report.csv", index=False
)

# Precision is stochastic; SRS explicitly allows ±0.05.
assert_true(
    (precision_report["precision_abs_error"] <= 0.05).all(),
    "FR-2: realized flag precision within ±0.05"
)

# Inter-rater reliability validation on double-reviewed cases.
try:
    from sklearn.metrics import cohen_kappa_score
    paired = reviews[reviews["is_double_review"]].sort_values(
        ["flag_id", "review_id"]
    ).groupby("flag_id")
    first_verdicts, second_verdicts = [], []
    for _, g in paired:
        if len(g) == 2:
            first_verdicts.append(g.iloc[0]["verdict"])
            second_verdicts.append(g.iloc[1]["verdict"])
    verdict_map = {"CONFIRMED": 1, "OVERTURNED": 0}
    if len(first_verdicts) >= 30:
        overall_kappa = float(cohen_kappa_score(
            [verdict_map[x] for x in first_verdicts],
            [verdict_map[x] for x in second_verdicts]
        ))
        validation["overall_double_review_kappa"] = overall_kappa
        assert_true(
            0.50 <= overall_kappa <= 0.80,
            "FR-3: double-review Cohen's kappa is in the configured range"
        )
except Exception as exc:
    validation["kappa_warning"] = str(exc)

# Reviewer agreement validation against hidden ground truth.
flag_truth_map = flag_events.set_index("flag_id")["ground_truth_flag"].to_dict()
reviews["truth_for_validation"] = reviews["flag_id"].map(flag_truth_map).astype(int)
reviews["correct_for_validation"] = (
    ((reviews["verdict"] == "CONFIRMED") & (reviews["truth_for_validation"] == 1))
    | ((reviews["verdict"] == "OVERTURNED") & (reviews["truth_for_validation"] == 0))
).astype(int)

reviewer_agreement_report = (
    reviews.groupby("reviewer_id")["correct_for_validation"]
    .agg(["mean", "count"])
    .reset_index()
    .rename(columns={"mean": "realized_agreement"})
)
reviewer_agreement_report = reviewer_agreement_report.merge(
    reviewers[["reviewer_id", "agreement_rate_configured"]],
    on="reviewer_id",
    how="left"
)
reviewer_agreement_report["abs_error"] = (
    reviewer_agreement_report["realized_agreement"]
    - reviewer_agreement_report["agreement_rate_configured"]
).abs()
reviewer_agreement_report.to_csv(
    DATA_DIR / "validation" / "reviewer_agreement_report.csv", index=False
)

# The generator is configured in 0.75–0.92. Realized agreement is stochastic,
# so use a broad engineering check rather than requiring exact equality.
assert_true(
    reviewer_agreement_report["realized_agreement"].between(0.65, 0.97).all(),
    "FR-3: reviewer realized agreement remains in plausible range"
)

# Double-review existence: every double case has two distinct reviewers.
double_counts = (
    reviews[reviews["is_double_review"]]
    .groupby("flag_id")["reviewer_id"].nunique()
)
assert_true(
    len(double_counts) > 0 and (double_counts >= 2).all(),
    "FR-3: double-reviewed cases have two distinct reviewers"
)

# Profiles actually affect response behavior.
profile_accuracy = (
    responses.merge(
        candidates[["candidate_id", "ability_theta"]],
        on="candidate_id",
        how="left"
    )
    .assign(profile=lambda d: d["candidate_id"].map(profile_by_candidate))
    .groupby("profile")["is_correct"].mean()
)
assert_true(
    "item_leak_group" in profile_accuracy.index
    and "colluding_ring" in profile_accuracy.index,
    "FR-1: ring and item-leak profiles are operational"
)

# FR-5: simple baseline should NOT be perfect.
session_truth = (
    sessions["candidate_id"].map(
        ground_truth_candidates.set_index("candidate_id")["violation"]
    ).astype(int)
)
risk_auc_like = None
try:
    from sklearn.metrics import roc_auc_score
    risk_auc_like = roc_auc_score(session_truth, session_risk["overall_risk"])
except Exception:
    pass

if risk_auc_like is not None:
    validation["simple_risk_auc"] = float(risk_auc_like)
    assert_true(
        0.60 <= risk_auc_like <= 0.97,
        "FR-5: risk-only baseline is informative but not near-perfect"
    )

# -----------------------------
# 13. CONTRACT VALIDATION + STAGING
# -----------------------------
def validate_chunk(df, table_name):
    errors = pd.Series("", index=df.index, dtype="object")

    required = {
        "exam_responses": {
            "response_id", "session_id", "candidate_id", "exam_id",
            "item_id", "selected_option", "is_correct",
            "response_time_ms", "sequence_position", "revisions"
        },
        "session_risk": {
            "schema", "session_id", "candidate_id", "exam_id",
            "overall_risk", "recommendation", "flags",
            "flag_count", "channels_missing", "nuisance", "model_version"
        },
        "review_verdicts": {
            "schema", "review_id", "session_id", "candidate_id",
            "exam_id", "flag_id", "reviewer_id", "verdict",
            "reviewer_confidence", "review_seconds",
            "position_in_shift", "is_double_review"
        }
    }[table_name]

    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{table_name}: missing columns {sorted(missing)}")

    if table_name == "exam_responses":
        bad = (
            ~df["selected_option"].isin(ALLOWED_OPTIONS)
            | ~df["is_correct"].isin([0, 1])
            | (df["response_time_ms"] <= 0)
            | (df["sequence_position"] <= 0)
            | df["response_id"].duplicated(keep=False)
        )
    elif table_name == "session_risk":
        bad = (
            ~df["recommendation"].isin(ALLOWED_RECOMMENDATIONS)
            | ~df["overall_risk"].between(0, 1)
            | (df["flag_count"] < 0)
        )
    else:
        bad = (
            ~df["verdict"].isin(ALLOWED_VERDICTS)
            | (df["review_seconds"] <= 0)
            | (df["position_in_shift"] <= 0)
        )

    return ~bad

def ingest_csv_contract(input_path, staging_path, rejected_path, table_name,
                        chunksize=250_000):
    first = True
    accepted = rejected = 0

    for chunk in pd.read_csv(input_path, chunksize=chunksize):
        good = validate_chunk(chunk, table_name)
        bad = ~good

        valid = chunk.loc[good]
        invalid = chunk.loc[bad].copy()
        if len(invalid):
            invalid["rejection_reason"] = "contract_validation_failed"
            invalid.to_csv(
                rejected_path, mode="a", header=not os.path.exists(rejected_path),
                index=False
            )
            rejected += len(invalid)

        valid.to_csv(
            staging_path, mode="a", header=first, index=False
        )
        first = False
        accepted += len(valid)

    return accepted, rejected

tables_for_ingestion = {
    "exam_responses": raw / "exam_responses.csv",
    "session_risk": raw / "session_risk.csv",
    "review_verdicts": raw / "review_verdicts.csv",
}

ingestion_report = []
for table_name, path in tables_for_ingestion.items():
    staging_path = DATA_DIR / "staging" / f"{table_name}.csv"
    rejected_path = DATA_DIR / "rejected" / f"{table_name}_rejected.csv"

    if staging_path.exists():
        staging_path.unlink()
    if rejected_path.exists():
        rejected_path.unlink()

    accepted, rejected = ingest_csv_contract(
        path, staging_path, rejected_path, table_name
    )
    ingestion_report.append({
        "table": table_name,
        "source_rows": accepted + rejected,
        "accepted_rows": accepted,
        "rejected_rows": rejected,
        "staging_path": str(staging_path),
        "rejected_path": str(rejected_path),
    })

ingestion_report = pd.DataFrame(ingestion_report)
ingestion_report.to_csv(
    DATA_DIR / "validation" / "ingestion_report.csv", index=False
)

# Nothing silently dropped.
assert_true(
    (ingestion_report["source_rows"]
     == ingestion_report["accepted_rows"] + ingestion_report["rejected_rows"]).all(),
    "FR-6: no records silently dropped"
)
assert_true(
    ingestion_report["rejected_rows"].sum() == 0,
    "FR-6: generated clean input has no rejected records"
)

# Deliberate corruption demonstration.
demo = pd.DataFrame([{
    "response_id": "BAD_001",
    "session_id": "sess_bad",
    "candidate_id": "cand_bad",
    "exam_id": "exam_bad",
    "item_id": "item_bad",
    "selected_option": "X",
    "is_correct": 1,
    "response_time_ms": -5,
    "sequence_position": 0,
    "revisions": 0,
}])
demo_good = validate_chunk(demo, "exam_responses")
assert_true((~demo_good).all(), "FR-6 demo: malformed record is rejected")

# -----------------------------
# 14. SANITY OUTPUT
# -----------------------------
summary = {
    "seed": SEED,
    "candidates": len(candidates),
    "exams": N_EXAMS,
    "items": len(items),
    "sessions": len(sessions),
    "responses": len(responses),
    "flags": len(flag_events),
    "reviews": len(reviews),
    "double_review_rate_by_case": float(
        reviews.groupby("flag_id")["is_double_review"].max().mean()
    ),
    "overall_accuracy": float(responses["is_correct"].mean()),
    "avg_response_time_ms": float(responses["response_time_ms"].mean()),
    "risk_distribution": session_risk["recommendation"].value_counts().to_dict(),
    "profile_distribution": candidate_profile_df["profile"].value_counts(
        normalize=True
    ).round(4).to_dict(),
    "simple_risk_auc": validation.get("simple_risk_auc"),
    "overall_double_review_kappa": validation.get("overall_double_review_kappa"),
}

with open(DATA_DIR / "validation" / "run_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\n" + "=" * 65)
print("INTEGRITYINSIGHT PERSON 1 — GENERATION + INGESTION COMPLETE")
print("=" * 65)
print(json.dumps(summary, indent=2))
print("\nRAW:", raw)
print("GROUND TRUTH (validation only):", DATA_DIR / "ground_truth")
print("STAGING:", DATA_DIR / "staging")
print("REJECTED:", DATA_DIR / "rejected")
print("VALIDATION:", DATA_DIR / "validation")
print("=" * 65)


PASS: FR-1: 5,000 candidates
PASS: FR-1: 20 exams
PASS: FR-1: 1,000 items
PASS: FR-1: 5,000 x 20 x 50 responses
PASS: FR-1: all 7 injected profiles present
PASS: Contract: exam_response.v1 fields
PASS: Contract: session_risk recommendations
PASS: Contract: reviewer verdict values
PASS: FR-3: ~10% of cases are double-reviewed
PASS: FR-2: realized flag precision within ±0.05
PASS: FR-3: double-review Cohen's kappa is in the configured range
PASS: FR-3: reviewer realized agreement remains in plausible range
PASS: FR-3: double-reviewed cases have two distinct reviewers
PASS: FR-1: ring and item-leak profiles are operational
PASS: FR-5: risk-only baseline is informative but not near-perfect


In [ ]:

#  INGESTION PIPELINE

from pathlib import Path
import pandas as pd
import numpy as np
import json
import os
import time

# Person 1 output
DATA_DIR = Path.cwd() / "integrityinsight_data"

RAW_DIR = DATA_DIR / "raw"
STAGING_DIR = DATA_DIR / "staging"
REJECTED_DIR = DATA_DIR / "rejected"
VALIDATION_DIR = DATA_DIR / "validation"

# Create our output folders
STAGING_DIR.mkdir(parents=True, exist_ok=True)
REJECTED_DIR.mkdir(parents=True, exist_ok=True)
VALIDATION_DIR.mkdir(parents=True, exist_ok=True)

print("RAW:", RAW_DIR)
print("STAGING:", STAGING_DIR)
print("REJECTED:", REJECTED_DIR)
print("VALIDATION:", VALIDATION_DIR)

RAW: /content/integrityinsight_data/raw
STAGING: /content/integrityinsight_data/staging
REJECTED: /content/integrityinsight_data/rejected
VALIDATION: /content/integrityinsight_data/validation


In [ ]:
#  CHECK RAW INPUT FILES

raw_files = sorted(RAW_DIR.glob("*.csv"))

print(f"Number of raw CSV files: {len(raw_files)}")
print()

for file in raw_files:
    print(file.name)

Number of raw CSV files: 9

candidates.csv
exam_responses.csv
infrastructure_cost.csv
items.csv
proctoring_flags.csv
review_verdicts.csv
reviewers.csv
session_risk.csv
sessions.csv


In [ ]:
#  SCHEMA CONTRACTS

REQUIRED_COLUMNS = {

    "candidates": [
    "candidate_id",
    "ability_theta"
],
    "items": [
        "item_id",
        "exam_id",
        "item_number",
        "correct_option",
        "difficulty_b",
        "discrimination_a",
        "guessing_c"
    ],

    "sessions": [
        "session_id",
        "candidate_id",
        "exam_id"
    ],

    "exam_responses": [
        "response_id",
        "session_id",
        "candidate_id",
        "exam_id",
        "item_id",
        "selected_option",
        "is_correct",
        "response_time_ms",
        "sequence_position",
        "revisions"
    ],

    "session_risk": [
        "session_id",
        "candidate_id",
        "exam_id",
        "overall_risk",
        "recommendation",
        "flag_count"
    ],

    "proctoring_flags": [
        "flag_id",
        "session_id",
        "candidate_id",
        "exam_id",
        "flag_type",
        "confidence",
        "t_start_ms",
        "t_end_ms"
    ],

    "review_verdicts": [
        "review_id",
        "session_id",
        "candidate_id",
        "exam_id",
        "reviewer_id",
        "verdict",
        "review_seconds",
        "position_in_shift"
    ],

    "reviewers": [
        "reviewer_id",
        "agreement_rate_configured"
    ],

    "infrastructure_cost": [
    "schema",
    "session_id",
    "compute_cost",
    "storage_cost",
    "egress_cost",
    "review_labor_cost",
    "total_cost"
]
}

print("Schema contracts created successfully.")
print()

for table, columns in REQUIRED_COLUMNS.items():
    print(f"{table}: {len(columns)} required columns")

Schema contracts created successfully.

candidates: 2 required columns
items: 7 required columns
sessions: 3 required columns
exam_responses: 10 required columns
session_risk: 6 required columns
proctoring_flags: 8 required columns
review_verdicts: 8 required columns
reviewers: 2 required columns
infrastructure_cost: 7 required columns


In [ ]:
# ALLOWED VALUES

ALLOWED_OPTIONS = {
    "A", "B", "C", "D"
}

ALLOWED_RECOMMENDATIONS = {
    "NO_ACTION",
    "ROUTINE_REVIEW",
    "HUMAN_REVIEW",
    "PRIORITY_REVIEW"
}

ALLOWED_VERDICTS = {
    "CONFIRMED",
    "OVERTURNED",
    "INCONCLUSIVE",
    "TECHNICAL_ISSUE",
    "ESCALATED"
}

ALLOWED_FLAG_TYPES = {
    "TAB_SWITCH",
    "SECOND_VOICE",
    "IDENTITY_MISMATCH",
    "PHONE_DETECTED",
    "GAZE_OFF_SCREEN",
    "MULTI_PERSON",
    "MULTI_MONITOR",
    "REMOTE_TOOL"
}

print("Allowed-value contracts created successfully.")
print()

print("Options:", ALLOWED_OPTIONS)
print("Recommendations:", ALLOWED_RECOMMENDATIONS)
print("Verdicts:", ALLOWED_VERDICTS)
print("Flag types:", ALLOWED_FLAG_TYPES)

Allowed-value contracts created successfully.

Options: {'C', 'B', 'D', 'A'}
Recommendations: {'HUMAN_REVIEW', 'NO_ACTION', 'PRIORITY_REVIEW', 'ROUTINE_REVIEW'}
Verdicts: {'ESCALATED', 'TECHNICAL_ISSUE', 'INCONCLUSIVE', 'CONFIRMED', 'OVERTURNED'}
Flag types: {'MULTI_MONITOR', 'TAB_SWITCH', 'MULTI_PERSON', 'REMOTE_TOOL', 'PHONE_DETECTED', 'GAZE_OFF_SCREEN', 'IDENTITY_MISMATCH', 'SECOND_VOICE'}


In [ ]:
# — SCHEMA VALIDATION FUNCTION

def validate_schema(df, table_name):
    """
    Check whether a table contains all required columns.
    Returns:
        is_valid : True / False
        missing_columns : list
    """

    required = REQUIRED_COLUMNS[table_name]

    missing_columns = [
        column for column in required
        if column not in df.columns
    ]

    is_valid = len(missing_columns) == 0

    return is_valid, missing_columns


print("Schema validation function created successfully.")

Schema validation function created successfully.


In [ ]:

#  RAW TABLE FILE MAPPING


TABLE_FILES = {
    "candidates": "candidates.csv",
    "items": "items.csv",
    "sessions": "sessions.csv",
    "exam_responses": "exam_responses.csv",
    "session_risk": "session_risk.csv",
    "proctoring_flags": "proctoring_flags.csv",
    "review_verdicts": "review_verdicts.csv",
    "reviewers": "reviewers.csv",
    "infrastructure_cost": "infrastructure_cost.csv",
}

print("Raw table mapping created successfully.")
print(f"Number of tables: {len(TABLE_FILES)}")

Raw table mapping created successfully.
Number of tables: 9


In [ ]:
# TEST SCHEMA VALIDATION

schema_results = []

for table_name, filename in TABLE_FILES.items():
    file_path = RAW_DIR / filename

    df = pd.read_csv(file_path)

    is_valid, missing_columns = validate_schema(
        df,
        table_name
    )

    schema_results.append({
        "table": table_name,
        "file": filename,
        "valid": is_valid,
        "missing_columns": ", ".join(missing_columns)
    })

schema_results_df = pd.DataFrame(schema_results)

print(schema_results_df.to_string(index=False))

              table                    file  valid missing_columns
         candidates          candidates.csv   True                
              items               items.csv   True                
           sessions            sessions.csv   True                
     exam_responses      exam_responses.csv   True                
       session_risk        session_risk.csv   True                
   proctoring_flags    proctoring_flags.csv   True                
    review_verdicts     review_verdicts.csv   True                
          reviewers           reviewers.csv   True                
infrastructure_cost infrastructure_cost.csv   True                


In [ ]:
#  ROW-LEVEL VALIDATION

def validate_rows(df, table_name):

    errors = pd.Series("", index=df.index, dtype="object")

    def add_error(mask, message):
        errors.loc[mask] = errors.loc[mask].apply(
            lambda x: message if x == "" else x + "; " + message
        )

    # --------------------------------------------------------
    # EXAM RESPONSES
    # --------------------------------------------------------
    if table_name == "exam_responses":

        add_error(
            ~df["selected_option"].isin(ALLOWED_OPTIONS),
            "invalid selected_option"
        )

        add_error(
            ~df["response_time_ms"].apply(
                lambda x: pd.notna(x) and x >= 0
            ),
            "invalid response_time_ms"
        )

        add_error(
            ~df["sequence_position"].apply(
                lambda x: pd.notna(x) and x >= 1
            ),
            "invalid sequence_position"
        )

        add_error(
            ~df["revisions"].apply(
                lambda x: pd.notna(x) and x >= 0
            ),
            "invalid revisions"
        )

        add_error(
            df["response_id"].duplicated(keep=False),
            "duplicate response_id"
        )

    # --------------------------------------------------------
    # SESSION RISK
    # --------------------------------------------------------
    elif table_name == "session_risk":

        add_error(
            ~df["recommendation"].isin(ALLOWED_RECOMMENDATIONS),
            "invalid recommendation"
        )

        add_error(
            ~df["overall_risk"].apply(
                lambda x: pd.notna(x) and 0 <= x <= 1
            ),
            "invalid overall_risk"
        )

        add_error(
            ~df["flag_count"].apply(
                lambda x: pd.notna(x) and x >= 0
            ),
            "invalid flag_count"
        )

    # --------------------------------------------------------
    # PROCTORING FLAGS
    # --------------------------------------------------------
    elif table_name == "proctoring_flags":

        add_error(
            ~df["flag_type"].isin(ALLOWED_FLAG_TYPES),
            "invalid flag_type"
        )

        add_error(
            ~df["confidence"].apply(
                lambda x: pd.notna(x) and 0 <= x <= 1
            ),
            "invalid confidence"
        )

        add_error(
            ~df["t_start_ms"].apply(
                lambda x: pd.notna(x) and x >= 0
            ),
            "invalid t_start_ms"
        )

        add_error(
            ~df["t_end_ms"].apply(
                lambda x: pd.notna(x) and x >= 0
            ),
            "invalid t_end_ms"
        )

        add_error(
            df["t_end_ms"] < df["t_start_ms"],
            "t_end_ms before t_start_ms"
        )

    # --------------------------------------------------------
    # REVIEW VERDICTS
    # --------------------------------------------------------
    elif table_name == "review_verdicts":

        add_error(
            ~df["verdict"].isin(ALLOWED_VERDICTS),
            "invalid verdict"
        )

        add_error(
            ~df["review_seconds"].apply(
                lambda x: pd.notna(x) and x >= 0
            ),
            "invalid review_seconds"
        )

        add_error(
            ~df["position_in_shift"].apply(
                lambda x: pd.notna(x) and x >= 1
            ),
            "invalid position_in_shift"
        )

    # --------------------------------------------------------
    # ITEMS
    # --------------------------------------------------------
    elif table_name == "items":

        add_error(
            ~df["correct_option"].isin(ALLOWED_OPTIONS),
            "invalid correct_option"
        )

        add_error(
            ~df["item_number"].apply(
                lambda x: pd.notna(x) and x >= 1
            ),
            "invalid item_number"
        )

        add_error(
            ~df["discrimination_a"].apply(
                lambda x: pd.notna(x) and x > 0
            ),
            "invalid discrimination_a"
        )

        add_error(
            ~df["guessing_c"].apply(
                lambda x: pd.notna(x) and 0 <= x <= 1
            ),
            "invalid guessing_c"
        )

    # --------------------------------------------------------
    # REVIEWERS
    # --------------------------------------------------------
    elif table_name == "reviewers":

        add_error(
            ~df["agreement_rate_configured"].apply(
                lambda x: pd.notna(x) and 0 <= x <= 1
            ),
            "invalid agreement_rate_configured"
        )

    # --------------------------------------------------------
    # INFRASTRUCTURE COST
    # --------------------------------------------------------
    elif table_name == "infrastructure_cost":

        cost_columns = [
            "compute_cost",
            "storage_cost",
            "egress_cost",
            "review_labor_cost",
            "total_cost"
        ]

        for column in cost_columns:
            add_error(
                ~df[column].apply(
                    lambda x: pd.notna(x) and x >= 0
                ),
                f"invalid {column}"
            )

    return errors


print("Row-level validation function created successfully.")

Row-level validation function created successfully.


In [ ]:
# FR-6.7 — APPLY ROW-LEVEL VALIDATION

row_validation_results = []
rejected_records = {}

for table_name, filename in TABLE_FILES.items():

    file_path = RAW_DIR / filename
    df = pd.read_csv(file_path)

    # Validate every row
    errors = validate_rows(df, table_name)

    # Rows with at least one error
    invalid_mask = errors != ""

    invalid_count = invalid_mask.sum()
    valid_count = (~invalid_mask).sum()

    row_validation_results.append({
        "table": table_name,
        "total_rows": len(df),
        "valid_rows": valid_count,
        "invalid_rows": invalid_count
    })

    # Save rejected rows in memory
    if invalid_count > 0:

        rejected = df.loc[invalid_mask].copy()

        rejected["rejection_reason"] = (
            errors.loc[invalid_mask].values
        )

        rejected_records[table_name] = rejected


# Convert results to DataFrame
row_validation_results_df = pd.DataFrame(
    row_validation_results
)

print(row_validation_results_df.to_string(index=False))

              table  total_rows  valid_rows  invalid_rows
         candidates         300         300             0
              items        1000        1000             0
           sessions        6000        6000             0
     exam_responses      300000      300000             0
       session_risk        6000        6000             0
   proctoring_flags        3450        3450             0
    review_verdicts        3795        3795             0
          reviewers          12          12             0
infrastructure_cost        6000        6000             0


In [ ]:
# FR-6.8 — SAVE REJECTED RECORDS

for table_name, rejected_df in rejected_records.items():

    output_file = REJECTED_DIR / f"{table_name}_rejected.csv"

    rejected_df.to_csv(
        output_file,
        index=False
    )

    print(
        f"Saved rejected records for {table_name}: "
        f"{len(rejected_df)} rows"
    )


if len(rejected_records) == 0:
    print("No rejected records found.")
else:
    print(f"\nNumber of tables with rejected records: {len(rejected_records)}")

No rejected records found.


In [ ]:
# FR-6.9 — LOAD VALID RECORDS INTO STAGING

staging_results = []

for table_name, filename in TABLE_FILES.items():

    # Read raw data
    raw_file = RAW_DIR / filename
    df = pd.read_csv(raw_file)

    # Re-run row validation
    errors = validate_rows(df, table_name)

    # Keep only valid rows
    valid_mask = errors == ""
    valid_df = df.loc[valid_mask].copy()

    # Save valid records to staging
    staging_file = STAGING_DIR / filename

    valid_df.to_csv(
        staging_file,
        index=False
    )

    staging_results.append({
        "table": table_name,
        "raw_rows": len(df),
        "staged_rows": len(valid_df),
        "rejected_rows": len(df) - len(valid_df)
    })


# Create summary
staging_results_df = pd.DataFrame(staging_results)

print(staging_results_df.to_string(index=False))

              table  raw_rows  staged_rows  rejected_rows
         candidates       300          300              0
              items      1000         1000              0
           sessions      6000         6000              0
     exam_responses    300000       300000              0
       session_risk      6000         6000              0
   proctoring_flags      3450         3450              0
    review_verdicts      3795         3795              0
          reviewers        12           12              0
infrastructure_cost      6000         6000              0


In [ ]:
# FR-6.10 — INGESTION AUDIT LOG

from datetime import datetime

audit_time = datetime.now().isoformat(timespec="seconds")

audit_log = []

for _, row in staging_results_df.iterrows():

    audit_log.append({
        "ingestion_time": audit_time,
        "table_name": row["table"],
        "raw_rows": int(row["raw_rows"]),
        "staged_rows": int(row["staged_rows"]),
        "rejected_rows": int(row["rejected_rows"]),
        "status": "SUCCESS" if row["rejected_rows"] == 0 else "PARTIAL"
    })

audit_log_df = pd.DataFrame(audit_log)

# Save audit log
audit_file = VALIDATION_DIR / "ingestion_audit_log.csv"

audit_log_df.to_csv(
    audit_file,
    index=False
)

print(audit_log_df.to_string(index=False))
print()
print(f"Audit log saved to: {audit_file}")

     ingestion_time          table_name  raw_rows  staged_rows  rejected_rows  status
2026-10-03T14:26:43          candidates       300          300              0 SUCCESS
2026-10-03T14:26:43               items      1000         1000              0 SUCCESS
2026-10-03T14:26:43            sessions      6000         6000              0 SUCCESS
2026-10-03T14:26:43      exam_responses    300000       300000              0 SUCCESS
2026-10-03T14:26:43        session_risk      6000         6000              0 SUCCESS
2026-10-03T14:26:43    proctoring_flags      3450         3450              0 SUCCESS
2026-10-03T14:26:43     review_verdicts      3795         3795              0 SUCCESS
2026-10-03T14:26:43           reviewers        12           12              0 SUCCESS
2026-10-03T14:26:43 infrastructure_cost      6000         6000              0 SUCCESS

Audit log saved to: /content/integrityinsight_data/validation/ingestion_audit_log.csv


In [ ]:
# ============================================================
# STAGING INTEGRITY CHECK
# ============================================================

integrity_results = []

for _, row in audit_log_df.iterrows():

    table_name = row["table_name"]
    expected_rows = int(row["staged_rows"])
    filename = TABLE_FILES[table_name]

    staging_file = STAGING_DIR / filename

    # Check that staging file exists
    file_exists = staging_file.exists()

    # Read staging file and count rows
    if file_exists:
        staging_df = pd.read_csv(staging_file)
        actual_rows = len(staging_df)
    else:
        actual_rows = 0

    # Compare expected vs actual
    row_count_match = (
        actual_rows == expected_rows
    )

    integrity_results.append({
        "table": table_name,
        "file_exists": file_exists,
        "expected_rows": expected_rows,
        "actual_rows": actual_rows,
        "row_count_match": row_count_match
    })


integrity_results_df = pd.DataFrame(
    integrity_results
)

print(integrity_results_df.to_string(index=False))

print()

if (
    integrity_results_df["file_exists"].all()
    and integrity_results_df["row_count_match"].all()
):
    print("STAGING INTEGRITY CHECK: PASSED")
else:
    print("STAGING INTEGRITY CHECK: FAILED")

              table  file_exists  expected_rows  actual_rows  row_count_match
         candidates         True            300          300             True
              items         True           1000         1000             True
           sessions         True           6000         6000             True
     exam_responses         True         300000       300000             True
       session_risk         True           6000         6000             True
   proctoring_flags         True           3450         3450             True
    review_verdicts         True           3795         3795             True
          reviewers         True             12           12             True
infrastructure_cost         True           6000         6000             True

STAGING INTEGRITY CHECK: PASSED


In [ ]:
#  FINAL INGESTION SUMMARY

total_raw_rows = int(
    audit_log_df["raw_rows"].sum()
)

total_staged_rows = int(
    audit_log_df["staged_rows"].sum()
)

total_rejected_rows = int(
    audit_log_df["rejected_rows"].sum()
)

tables_processed = len(
    audit_log_df
)

successful_tables = int(
    (audit_log_df["status"] == "SUCCESS").sum()
)

print("=" * 60)
print(" INGESTION PIPELINE SUMMARY")
print("=" * 60)

print(f"Tables processed       : {tables_processed}")
print(f"Successful tables      : {successful_tables}")
print(f"Total raw rows         : {total_raw_rows:,}")
print(f"Total staged rows      : {total_staged_rows:,}")
print(f"Total rejected rows    : {total_rejected_rows:,}")

print()

if (
    successful_tables == tables_processed
    and total_rejected_rows == 0
    and integrity_results_df["row_count_match"].all()
):
    print("FR-6 INGESTION STATUS: PASSED")
else:
    print("FR-6 INGESTION STATUS: CHECK REQUIRED")

 INGESTION PIPELINE SUMMARY
Tables processed       : 9
Successful tables      : 9
Total raw rows         : 326,557
Total staged rows      : 326,557
Total rejected rows    : 0

FR-6 INGESTION STATUS: PASSED


In [ ]:
# ============================================================
# INTEGRITYINSIGHT
# FR-7 — TRANSFORMATION LAYER
# Star Schema + SCD Type 2 + Data Lineage
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json


# ============================================================
# 1. PATHS
# ============================================================

DATA_DIR = Path("/content/integrityinsight_data")

STAGING_DIR = DATA_DIR / "staging"
WAREHOUSE_DIR = DATA_DIR / "warehouse"

DIM_DIR = WAREHOUSE_DIR / "dimensions"
FACT_DIR = WAREHOUSE_DIR / "facts"
LINEAGE_DIR = WAREHOUSE_DIR / "lineage"

DIM_DIR.mkdir(parents=True, exist_ok=True)
FACT_DIR.mkdir(parents=True, exist_ok=True)
LINEAGE_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 70)
print("INTEGRITYINSIGHT — FR-7 TRANSFORMATION LAYER")
print("=" * 70)

print("STAGING   :", STAGING_DIR)
print("WAREHOUSE :", WAREHOUSE_DIR)


# ============================================================
# 2. LOAD STAGING TABLES
# ============================================================

candidates = pd.read_csv(
    STAGING_DIR / "candidates.csv"
)

items = pd.read_csv(
    STAGING_DIR / "items.csv"
)

sessions = pd.read_csv(
    STAGING_DIR / "sessions.csv"
)

exam_responses = pd.read_csv(
    STAGING_DIR / "exam_responses.csv"
)

session_risk = pd.read_csv(
    STAGING_DIR / "session_risk.csv"
)

proctoring_flags = pd.read_csv(
    STAGING_DIR / "proctoring_flags.csv"
)

review_verdicts = pd.read_csv(
    STAGING_DIR / "review_verdicts.csv"
)

reviewers = pd.read_csv(
    STAGING_DIR / "reviewers.csv"
)

infrastructure_cost = pd.read_csv(
    STAGING_DIR / "infrastructure_cost.csv"
)

print("\nStaging tables loaded successfully.")

print("\nStaging row counts:")

for name, df in {
    "candidates": candidates,
    "items": items,
    "sessions": sessions,
    "exam_responses": exam_responses,
    "session_risk": session_risk,
    "proctoring_flags": proctoring_flags,
    "review_verdicts": review_verdicts,
    "reviewers": reviewers,
    "infrastructure_cost": infrastructure_cost
}.items():

    print(f"{name:<25} {len(df):>10,}")


# ============================================================
# 3. DIM_CANDIDATE
# ============================================================

dim_candidate = candidates[
    [
        "candidate_id",
        "ability_theta"
    ]
].copy()

dim_candidate.insert(
    0,
    "candidate_key",
    range(1, len(dim_candidate) + 1)
)

dim_candidate["source_system"] = (
    "person1_raw_generator"
)

dim_candidate.to_csv(
    DIM_DIR / "dim_candidate.csv",
    index=False
)

print("\nDIM_CANDIDATE:",
      len(dim_candidate), "rows")


# ============================================================
# 4. DIM_ITEM
# ============================================================

dim_item = items[
    [
        "item_id",
        "exam_id",
        "item_number",
        "correct_option",
        "difficulty_b",
        "discrimination_a",
        "guessing_c"
    ]
].copy()

dim_item.insert(
    0,
    "item_key",
    range(1, len(dim_item) + 1)
)

dim_item["source_system"] = (
    "person1_raw_generator"
)

dim_item.to_csv(
    DIM_DIR / "dim_item.csv",
    index=False
)

print("DIM_ITEM:",
      len(dim_item), "rows")


# ============================================================
# 5. DIM_EXAM — TYPE 2 SCD
# ============================================================

exam_ids = (
    items["exam_id"]
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

dim_exam = pd.DataFrame({
    "exam_key": range(
        1,
        len(exam_ids) + 1
    ),

    "exam_id": exam_ids,

    "exam_version": 1,

    "valid_from": pd.Timestamp(
        "2026-01-01"
    ),

    "valid_to": pd.Timestamp(
        "2262-04-11"
    ),

    "is_current": True,

    "source_system":
        "person1_raw_generator"
})

dim_exam.to_csv(
    DIM_DIR / "dim_exam.csv",
    index=False
)

print("DIM_EXAM:",
      len(dim_exam), "rows")

print(
    "DIM_EXAM Type-2 structure created."
)


# ============================================================
# 6. DIM_REVIEWER — TYPE 2 SCD
# ============================================================

dim_reviewer = reviewers[
    [
        "reviewer_id",
        "agreement_rate_configured"
    ]
].copy()

dim_reviewer.insert(
    0,
    "reviewer_key",
    range(
        1,
        len(dim_reviewer) + 1
    )
)

dim_reviewer["reviewer_version"] = 1

dim_reviewer["valid_from"] = pd.Timestamp(
    "2026-01-01"
)

dim_reviewer["valid_to"] = pd.Timestamp(
    "2262-04-11"
)

dim_reviewer["is_current"] = True

dim_reviewer["source_system"] = (
    "person1_raw_generator"
)

dim_reviewer.to_csv(
    DIM_DIR / "dim_reviewer.csv",
    index=False
)

print("DIM_REVIEWER:",
      len(dim_reviewer), "rows")

print(
    "DIM_REVIEWER Type-2 structure created."
)


# ============================================================
# 7. DIM_FLAG_TYPE
# ============================================================

flag_types = (
    proctoring_flags["flag_type"]
    .dropna()
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

dim_flag_type = pd.DataFrame({

    "flag_key":
        range(
            1,
            len(flag_types) + 1
        ),

    "flag_type":
        flag_types,

    "source_system":
        "person1_raw_generator"
})

dim_flag_type.to_csv(
    DIM_DIR / "dim_flag_type.csv",
    index=False
)

print("DIM_FLAG_TYPE:",
      len(dim_flag_type), "rows")


# ============================================================
# 8. DIM_DATE
# ============================================================

# Person 1 source currently does not provide
# event-level dates.
#
# Therefore a project reference date is used.

date_value = pd.Timestamp(
    "2026-01-01"
)

dim_date = pd.DataFrame({

    "date_key":
        [20260101],

    "full_date":
        [date_value],

    "year":
        [2026],

    "month":
        [1],

    "month_name":
        [date_value.month_name()],

    "day":
        [1],

    "day_name":
        [date_value.day_name()]

})

dim_date.to_csv(
    DIM_DIR / "dim_date.csv",
    index=False
)

print("DIM_DATE:",
      len(dim_date), "row")


# ============================================================
# 9. SESSION KEY MAPPING
# ============================================================

# DR-5 requires session_key in FACT_SESSION_RISK
# and FACT_REVIEW.
#
# There is no DIM_SESSION in the required star schema.
# Therefore session_key is generated from unique session_id.

session_ids = (
    sessions["session_id"]
    .dropna()
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

session_key_map = pd.DataFrame({

    "session_id":
        session_ids,

    "session_key":
        range(
            1,
            len(session_ids) + 1
        )
})

print(
    "\nSession key mapping:",
    len(session_key_map),
    "sessions"
)


# ============================================================
# 10. FACT_RESPONSE
# ============================================================

fact_response = exam_responses.copy()


# Candidate key
fact_response = fact_response.merge(
    dim_candidate[
        [
            "candidate_key",
            "candidate_id"
        ]
    ],
    on="candidate_id",
    how="left"
)


# Item key
fact_response = fact_response.merge(
    dim_item[
        [
            "item_key",
            "item_id"
        ]
    ],
    on="item_id",
    how="left"
)


# Exam key
fact_response = fact_response.merge(
    dim_exam[
        [
            "exam_key",
            "exam_id"
        ]
    ],
    on="exam_id",
    how="left"
)


# Date key
fact_response["date_key"] = 20260101


# Response surrogate key
fact_response.insert(
    0,
    "response_key",
    range(
        1,
        len(fact_response) + 1
    )
)


# Keep both DR-5 fields and
# existing pipeline fields.

fact_response = fact_response[
    [
        "response_key",
        "candidate_key",
        "item_key",
        "exam_key",
        "date_key",
        "response_id",
        "session_id",
        "selected_option",
        "is_correct",
        "response_time_ms",
        "sequence_position",
        "revisions"
    ]
]

fact_response.to_csv(
    FACT_DIR / "fact_response.csv",
    index=False
)

print(
    "FACT_RESPONSE:",
    len(fact_response),
    "rows"
)


# ============================================================
# 11. FACT_SESSION_RISK
# ============================================================

fact_session_risk = session_risk.copy()


# ------------------------------------------------------------
# Session key
# ------------------------------------------------------------

fact_session_risk = fact_session_risk.merge(
    session_key_map,
    on="session_id",
    how="left"
)


# ------------------------------------------------------------
# Candidate key
# ------------------------------------------------------------

fact_session_risk = fact_session_risk.merge(
    dim_candidate[
        [
            "candidate_key",
            "candidate_id"
        ]
    ],
    on="candidate_id",
    how="left"
)


# ------------------------------------------------------------
# Exam key
# ------------------------------------------------------------

fact_session_risk = fact_session_risk.merge(
    dim_exam[
        [
            "exam_key",
            "exam_id"
        ]
    ],
    on="exam_id",
    how="left"
)


# ------------------------------------------------------------
# Date key
# ------------------------------------------------------------

fact_session_risk["date_key"] = 20260101


# ------------------------------------------------------------
# DR-5 field
#
# Keep original overall_risk because FR-8 currently
# depends on it.
#
# Create risk_score as the warehouse-standard name.
# ------------------------------------------------------------

if "overall_risk" in fact_session_risk.columns:

    fact_session_risk["risk_score"] = (
        fact_session_risk["overall_risk"]
    )

else:

    fact_session_risk["risk_score"] = np.nan


# ------------------------------------------------------------
# DR-5 optional fields
#
# Do not invent values if the source does not provide them.
# ------------------------------------------------------------

if "channels_missing" not in fact_session_risk.columns:

    fact_session_risk["channels_missing"] = np.nan


if "model_version" not in fact_session_risk.columns:

    fact_session_risk["model_version"] = np.nan


# ------------------------------------------------------------
# Session Risk surrogate key
# ------------------------------------------------------------

# session_key is already the warehouse session identifier.
# Do not create another independent key here.


# ------------------------------------------------------------
# Final FACT_SESSION_RISK
# ------------------------------------------------------------

fact_session_risk = fact_session_risk[
    [
        "session_key",
        "candidate_key",
        "exam_key",
        "date_key",
        "session_id",
        "overall_risk",
        "risk_score",
        "recommendation",
        "flag_count",
        "channels_missing",
        "model_version"
    ]
]


fact_session_risk.to_csv(
    FACT_DIR / "fact_session_risk.csv",
    index=False
)

print(
    "FACT_SESSION_RISK:",
    len(fact_session_risk),
    "rows"
)


# ============================================================
# 12. FACT_REVIEW
# ============================================================

fact_review = review_verdicts.copy()


# ------------------------------------------------------------
# Session key
# ------------------------------------------------------------

fact_review = fact_review.merge(
    session_key_map,
    on="session_id",
    how="left"
)


# ------------------------------------------------------------
# Candidate key
# ------------------------------------------------------------

fact_review = fact_review.merge(
    dim_candidate[
        [
            "candidate_key",
            "candidate_id"
        ]
    ],
    on="candidate_id",
    how="left"
)


# ------------------------------------------------------------
# Exam key
# ------------------------------------------------------------

fact_review = fact_review.merge(
    dim_exam[
        [
            "exam_key",
            "exam_id"
        ]
    ],
    on="exam_id",
    how="left"
)


# ------------------------------------------------------------
# Reviewer key
# ------------------------------------------------------------

fact_review = fact_review.merge(
    dim_reviewer[
        [
            "reviewer_key",
            "reviewer_id"
        ]
    ],
    on="reviewer_id",
    how="left"
)


# ------------------------------------------------------------
# Date key
# ------------------------------------------------------------

fact_review["date_key"] = 20260101


# ------------------------------------------------------------
# FLAG KEY
#
# If review_verdicts has flag_id:
#   flag_id → proctoring_flags.flag_type
#          → DIM_FLAG_TYPE.flag_key
#
# Otherwise keep flag_key nullable.
# ------------------------------------------------------------

if "flag_id" in fact_review.columns:

    flag_mapping = (
        proctoring_flags[
            [
                "flag_id",
                "flag_type"
            ]
        ]
        .drop_duplicates(
            subset=["flag_id"]
        )
    )

    fact_review = fact_review.merge(
        flag_mapping,
        on="flag_id",
        how="left"
    )

    fact_review = fact_review.merge(
        dim_flag_type[
            [
                "flag_key",
                "flag_type"
            ]
        ],
        on="flag_type",
        how="left"
    )

else:

    fact_review["flag_key"] = np.nan


# ------------------------------------------------------------
# CONFIDENCE
#
# Source uses reviewer_confidence.
# Transform it to DR-5 confidence.
# ------------------------------------------------------------

if "reviewer_confidence" in fact_review.columns:

    fact_review["confidence"] = (
        fact_review["reviewer_confidence"]
    )

elif "confidence" not in fact_review.columns:

    fact_review["confidence"] = np.nan


# ------------------------------------------------------------
# NOTES LENGTH
# ------------------------------------------------------------

if "notes_length" not in fact_review.columns:

    if "notes" in fact_review.columns:

        fact_review["notes_length"] = (
            fact_review["notes"]
            .fillna("")
            .astype(str)
            .str.len()
        )

    else:

        fact_review["notes_length"] = np.nan


# ------------------------------------------------------------
# Review surrogate key
# ------------------------------------------------------------

fact_review.insert(
    0,
    "review_key",
    range(
        1,
        len(fact_review) + 1
    )
)


# ------------------------------------------------------------
# Keep DR-5 + existing pipeline fields
# ------------------------------------------------------------

fact_review = fact_review[
    [
        "review_key",
        "session_key",
        "candidate_key",
        "exam_key",
        "reviewer_key",
        "date_key",
        "review_id",
        "flag_key",
        "verdict",
        "confidence",
        "notes_length",
        "review_seconds",
        "position_in_shift"
    ]
]


fact_review.to_csv(
    FACT_DIR / "fact_review.csv",
    index=False
)

print(
    "FACT_REVIEW:",
    len(fact_review),
    "rows"
)


# ============================================================
# 13. STAR SCHEMA VALIDATION
# ============================================================

print("\n")
print("=" * 70)
print("STAR SCHEMA VALIDATION")
print("=" * 70)


warehouse_tables = {

    "DIM_CANDIDATE":
        dim_candidate,

    "DIM_EXAM":
        dim_exam,

    "DIM_ITEM":
        dim_item,

    "DIM_REVIEWER":
        dim_reviewer,

    "DIM_DATE":
        dim_date,

    "DIM_FLAG_TYPE":
        dim_flag_type,

    "FACT_RESPONSE":
        fact_response,

    "FACT_SESSION_RISK":
        fact_session_risk,

    "FACT_REVIEW":
        fact_review
}


for table_name, df in warehouse_tables.items():

    print(
        f"{table_name:<25}"
        f"{len(df):>10,} rows"
    )


# ============================================================
# 14. FOREIGN KEY VALIDATION
# ============================================================

print("\n")
print("=" * 70)
print("FOREIGN KEY VALIDATION")
print("=" * 70)


fk_checks = {}


# FACT_RESPONSE

fk_checks[
    "FACT_RESPONSE → DIM_CANDIDATE"
] = (
    fact_response["candidate_key"]
    .notna()
    .all()
    and
    fact_response["candidate_key"]
    .isin(
        dim_candidate["candidate_key"]
    )
    .all()
)


fk_checks[
    "FACT_RESPONSE → DIM_ITEM"
] = (
    fact_response["item_key"]
    .notna()
    .all()
    and
    fact_response["item_key"]
    .isin(
        dim_item["item_key"]
    )
    .all()
)


fk_checks[
    "FACT_RESPONSE → DIM_EXAM"
] = (
    fact_response["exam_key"]
    .notna()
    .all()
    and
    fact_response["exam_key"]
    .isin(
        dim_exam["exam_key"]
    )
    .all()
)


fk_checks[
    "FACT_RESPONSE → DIM_DATE"
] = (
    fact_response["date_key"]
    .isin(
        dim_date["date_key"]
    )
    .all()
)


# FACT_SESSION_RISK

fk_checks[
    "FACT_SESSION_RISK → DIM_CANDIDATE"
] = (
    fact_session_risk["candidate_key"]
    .notna()
    .all()
    and
    fact_session_risk["candidate_key"]
    .isin(
        dim_candidate["candidate_key"]
    )
    .all()
)


fk_checks[
    "FACT_SESSION_RISK → DIM_EXAM"
] = (
    fact_session_risk["exam_key"]
    .notna()
    .all()
    and
    fact_session_risk["exam_key"]
    .isin(
        dim_exam["exam_key"]
    )
    .all()
)


fk_checks[
    "FACT_SESSION_RISK → DIM_DATE"
] = (
    fact_session_risk["date_key"]
    .isin(
        dim_date["date_key"]
    )
    .all()
)


# FACT_REVIEW

fk_checks[
    "FACT_REVIEW → DIM_CANDIDATE"
] = (
    fact_review["candidate_key"]
    .notna()
    .all()
    and
    fact_review["candidate_key"]
    .isin(
        dim_candidate["candidate_key"]
    )
    .all()
)


fk_checks[
    "FACT_REVIEW → DIM_EXAM"
] = (
    fact_review["exam_key"]
    .notna()
    .all()
    and
    fact_review["exam_key"]
    .isin(
        dim_exam["exam_key"]
    )
    .all()
)


fk_checks[
    "FACT_REVIEW → DIM_REVIEWER"
] = (
    fact_review["reviewer_key"]
    .notna()
    .all()
    and
    fact_review["reviewer_key"]
    .isin(
        dim_reviewer["reviewer_key"]
    )
    .all()
)


# FLAG KEY
# Nullable is allowed because the source may not contain
# a review-level flag relationship.

non_null_flags = (
    fact_review["flag_key"]
    .dropna()
)

if len(non_null_flags) == 0:

    fk_checks[
        "FACT_REVIEW → DIM_FLAG_TYPE"
    ] = True

else:

    fk_checks[
        "FACT_REVIEW → DIM_FLAG_TYPE"
    ] = (
        non_null_flags
        .isin(
            dim_flag_type["flag_key"]
        )
        .all()
    )


for check_name, result in fk_checks.items():

    print(
        f"{check_name:<50}"
        f"{'PASS' if result else 'FAIL'}"
    )


# ============================================================
# 15. TYPE-2 SCD VALIDATION
# ============================================================

print("\n")
print("=" * 70)
print("TYPE-2 SCD VALIDATION")
print("=" * 70)


scd_exam_columns = [
    "exam_key",
    "exam_id",
    "exam_version",
    "valid_from",
    "valid_to",
    "is_current"
]

scd_reviewer_columns = [
    "reviewer_key",
    "reviewer_id",
    "reviewer_version",
    "valid_from",
    "valid_to",
    "is_current"
]


exam_scd_ok = all(
    column in dim_exam.columns
    for column in scd_exam_columns
)


reviewer_scd_ok = all(
    column in dim_reviewer.columns
    for column in scd_reviewer_columns
)


print(
    "DIM_EXAM Type-2:",
    "PASS" if exam_scd_ok else "FAIL"
)

print(
    "DIM_REVIEWER Type-2:",
    "PASS" if reviewer_scd_ok else "FAIL"
)


# ============================================================
# 16. DR-5 REQUIRED COLUMN VALIDATION
# ============================================================

print("\n")
print("=" * 70)
print("DR-5 COLUMN VALIDATION")
print("=" * 70)


dr5_required = {

    "FACT_RESPONSE": [

        "response_key",
        "candidate_key",
        "item_key",
        "exam_key",
        "date_key",
        "selected_option",
        "is_correct",
        "response_time_ms",
        "sequence_position",
        "revisions"

    ],

    "FACT_SESSION_RISK": [

        "session_key",
        "candidate_key",
        "exam_key",
        "date_key",
        "risk_score",
        "recommendation",
        "flag_count",
        "channels_missing",
        "model_version"

    ],

    "FACT_REVIEW": [

        "review_key",
        "session_key",
        "reviewer_key",
        "flag_key",
        "verdict",
        "confidence",
        "notes_length"

    ]
}


dr5_checks = {}


for table_name, required_columns in dr5_required.items():

    missing = [
        column
        for column in required_columns
        if column not in warehouse_tables[
            table_name
        ].columns
    ]

    dr5_checks[table_name] = (
        len(missing) == 0
    )

    if missing:

        print(
            f"{table_name}: FAIL"
        )

        print(
            "Missing:",
            missing
        )

    else:

        print(
            f"{table_name}: PASS"
        )


# ============================================================
# 17. DATA LINEAGE DOCUMENTATION
# ============================================================

lineage = pd.DataFrame([

    {
        "target_table":
            "DIM_CANDIDATE",

        "source_table":
            "candidates.csv",

        "transformation":
            "Surrogate candidate_key added",

        "grain":
            "One row per candidate"
    },

    {
        "target_table":
            "DIM_EXAM",

        "source_table":
            "items.csv",

        "transformation":
            "Distinct exam_id + Type-2 SCD structure",

        "grain":
            "One row per exam version"
    },

    {
        "target_table":
            "DIM_ITEM",

        "source_table":
            "items.csv",

        "transformation":
            "Surrogate item_key added",

        "grain":
            "One row per item"
    },

    {
        "target_table":
            "DIM_REVIEWER",

        "source_table":
            "reviewers.csv",

        "transformation":
            "Surrogate reviewer_key + Type-2 SCD structure",

        "grain":
            "One row per reviewer version"
    },

    {
        "target_table":
            "DIM_DATE",

        "source_table":
            "Warehouse reference date",

        "transformation":
            "Date attributes generated",

        "grain":
            "One row per date"
    },

    {
        "target_table":
            "DIM_FLAG_TYPE",

        "source_table":
            "proctoring_flags.csv",

        "transformation":
            "Distinct flag_type values + flag_key",

        "grain":
            "One row per flag type"
    },

    {
        "target_table":
            "FACT_RESPONSE",

        "source_table":
            "exam_responses.csv",

        "transformation":
            "Candidate, item, exam and date surrogate keys added",

        "grain":
            "One row per candidate response"
    },

    {
        "target_table":
            "FACT_SESSION_RISK",

        "source_table":
            "session_risk.csv",

        "transformation":
            "Session, candidate, exam and date keys added; risk_score derived from overall_risk",

        "grain":
            "One row per session"
    },

    {
        "target_table":
            "FACT_REVIEW",

        "source_table":
            "review_verdicts.csv",

        "transformation":
            "Session, candidate, exam and reviewer keys added; reviewer_confidence mapped to confidence; flag relationship mapped when available",

        "grain":
            "One row per review"
    }

])


lineage.to_csv(
    LINEAGE_DIR /
    "data_lineage.csv",
    index=False
)


print(
    "\nData lineage created:"
)

print(
    LINEAGE_DIR /
    "data_lineage.csv"
)


# ============================================================
# 18. STAR SCHEMA METADATA
# ============================================================

schema_metadata = {

    "project":
        "IntegrityInsight",

    "layer":
        "FR-7 Transformation Layer",

    "dimensions": [

        "DIM_CANDIDATE",
        "DIM_EXAM",
        "DIM_ITEM",
        "DIM_REVIEWER",
        "DIM_DATE",
        "DIM_FLAG_TYPE"

    ],

    "facts": [

        "FACT_RESPONSE",
        "FACT_SESSION_RISK",
        "FACT_REVIEW"

    ],

    "scd_type_2": [

        "DIM_EXAM",
        "DIM_REVIEWER"

    ],

    "lineage":
        "lineage/data_lineage.csv"

}


with open(
    WAREHOUSE_DIR /
    "star_schema_metadata.json",
    "w"
) as f:

    json.dump(
        schema_metadata,
        f,
        indent=4
    )


print(
    "\nStar schema metadata created."
)


# ============================================================
# 19. FINAL FR-7 STATUS
# ============================================================

all_fk_passed = all(
    fk_checks.values()
)

all_dr5_passed = all(
    dr5_checks.values()
)

fr7_passed = (

    len(warehouse_tables) == 9

    and all_fk_passed

    and exam_scd_ok

    and reviewer_scd_ok

    and all_dr5_passed

    and len(lineage) == 9

)


print("\n")
print("=" * 70)
print("FR-7 FINAL STATUS")
print("=" * 70)


if fr7_passed:

    print(
        "FR-7 TRANSFORMATION STATUS: PASSED"
    )

else:

    print(
        "FR-7 TRANSFORMATION STATUS: CHECK REQUIRED"
    )


print("\nWarehouse:")
print(WAREHOUSE_DIR)

print("\nDimensions:")

for file in sorted(
    DIM_DIR.glob("*.csv")
):

    print(
        " -",
        file.name
    )


print("\nFacts:")

for file in sorted(
    FACT_DIR.glob("*.csv")
):

    print(
        " -",
        file.name
    )


print("\nLineage:")
print(
    " - data_lineage.csv"
)

print("\nMetadata:")
print(
    " - star_schema_metadata.json"
)


INTEGRITYINSIGHT — FR-7 TRANSFORMATION LAYER
STAGING   : /content/integrityinsight_data/staging
WAREHOUSE : /content/integrityinsight_data/warehouse

Staging tables loaded successfully.

Staging row counts:
candidates                       300
items                          1,000
sessions                       6,000
exam_responses               300,000
session_risk                   6,000
proctoring_flags               3,450
review_verdicts                3,795
reviewers                         12
infrastructure_cost            6,000

DIM_CANDIDATE: 300 rows
DIM_ITEM: 1000 rows
DIM_EXAM: 20 rows
DIM_EXAM Type-2 structure created.
DIM_REVIEWER: 12 rows
DIM_REVIEWER Type-2 structure created.
DIM_FLAG_TYPE: 8 rows
DIM_DATE: 1 row

Session key mapping: 6000 sessions
FACT_RESPONSE: 300000 rows
FACT_SESSION_RISK: 6000 rows
FACT_REVIEW: 3795 rows


STAR SCHEMA VALIDATION
DIM_CANDIDATE                   300 rows
DIM_EXAM                         20 rows
DIM_ITEM                      1,000 rows


In [ ]:
# =====================================================================
# INTEGRITYINSIGHT — FR-8 DATA QUALITY TEST SUITE
# =====================================================================
# FR-8 — Data Quality
#
# Current DR-5 Star Schema:
#
# DIMENSIONS:
#   DIM_CANDIDATE
#   DIM_EXAM
#   DIM_ITEM
#   DIM_REVIEWER
#   DIM_FLAG_TYPE
#   DIM_DATE
#
# FACTS:
#   FACT_RESPONSE
#   FACT_SESSION_RISK
#   FACT_REVIEW
#
# IMPORTANT:
#
#   - FACT_REVIEW contains flag_key
#   - confidence = low / medium / high
#   - notes_length is currently NULL for all reviews
# =====================================================================

from pathlib import Path
from datetime import datetime
import json
import numpy as np
import pandas as pd


# =====================================================================
# 1. PATHS
# =====================================================================

DATA_DIR = Path("/content/integrityinsight_data")

if not DATA_DIR.exists():
    DATA_DIR = Path.cwd() / "integrityinsight_data"

WAREHOUSE_DIR = DATA_DIR / "warehouse"
DIM_DIR = WAREHOUSE_DIR / "dimensions"
FACT_DIR = WAREHOUSE_DIR / "facts"

VALIDATION_DIR = DATA_DIR / "validation"
DQ_DIR = VALIDATION_DIR / "data_quality"

DQ_DIR.mkdir(parents=True, exist_ok=True)


print("=" * 70)
print("INTEGRITYINSIGHT — FR-8 DATA QUALITY TEST SUITE")
print("=" * 70)

print(f"DATA_DIR      : {DATA_DIR}")
print(f"WAREHOUSE_DIR : {WAREHOUSE_DIR}")
print(f"DIM_DIR       : {DIM_DIR}")
print(f"FACT_DIR      : {FACT_DIR}")
print(f"DQ OUTPUT     : {DQ_DIR}")
print()


# =====================================================================
# 2. LOAD WAREHOUSE
# =====================================================================

def load_csv(folder, filename):

    path = folder / filename

    if not path.exists():
        raise FileNotFoundError(
            f"Required warehouse file not found:\n{path}"
        )

    return pd.read_csv(path)


warehouse = {

    # ---------------------------------------------------------------
    # Dimensions
    # ---------------------------------------------------------------

    "dim_candidate":
        load_csv(DIM_DIR, "dim_candidate.csv"),

    "dim_exam":
        load_csv(DIM_DIR, "dim_exam.csv"),

    "dim_item":
        load_csv(DIM_DIR, "dim_item.csv"),

    "dim_reviewer":
        load_csv(DIM_DIR, "dim_reviewer.csv"),

    "dim_flag_type":
        load_csv(DIM_DIR, "dim_flag_type.csv"),

    "dim_date":
        load_csv(DIM_DIR, "dim_date.csv"),

    # ---------------------------------------------------------------
    # Facts
    # ---------------------------------------------------------------

    "fact_response":
        load_csv(FACT_DIR, "fact_response.csv"),

    "fact_session_risk":
        load_csv(FACT_DIR, "fact_session_risk.csv"),

    "fact_review":
        load_csv(FACT_DIR, "fact_review.csv"),
}


print("Warehouse tables loaded successfully:")

for name, df in warehouse.items():

    print(
        f"  {name:<25} "
        f"{len(df):>10,} rows"
    )

print()


# =====================================================================
# 3. RESULTS STORAGE
# =====================================================================

results = []


def add_result(
    test_id,
    category,
    table,
    column,
    test_name,
    passed,
    invalid_count=0,
    details=""
):

    results.append({

        "test_id": test_id,

        "category": category,

        "table": table,

        "column": column,

        "test_name": test_name,

        "status":
            "PASS" if passed else "FAIL",

        "invalid_count":
            int(invalid_count)
            if isinstance(
                invalid_count,
                (int, np.integer)
            )
            else invalid_count,

        "details":
            str(details)
    })


# =====================================================================
# 4. HELPER FUNCTIONS
# =====================================================================

def test_not_null(df, column):

    if column not in df.columns:
        return False, len(df)

    invalid = int(
        df[column].isna().sum()
    )

    return invalid == 0, invalid


def test_unique(df, column):

    if column not in df.columns:
        return False, len(df)

    invalid = int(
        df[column]
        .duplicated(keep=False)
        .sum()
    )

    return invalid == 0, invalid


def test_range(
    df,
    column,
    min_value=None,
    max_value=None
):
    """
    Robust numeric range check.

    Numeric strings such as:
        "0.5"
        "0.85"

    are accepted.

    Non-numeric values are detected as DQ failures.
    """

    if column not in df.columns:
        return False, len(df)

    numeric_values = pd.to_numeric(
        df[column],
        errors="coerce"
    )

    non_numeric_mask = (
        df[column].notna()
        &
        numeric_values.isna()
    )

    invalid_mask = non_numeric_mask.copy()

    if min_value is not None:

        invalid_mask |= (
            numeric_values.notna()
            &
            (numeric_values < min_value)
        )

    if max_value is not None:

        invalid_mask |= (
            numeric_values.notna()
            &
            (numeric_values > max_value)
        )

    invalid_count = int(
        invalid_mask.sum()
    )

    return invalid_count == 0, invalid_count


def test_accepted_values(
    df,
    column,
    allowed_values
):

    if column not in df.columns:
        return False, len(df)

    invalid_mask = ~df[column].isin(
        allowed_values
    )

    invalid_count = int(
        invalid_mask.sum()
    )

    return invalid_count == 0, invalid_count


def test_fk(
    fact_df,
    fact_column,
    dim_df,
    dim_column
):

    if fact_column not in fact_df.columns:
        return False, len(fact_df)

    if dim_column not in dim_df.columns:
        return False, len(fact_df)

    valid_keys = set(
        dim_df[dim_column]
        .dropna()
        .tolist()
    )

    invalid_mask = (

        fact_df[fact_column].notna()

        &

        ~fact_df[fact_column].isin(
            valid_keys
        )
    )

    invalid_count = int(
        invalid_mask.sum()
    )

    return invalid_count == 0, invalid_count


def test_row_count(
    df,
    expected
):

    actual = len(df)

    invalid = abs(
        actual - expected
    )

    return actual == expected, invalid


def test_non_negative(
    df,
    column
):

    if column not in df.columns:
        return False, len(df)

    # ---------------------------------------------------------------
    # Important:
    # If the column is completely NULL, there is no numeric
    # observation to validate.
    #
    # We don't classify the NULLs themselves as negative values.
    # NULL completeness is handled separately.
    # ---------------------------------------------------------------

    numeric_values = pd.to_numeric(
        df[column],
        errors="coerce"
    )

    non_numeric_mask = (
        df[column].notna()
        &
        numeric_values.isna()
    )

    negative_mask = (
        numeric_values.notna()
        &
        (numeric_values < 0)
    )

    invalid_mask = (
        non_numeric_mask
        |
        negative_mask
    )

    invalid_count = int(
        invalid_mask.sum()
    )

    return invalid_count == 0, invalid_count


def test_column_exists(
    df,
    required_columns
):

    missing = [
        c
        for c in required_columns
        if c not in df.columns
    ]

    return (
        len(missing) == 0,
        len(missing),
        missing
    )


# =====================================================================
# 5. TEST RUNNER
# =====================================================================

test_number = 1


def RUN(
    category,
    table,
    column,
    test_name,
    passed,
    invalid_count=0,
    details=""
):

    global test_number

    test_id = f"DQT-{test_number:03d}"

    add_result(
        test_id=test_id,
        category=category,
        table=table,
        column=column,
        test_name=test_name,
        passed=passed,
        invalid_count=invalid_count,
        details=details
    )

    test_number += 1


# =====================================================================
# 6. REQUIRED TABLE EXISTENCE
# =====================================================================

required_tables = [

    "dim_candidate",
    "dim_exam",
    "dim_item",
    "dim_reviewer",
    "dim_flag_type",
    "dim_date",

    "fact_response",
    "fact_session_risk",
    "fact_review",
]


for table_name in required_tables:

    exists = table_name in warehouse

    RUN(
        "TABLE_EXISTENCE",
        table_name,
        "",
        "Required warehouse table exists",
        exists,
        0 if exists else 1
    )


# =====================================================================
# 7. EXPECTED ROW COUNTS
# =====================================================================

EXPECTED_ROW_COUNTS = {

    "dim_candidate": 300,

    "dim_exam": 20,

    "dim_item": 1000,

    "dim_reviewer": 12,

    "dim_flag_type": 8,

    "dim_date": 1,

    "fact_response": 300000,

    "fact_session_risk": 6000,

    "fact_review": 3795,
}


for table_name, expected_count in EXPECTED_ROW_COUNTS.items():

    passed, invalid = test_row_count(
        warehouse[table_name],
        expected_count
    )

    RUN(
        "ROW_COUNT",
        table_name,
        "",
        f"Expected row count = {expected_count:,}",
        passed,
        invalid,
        f"Actual rows = {len(warehouse[table_name]):,}"
    )


# =====================================================================
# 8. REQUIRED DR-5 COLUMNS
# =====================================================================

REQUIRED_COLUMNS = {

    # ---------------------------------------------------------------
    # Dimensions
    # ---------------------------------------------------------------

    "dim_candidate": [

        "candidate_key",
        "candidate_id",
    ],

    "dim_exam": [

        "exam_key",
        "exam_id",
    ],

    "dim_item": [

        "item_key",
        "item_id",
    ],

    "dim_reviewer": [

        "reviewer_key",
        "reviewer_id",
    ],

    "dim_flag_type": [

        "flag_key",
        "flag_type",
    ],

    "dim_date": [

        "date_key",
    ],

    # ---------------------------------------------------------------
    # FACT_RESPONSE
    # ---------------------------------------------------------------

    "fact_response": [

        "response_key",

        "candidate_key",

        "item_key",

        "exam_key",

        "date_key",

        "selected_option",

        "is_correct",

        "response_time_ms",

        "sequence_position",

        "revisions",
    ],

    # ---------------------------------------------------------------
    # FACT_SESSION_RISK
    # ---------------------------------------------------------------

    "fact_session_risk": [

        "session_key",

        "candidate_key",

        "exam_key",

        "date_key",

        "risk_score",

        "recommendation",

        "flag_count",

        "channels_missing",

        "model_version",
    ],

    # ---------------------------------------------------------------
    # FACT_REVIEW
    # ---------------------------------------------------------------

    "fact_review": [

        "review_key",

        "session_key",

        "reviewer_key",

        "flag_key",

        "verdict",

        "confidence",

        "notes_length",
    ],
}


for table_name, required_cols in REQUIRED_COLUMNS.items():

    passed, missing_count, missing_columns = test_column_exists(
        warehouse[table_name],
        required_cols
    )

    RUN(
        "SCHEMA",
        table_name,
        "",
        "Required DR-5 columns exist",
        passed,
        missing_count,
        f"Missing columns: {missing_columns}"
    )


# =====================================================================
# 9. NOT NULL TESTS
# =====================================================================

NOT_NULL_COLUMNS = {

    "dim_candidate": [

        "candidate_key",
        "candidate_id",
    ],

    "dim_exam": [

        "exam_key",
        "exam_id",
    ],

    "dim_item": [

        "item_key",
        "item_id",
    ],

    "dim_reviewer": [

        "reviewer_key",
        "reviewer_id",
    ],

    "dim_flag_type": [

        "flag_key",
        "flag_type",
    ],

    "dim_date": [

        "date_key",
    ],

    "fact_response": [

        "response_key",
        "candidate_key",
        "item_key",
        "exam_key",
        "date_key",
    ],

    "fact_session_risk": [

        "session_key",
        "candidate_key",
        "exam_key",
        "date_key",
        "risk_score",
        "recommendation",
    ],

    "fact_review": [

        "review_key",
        "session_key",
        "reviewer_key",
        "flag_key",
        "verdict",
        "confidence",
    ],
}


for table_name, columns in NOT_NULL_COLUMNS.items():

    df = warehouse[table_name]

    for column in columns:

        passed, invalid = test_not_null(
            df,
            column
        )

        RUN(
            "NOT_NULL",
            table_name,
            column,
            "Required field is not NULL",
            passed,
            invalid
        )


# =====================================================================
# 10. UNIQUENESS TESTS
# =====================================================================

UNIQUE_COLUMNS = {

    "dim_candidate": [

        "candidate_key",
        "candidate_id",
    ],

    "dim_exam": [

        "exam_key",
        "exam_id",
    ],

    "dim_item": [

        "item_key",
        "item_id",
    ],

    "dim_reviewer": [

        "reviewer_key",
        "reviewer_id",
    ],

    "dim_flag_type": [

        "flag_key",
        "flag_type",
    ],

    "dim_date": [

        "date_key",
    ],

    "fact_response": [

        "response_key",
    ],

    "fact_session_risk": [

        "session_key",
    ],

    "fact_review": [

        "review_key",
    ],
}


for table_name, columns in UNIQUE_COLUMNS.items():

    df = warehouse[table_name]

    for column in columns:

        passed, invalid = test_unique(
            df,
            column
        )

        RUN(
            "UNIQUENESS",
            table_name,
            column,
            "Key / identifier is unique",
            passed,
            invalid
        )


# =====================================================================
# 11. FOREIGN KEY TESTS
# =====================================================================

FK_TESTS = [

    # FACT_RESPONSE
    (
        "fact_response",
        "candidate_key",
        "dim_candidate",
        "candidate_key"
    ),

    (
        "fact_response",
        "item_key",
        "dim_item",
        "item_key"
    ),

    (
        "fact_response",
        "exam_key",
        "dim_exam",
        "exam_key"
    ),

    (
        "fact_response",
        "date_key",
        "dim_date",
        "date_key"
    ),

    # FACT_SESSION_RISK
    (
        "fact_session_risk",
        "candidate_key",
        "dim_candidate",
        "candidate_key"
    ),

    (
        "fact_session_risk",
        "exam_key",
        "dim_exam",
        "exam_key"
    ),

    (
        "fact_session_risk",
        "date_key",
        "dim_date",
        "date_key"
    ),

    # FACT_REVIEW
    (
        "fact_review",
        "reviewer_key",
        "dim_reviewer",
        "reviewer_key"
    ),

    (
        "fact_review",
        "flag_key",
        "dim_flag_type",
        "flag_key"
    ),
]


for (
    fact_table,
    fact_column,
    dim_table,
    dim_column
) in FK_TESTS:

    passed, invalid = test_fk(
        warehouse[fact_table],
        fact_column,
        warehouse[dim_table],
        dim_column
    )

    RUN(
        "FOREIGN_KEY",
        fact_table,
        fact_column,
        f"FK -> {dim_table}.{dim_column}",
        passed,
        invalid
    )


# ---------------------------------------------------------------------
# Optional FACT_REVIEW candidate/exam FK checks
# ---------------------------------------------------------------------

OPTIONAL_FK_TESTS = [

    (
        "fact_review",
        "candidate_key",
        "dim_candidate",
        "candidate_key"
    ),

    (
        "fact_review",
        "exam_key",
        "dim_exam",
        "exam_key"
    ),
]


for (
    fact_table,
    fact_column,
    dim_table,
    dim_column
) in OPTIONAL_FK_TESTS:

    if fact_column in warehouse[fact_table].columns:

        passed, invalid = test_fk(
            warehouse[fact_table],
            fact_column,
            warehouse[dim_table],
            dim_column
        )

        RUN(
            "FOREIGN_KEY",
            fact_table,
            fact_column,
            f"Optional FK -> {dim_table}.{dim_column}",
            passed,
            invalid
        )


# =====================================================================
# 12. ACCEPTED VALUES
# =====================================================================

# ---------------------------------------------------------------------
# FACT_RESPONSE.selected_option
# ---------------------------------------------------------------------

if "selected_option" in warehouse["fact_response"].columns:

    passed, invalid = test_accepted_values(

        warehouse["fact_response"],

        "selected_option",

        {
            "A",
            "B",
            "C",
            "D"
        }
    )

    RUN(
        "ACCEPTED_VALUES",
        "fact_response",
        "selected_option",
        "Selected option must be A/B/C/D",
        passed,
        invalid
    )


# ---------------------------------------------------------------------
# FACT_RESPONSE.is_correct
# ---------------------------------------------------------------------

if "is_correct" in warehouse["fact_response"].columns:

    normalized = (
        warehouse["fact_response"]["is_correct"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    invalid_mask = ~normalized.isin(
        {
            "0",
            "1",
            "true",
            "false"
        }
    )

    invalid = int(
        invalid_mask.sum()
    )

    RUN(
        "ACCEPTED_VALUES",
        "fact_response",
        "is_correct",
        "is_correct must be binary",
        invalid == 0,
        invalid
    )


# ---------------------------------------------------------------------
# FACT_SESSION_RISK.recommendation
# ---------------------------------------------------------------------

if "recommendation" in warehouse["fact_session_risk"].columns:

    allowed_recommendations = {

        "NO_ACTION",

        "ROUTINE_REVIEW",

        "HUMAN_REVIEW",

        "PRIORITY_REVIEW",
    }

    passed, invalid = test_accepted_values(

        warehouse["fact_session_risk"],

        "recommendation",

        allowed_recommendations
    )

    RUN(
        "ACCEPTED_VALUES",
        "fact_session_risk",
        "recommendation",
        "Recommendation uses approved values",
        passed,
        invalid
    )


# ---------------------------------------------------------------------
# FACT_REVIEW.verdict
# ---------------------------------------------------------------------

if "verdict" in warehouse["fact_review"].columns:

    allowed_verdicts = {

        "CONFIRMED",

        "OVERTURNED",

        "INCONCLUSIVE",

        "TECHNICAL_ISSUE",

        "ESCALATED",
    }

    passed, invalid = test_accepted_values(

        warehouse["fact_review"],

        "verdict",

        allowed_verdicts
    )

    RUN(
        "ACCEPTED_VALUES",
        "fact_review",
        "verdict",
        "Review verdict uses approved values",
        passed,
        invalid
    )


# ---------------------------------------------------------------------
# FACT_REVIEW.confidence
#
# CURRENT SOURCE FORMAT:
#   low
#   medium
#   high
# ---------------------------------------------------------------------

if "confidence" in warehouse["fact_review"].columns:

    allowed_confidence = {

        "low",

        "medium",

        "high",
    }

    passed, invalid = test_accepted_values(

        warehouse["fact_review"],

        "confidence",

        allowed_confidence
    )

    RUN(
        "ACCEPTED_VALUES",
        "fact_review",
        "confidence",
        "Review confidence uses approved categorical values",
        passed,
        invalid
    )


# ---------------------------------------------------------------------
# DIM_FLAG_TYPE.flag_type
# ---------------------------------------------------------------------

if "flag_type" in warehouse["dim_flag_type"].columns:

    allowed_flag_types = {

        "TAB_SWITCH",

        "SECOND_VOICE",

        "IDENTITY_MISMATCH",

        "PHONE_DETECTED",

        "GAZE_OFF_SCREEN",

        "MULTI_PERSON",

        "MULTI_MONITOR",

        "REMOTE_TOOL",
    }

    passed, invalid = test_accepted_values(

        warehouse["dim_flag_type"],

        "flag_type",

        allowed_flag_types
    )

    RUN(
        "ACCEPTED_VALUES",
        "dim_flag_type",
        "flag_type",
        "Flag type uses approved values",
        passed,
        invalid
    )


# =====================================================================
# 13. NUMERIC RANGE TESTS
# =====================================================================

# ---------------------------------------------------------------------
# FACT_SESSION_RISK.risk_score
# ---------------------------------------------------------------------

if "risk_score" in warehouse["fact_session_risk"].columns:

    passed, invalid = test_range(

        warehouse["fact_session_risk"],

        "risk_score",

        0,

        1
    )

    RUN(
        "RANGE",
        "fact_session_risk",
        "risk_score",
        "Risk score must be between 0 and 1",
        passed,
        invalid
    )


# ---------------------------------------------------------------------
# FACT_SESSION_RISK.overall_risk
# ---------------------------------------------------------------------

if "overall_risk" in warehouse["fact_session_risk"].columns:

    passed, invalid = test_range(

        warehouse["fact_session_risk"],

        "overall_risk",

        0,

        1
    )

    RUN(
        "RANGE",
        "fact_session_risk",
        "overall_risk",
        "Overall risk must be between 0 and 1",
        passed,
        invalid
    )


# ---------------------------------------------------------------------
# FACT_RESPONSE.response_time_ms
# ---------------------------------------------------------------------

if "response_time_ms" in warehouse["fact_response"].columns:

    passed, invalid = test_non_negative(

        warehouse["fact_response"],

        "response_time_ms"
    )

    RUN(
        "RANGE",
        "fact_response",
        "response_time_ms",
        "Response time must be non-negative",
        passed,
        invalid
    )


# ---------------------------------------------------------------------
# FACT_RESPONSE.revisions
# ---------------------------------------------------------------------

if "revisions" in warehouse["fact_response"].columns:

    passed, invalid = test_non_negative(

        warehouse["fact_response"],

        "revisions"
    )

    RUN(
        "RANGE",
        "fact_response",
        "revisions",
        "Revision count must be non-negative",
        passed,
        invalid
    )


# ---------------------------------------------------------------------
# FACT_RESPONSE.sequence_position
# ---------------------------------------------------------------------

if "sequence_position" in warehouse["fact_response"].columns:

    passed, invalid = test_non_negative(

        warehouse["fact_response"],

        "sequence_position"
    )

    RUN(
        "RANGE",
        "fact_response",
        "sequence_position",
        "Sequence position must be non-negative",
        passed,
        invalid
    )


# ---------------------------------------------------------------------
# FACT_SESSION_RISK.flag_count
# ---------------------------------------------------------------------

if "flag_count" in warehouse["fact_session_risk"].columns:

    passed, invalid = test_non_negative(

        warehouse["fact_session_risk"],

        "flag_count"
    )

    RUN(
        "RANGE",
        "fact_session_risk",
        "flag_count",
        "Flag count must be non-negative",
        passed,
        invalid
    )


# ---------------------------------------------------------------------
# FACT_REVIEW.notes_length
#
# Currently all values are NULL.
#
# We only validate actual values if they exist.
# NULL completeness is handled separately.
# ---------------------------------------------------------------------

if "notes_length" in warehouse["fact_review"].columns:

    non_null_count = (
        warehouse["fact_review"]["notes_length"]
        .notna()
        .sum()
    )

    if non_null_count > 0:

        passed, invalid = test_non_negative(

            warehouse["fact_review"],

            "notes_length"
        )

        RUN(
            "RANGE",
            "fact_review",
            "notes_length",
            "Notes length must be non-negative",
            passed,
            invalid
        )

    else:

        RUN(
            "DATA_AVAILABILITY",
            "fact_review",
            "notes_length",
            "Notes length has usable observations",
            True,
            0,
            "All current values are NULL; no numeric range test applied."
        )


# =====================================================================
# 14. OPTIONAL REVIEW NUMERIC TESTS
# =====================================================================

# review_seconds

if "review_seconds" in warehouse["fact_review"].columns:

    non_null_count = (
        warehouse["fact_review"]["review_seconds"]
        .notna()
        .sum()
    )

    if non_null_count > 0:

        passed, invalid = test_non_negative(

            warehouse["fact_review"],

            "review_seconds"
        )

        RUN(
            "RANGE",
            "fact_review",
            "review_seconds",
            "Review duration must be non-negative",
            passed,
            invalid
        )


# position_in_shift

if "position_in_shift" in warehouse["fact_review"].columns:

    non_null_count = (
        warehouse["fact_review"]["position_in_shift"]
        .notna()
        .sum()
    )

    if non_null_count > 0:

        passed, invalid = test_non_negative(

            warehouse["fact_review"],

            "position_in_shift"
        )

        RUN(
            "RANGE",
            "fact_review",
            "position_in_shift",
            "Reviewer shift position must be non-negative",
            passed,
            invalid
        )


# =====================================================================
# 15. SCD TYPE-2 TESTS
# =====================================================================

def check_scd2(
    df,
    key_column,
    start_column,
    end_column=None
):

    if key_column not in df.columns:

        return (
            False,
            len(df),
            "Missing key column"
        )

    if start_column not in df.columns:

        return (
            False,
            1,
            "SCD start column not found"
        )

    duplicate_current = 0

    if (
        end_column
        and
        end_column in df.columns
    ):

        current_mask = df[end_column].isna()

        current_counts = (
            df.loc[current_mask]
            .groupby(key_column)
            .size()
        )

        duplicate_current = int(
            (current_counts > 1).sum()
        )

    return (

        duplicate_current == 0,

        duplicate_current,

        f"Multiple current records = "
        f"{duplicate_current}"
    )


# ---------------------------------------------------------------------
# DIM_EXAM SCD2
# ---------------------------------------------------------------------

exam_scd_start = None
exam_scd_end = None

for col in warehouse["dim_exam"].columns:

    lower = col.lower()

    if lower in {
        "valid_from",
        "effective_from",
        "start_date"
    }:

        exam_scd_start = col

    if lower in {
        "valid_to",
        "effective_to",
        "end_date"
    }:

        exam_scd_end = col


if exam_scd_start:

    passed, invalid, details = check_scd2(

        warehouse["dim_exam"],

        "exam_id",

        exam_scd_start,

        exam_scd_end
    )

    RUN(
        "SCD2",
        "dim_exam",
        "",
        "DIM_EXAM Type-2 SCD has at most one current record",
        passed,
        invalid,
        details
    )

else:

    RUN(
        "SCD2",
        "dim_exam",
        "",
        "DIM_EXAM Type-2 SCD structure",
        False,
        1,
        "No valid_from/effective_from column detected"
    )


# ---------------------------------------------------------------------
# DIM_REVIEWER SCD2
# ---------------------------------------------------------------------

reviewer_scd_start = None
reviewer_scd_end = None

for col in warehouse["dim_reviewer"].columns:

    lower = col.lower()

    if lower in {
        "valid_from",
        "effective_from",
        "start_date"
    }:

        reviewer_scd_start = col

    if lower in {
        "valid_to",
        "effective_to",
        "end_date"
    }:

        reviewer_scd_end = col


if reviewer_scd_start:

    passed, invalid, details = check_scd2(

        warehouse["dim_reviewer"],

        "reviewer_id",

        reviewer_scd_start,

        reviewer_scd_end
    )

    RUN(
        "SCD2",
        "dim_reviewer",
        "",
        "DIM_REVIEWER Type-2 SCD has at most one current record",
        passed,
        invalid,
        details
    )

else:

    RUN(
        "SCD2",
        "dim_reviewer",
        "",
        "DIM_REVIEWER Type-2 SCD structure",
        False,
        1,
        "No valid_from/effective_from column detected"
    )


# =====================================================================
# 16. DISTRIBUTION / STATISTICAL SANITY TESTS
# =====================================================================
#
# IMPORTANT:
# confidence is categorical -> NOT included
#
# notes_length is completely NULL -> NOT included
# =====================================================================

def distribution_test(
    df,
    column
):

    if column not in df.columns:

        return (
            False,
            len(df),
            "Column not found"
        )

    values = pd.to_numeric(
        df[column],
        errors="coerce"
    ).dropna()

    if len(values) == 0:

        return (
            False,
            len(df),
            "No numeric observations"
        )

    finite = np.isfinite(values)

    invalid = int(
        (~finite).sum()
    )

    if invalid > 0:

        return (
            False,
            invalid,
            "Non-finite numeric values detected"
        )

    return (
        True,
        0,
        f"n={len(values):,}"
    )


DISTRIBUTION_COLUMNS = [

    (
        "fact_response",
        "response_time_ms"
    ),

    (
        "fact_response",
        "sequence_position"
    ),

    (
        "fact_response",
        "revisions"
    ),

    (
        "fact_session_risk",
        "risk_score"
    ),

    (
        "fact_session_risk",
        "flag_count"
    ),
]


for table_name, column in DISTRIBUTION_COLUMNS:

    passed, invalid, details = distribution_test(

        warehouse[table_name],

        column
    )

    RUN(
        "DISTRIBUTION",
        table_name,
        column,
        "Numeric distribution contains finite observations",
        passed,
        invalid,
        details
    )


# =====================================================================
# 17. FRESHNESS TEST
# =====================================================================

warehouse_files = list(
    WAREHOUSE_DIR.rglob("*.csv")
)

if len(warehouse_files) > 0:

    latest_mtime = max(

        file.stat().st_mtime

        for file in warehouse_files
    )

    latest_datetime = datetime.fromtimestamp(
        latest_mtime
    )

    age_hours = (

        datetime.now()
        -
        latest_datetime

    ).total_seconds() / 3600

    freshness_pass = (
        age_hours <= 24
    )

    RUN(
        "FRESHNESS",
        "warehouse",
        "",
        "Warehouse files are fresh within 24 hours",
        freshness_pass,
        0 if freshness_pass else 1,
        f"Latest warehouse file: "
        f"{latest_datetime} | "
        f"Age = {age_hours:.2f} hours"
    )


# =====================================================================
# 18. NULL RATE SANITY
# =====================================================================

NULL_RATE_LIMIT = 0.05


NULL_RATE_COLUMNS = [

    (
        "dim_candidate",
        "candidate_id"
    ),

    (
        "dim_exam",
        "exam_id"
    ),

    (
        "dim_item",
        "item_id"
    ),

    (
        "dim_reviewer",
        "reviewer_id"
    ),

    (
        "dim_flag_type",
        "flag_type"
    ),

    (
        "fact_response",
        "candidate_key"
    ),

    (
        "fact_response",
        "item_key"
    ),

    (
        "fact_response",
        "exam_key"
    ),

    (
        "fact_session_risk",
        "candidate_key"
    ),

    (
        "fact_session_risk",
        "risk_score"
    ),

    (
        "fact_session_risk",
        "recommendation"
    ),

    (
        "fact_review",
        "reviewer_key"
    ),

    (
        "fact_review",
        "verdict"
    ),

    (
        "fact_review",
        "confidence"
    ),
]


for table_name, column in NULL_RATE_COLUMNS:

    df = warehouse[table_name]

    if column not in df.columns:

        RUN(
            "NULL_RATE",
            table_name,
            column,
            f"NULL rate <= {NULL_RATE_LIMIT:.0%}",
            False,
            len(df),
            "Column not found"
        )

        continue

    null_rate = (
        df[column].isna().mean()
    )

    passed = (
        null_rate <= NULL_RATE_LIMIT
    )

    RUN(
        "NULL_RATE",
        table_name,
        column,
        f"NULL rate <= {NULL_RATE_LIMIT:.0%}",
        passed,
        int(df[column].isna().sum()),
        f"NULL rate = {null_rate:.4%}"
    )


# =====================================================================
# 19. BINARY SANITY
# =====================================================================

if "is_correct" in warehouse["fact_response"].columns:

    normalized = (

        warehouse["fact_response"]
        ["is_correct"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    invalid_mask = ~normalized.isin(
        {
            "0",
            "1",
            "true",
            "false"
        }
    )

    invalid = int(
        invalid_mask.sum()
    )

    RUN(
        "BINARY",
        "fact_response",
        "is_correct",
        "is_correct contains only 0/1 or TRUE/FALSE",
        invalid == 0,
        invalid
    )


# =====================================================================
# 20. RESPONSE BUSINESS KEY
# =====================================================================

response = warehouse["fact_response"]


response_business_columns = [

    "candidate_key",

    "item_key",

    "exam_key",

    "sequence_position",
]


if all(
    c in response.columns
    for c in response_business_columns
):

    duplicated = response.duplicated(

        subset=response_business_columns,

        keep=False
    )

    invalid = int(
        duplicated.sum()
    )

    RUN(
        "BUSINESS_KEY",
        "fact_response",
        "",
        "Response business key is unique",
        invalid == 0,
        invalid,
        "candidate_key + item_key + exam_key + sequence_position"
    )


# =====================================================================
# 21. SESSION RISK BUSINESS KEY
# =====================================================================

session_risk = warehouse[
    "fact_session_risk"
]


if "session_key" in session_risk.columns:

    duplicated = (
        session_risk["session_key"]
        .duplicated(keep=False)
    )

    invalid = int(
        duplicated.sum()
    )

    RUN(
        "BUSINESS_KEY",
        "fact_session_risk",
        "session_key",
        "One risk record per session",
        invalid == 0,
        invalid
    )


# =====================================================================
# 22. REVIEW MONITORING
# =====================================================================

review = warehouse[
    "fact_review"
]


review_business_columns = [

    "session_key",

    "reviewer_key",
]


if all(
    c in review.columns
    for c in review_business_columns
):

    duplicated = review.duplicated(

        subset=review_business_columns,

        keep=False
    )

    duplicate_count = int(
        duplicated.sum()
    )

    # Multiple reviews for same session/reviewer
    # can be legitimate because multiple flags may exist.

    RUN(
        "BUSINESS_KEY",
        "fact_review",
        "",
        "Review session/reviewer duplication is monitored",
        True,
        0,
        f"Duplicate rows observed = "
        f"{duplicate_count:,}; "
        f"multiple review records may be legitimate."
    )


# =====================================================================
# 23. CORRUPTION DEMONSTRATIONS
# =====================================================================

print()
print("=" * 70)
print("CORRUPTION DEMONSTRATIONS")
print("=" * 70)


corruption_results = []


def corruption_check(
    corruption_id,
    description,
    detected
):

    corruption_results.append({

        "corruption_id":
            corruption_id,

        "description":
            description,

        "detected":
            bool(detected),

        "status":
            "PASS"
            if detected
            else "FAIL"
    })


# ---------------------------------------------------------------------
# CORR-001
# Inject NULL candidate_id
# ---------------------------------------------------------------------

corrupt_candidate = (
    warehouse["dim_candidate"]
    .copy()
)


if len(corrupt_candidate) > 0:

    corrupt_candidate.loc[
        corrupt_candidate.index[0],
        "candidate_id"
    ] = np.nan

    passed, invalid = test_not_null(

        corrupt_candidate,

        "candidate_id"
    )

    corruption_check(

        "CORR-001",

        "Injected NULL into DIM_CANDIDATE.candidate_id",

        not passed
    )


# ---------------------------------------------------------------------
# CORR-002
# Inject duplicate item_id
# ---------------------------------------------------------------------

corrupt_item = (
    warehouse["dim_item"]
    .copy()
)


if len(corrupt_item) >= 2:

    corrupt_item.loc[
        corrupt_item.index[1],
        "item_id"
    ] = corrupt_item.loc[
        corrupt_item.index[0],
        "item_id"
    ]

    passed, invalid = test_unique(

        corrupt_item,

        "item_id"
    )

    corruption_check(

        "CORR-002",

        "Injected duplicate DIM_ITEM.item_id",

        not passed
    )


# ---------------------------------------------------------------------
# CORR-003
# Inject invalid candidate FK
# ---------------------------------------------------------------------

corrupt_response = (
    warehouse["fact_response"]
    .copy()
)


if len(corrupt_response) > 0:

    corrupt_response.loc[
        corrupt_response.index[0],
        "candidate_key"
    ] = 999999999

    passed, invalid = test_fk(

        corrupt_response,

        "candidate_key",

        warehouse["dim_candidate"],

        "candidate_key"
    )

    corruption_check(

        "CORR-003",

        "Injected invalid FACT_RESPONSE.candidate_key",

        not passed
    )


# ---------------------------------------------------------------------
# CORR-004
# Inject invalid review verdict
# ---------------------------------------------------------------------

corrupt_review = (
    warehouse["fact_review"]
    .copy()
)


if len(corrupt_review) > 0:

    corrupt_review.loc[
        corrupt_review.index[0],
        "verdict"
    ] = "INVALID_VERDICT"

    passed, invalid = test_accepted_values(

        corrupt_review,

        "verdict",

        {
            "CONFIRMED",
            "OVERTURNED",
            "INCONCLUSIVE",
            "TECHNICAL_ISSUE",
            "ESCALATED",
        }
    )

    corruption_check(

        "CORR-004",

        "Injected invalid FACT_REVIEW.verdict",

        not passed
    )


# ---------------------------------------------------------------------
# CORR-005
# Inject invalid flag FK
# ---------------------------------------------------------------------

corrupt_review_flag = (
    warehouse["fact_review"]
    .copy()
)


if len(corrupt_review_flag) > 0:

    corrupt_review_flag.loc[
        corrupt_review_flag.index[0],
        "flag_key"
    ] = 999999999

    passed, invalid = test_fk(

        corrupt_review_flag,

        "flag_key",

        warehouse["dim_flag_type"],

        "flag_key"
    )

    corruption_check(

        "CORR-005",

        "Injected invalid FACT_REVIEW.flag_key",

        not passed
    )


# ---------------------------------------------------------------------
# CORR-006
# Inject invalid categorical confidence
# ---------------------------------------------------------------------

corrupt_confidence = (
    warehouse["fact_review"]
    .copy()
)


if len(corrupt_confidence) > 0:

    corrupt_confidence.loc[
        corrupt_confidence.index[0],
        "confidence"
    ] = "INVALID_CONFIDENCE"

    passed, invalid = test_accepted_values(

        corrupt_confidence,

        "confidence",

        {
            "low",
            "medium",
            "high",
        }
    )

    corruption_check(

        "CORR-006",

        "Injected invalid FACT_REVIEW.confidence",

        not passed
    )


# =====================================================================
# 24. DISTRIBUTION BASELINE / PSI
# =====================================================================

BASELINE_FILE = (
    DQ_DIR /
    "dq_distribution_baseline.json"
)


def calculate_psi(
    expected,
    actual,
    bins=10
):

    expected = np.asarray(
        expected,
        dtype=float
    )

    actual = np.asarray(
        actual,
        dtype=float
    )

    expected = expected[
        np.isfinite(expected)
    ]

    actual = actual[
        np.isfinite(actual)
    ]

    if (
        len(expected) == 0
        or
        len(actual) == 0
    ):

        return np.nan

    combined = np.concatenate(
        [
            expected,
            actual
        ]
    )

    min_value = np.min(
        combined
    )

    max_value = np.max(
        combined
    )

    if min_value == max_value:

        return 0.0

    edges = np.linspace(

        min_value,

        max_value,

        bins + 1
    )

    expected_hist, _ = np.histogram(

        expected,

        bins=edges
    )

    actual_hist, _ = np.histogram(

        actual,

        bins=edges
    )

    expected_pct = (
        expected_hist
        /
        max(
            expected_hist.sum(),
            1
        )
    )

    actual_pct = (
        actual_hist
        /
        max(
            actual_hist.sum(),
            1
        )
    )

    epsilon = 1e-6

    expected_pct = np.clip(

        expected_pct,

        epsilon,

        None
    )

    actual_pct = np.clip(

        actual_pct,

        epsilon,

        None
    )

    psi = np.sum(

        (
            actual_pct
            -
            expected_pct
        )

        *

        np.log(
            actual_pct
            /
            expected_pct
        )
    )

    return float(psi)


BASELINE_DATA = {}


if BASELINE_FILE.exists():

    try:

        with open(
            BASELINE_FILE,
            "r",
            encoding="utf-8"
        ) as f:

            BASELINE_DATA = json.load(f)

    except Exception:

        BASELINE_DATA = {}


PSI_COLUMNS = [

    (
        "fact_response",
        "response_time_ms"
    ),

    (
        "fact_session_risk",
        "risk_score"
    ),
]


current_baseline_data = {}


for table_name, column in PSI_COLUMNS:

    if column not in warehouse[table_name].columns:

        continue

    values = pd.to_numeric(

        warehouse[table_name][column],

        errors="coerce"
    ).dropna()

    values = values[
        np.isfinite(values)
    ]

    baseline_key = (
        f"{table_name}.{column}"
    )

    current_baseline_data[
        baseline_key
    ] = values.tolist()


    if baseline_key in BASELINE_DATA:

        psi = calculate_psi(

            BASELINE_DATA[baseline_key],

            values.tolist()
        )

        passed = (

            np.isnan(psi)
            or
            psi < 0.25
        )

        RUN(
            "DRIFT",
            table_name,
            column,
            "Population Stability Index (PSI) < 0.25",
            passed,
            0 if passed else 1,
            f"PSI = {psi:.6f}"
        )

    else:

        RUN(
            "DRIFT",
            table_name,
            column,
            "Baseline created for future PSI monitoring",
            True,
            0,
            "No previous baseline existed."
        )


# ---------------------------------------------------------------------
# Create baseline if it does not exist
# ---------------------------------------------------------------------

if not BASELINE_FILE.exists():

    with open(
        BASELINE_FILE,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(

            current_baseline_data,

            f
        )

    print(
        f"Created baseline: "
        f"{BASELINE_FILE}"
    )


# =====================================================================
# 25. SAVE RESULTS
# =====================================================================

results_df = pd.DataFrame(
    results
)

corruption_df = pd.DataFrame(
    corruption_results
)


RESULTS_FILE = (
    DQ_DIR /
    "dq_test_results.csv"
)

CORRUPTION_FILE = (
    DQ_DIR /
    "corruption_detection_results.csv"
)

SUMMARY_FILE = (
    DQ_DIR /
    "dq_summary.json"
)


results_df.to_csv(

    RESULTS_FILE,

    index=False
)


corruption_df.to_csv(

    CORRUPTION_FILE,

    index=False
)


# =====================================================================
# 26. SUMMARY
# =====================================================================

total_tests = len(
    results_df
)

passed_tests = int(
    (
        results_df["status"]
        ==
        "PASS"
    ).sum()
)

failed_tests = int(
    (
        results_df["status"]
        ==
        "FAIL"
    ).sum()
)


total_corruptions = len(
    corruption_df
)

detected_corruptions = int(
    (
        corruption_df["detected"]
        ==
        True
    ).sum()
)


overall_status = (

    "PASSED"

    if (
        failed_tests == 0

        and

        detected_corruptions
        ==
        total_corruptions
    )

    else

    "FAILED"
)


summary = {

    "project":
        "IntegrityInsight",

    "requirement":
        "FR-8 Data Quality Test Suite",

    "generated_at":
        datetime.now().isoformat(),

    "total_dq_tests":
        total_tests,

    "passed_tests":
        passed_tests,

    "failed_tests":
        failed_tests,

    "pass_rate":
        round(

            passed_tests
            /
            total_tests
            *
            100,

            2
        )

        if total_tests > 0

        else 0,

    "corruption_tests":
        total_corruptions,

    "corruptions_detected":
        detected_corruptions,

    "overall_status":
        overall_status,

    "warehouse_tables":
        list(
            warehouse.keys()
        ),

    "fact_model": [

        "FACT_RESPONSE",

        "FACT_SESSION_RISK",

        "FACT_REVIEW"
    ],

    "confidence_format":
        [
            "low",
            "medium",
            "high"
        ],

    "notes_length_status":
        "NULL for current review records",

    "dq_output_directory":
        str(DQ_DIR)
}


with open(

    SUMMARY_FILE,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        summary,

        f,

        indent=2
    )


# =====================================================================
# 27. FINAL OUTPUT
# =====================================================================

print()
print("=" * 70)
print("FR-8 DATA QUALITY RESULTS")
print("=" * 70)

print(
    f"Total DQ Tests       : "
    f"{total_tests}"
)

print(
    f"Passed Tests         : "
    f"{passed_tests}"
)

print(
    f"Failed Tests         : "
    f"{failed_tests}"
)

print(
    f"Pass Rate            : "
    f"{summary['pass_rate']:.2f}%"
)

print()

print(
    f"Corruption Tests     : "
    f"{total_corruptions}"
)

print(
    f"Corruptions Detected : "
    f"{detected_corruptions}"
)

print()

print(
    f"OVERALL FR-8 STATUS  : "
    f"{overall_status}"
)

print()
print("=" * 70)
print("OUTPUT FILES")
print("=" * 70)

print(
    f"DQ Test Results      : "
    f"{RESULTS_FILE}"
)

print(
    f"Corruption Results   : "
    f"{CORRUPTION_FILE}"
)

print(
    f"DQ Summary           : "
    f"{SUMMARY_FILE}"
)

print(
    f"Distribution Baseline: "
    f"{BASELINE_FILE}"
)

print("=" * 70)


# =====================================================================
# 28. FAILED TESTS
# =====================================================================

if failed_tests > 0:

    print()
    print("=" * 70)
    print("FAILED DQ TESTS")
    print("=" * 70)

    print(

        results_df[
            results_df["status"] == "FAIL"
        ][
            [
                "test_id",
                "category",
                "table",
                "column",
                "test_name",
                "invalid_count",
                "details"
            ]
        ].to_string(
            index=False
        )
    )

else:

    print()
    print(
        "No failed DQ tests."
    )


# =====================================================================
# 29. CORRUPTION DETECTION RESULTS
# =====================================================================

print()
print("=" * 70)
print("CORRUPTION DETECTION RESULTS")
print("=" * 70)

print(
    corruption_df.to_string(
        index=False
    )
)

print()
print("=" * 70)
print("FR-8 COMPLETE")
print("=" * 70)

INTEGRITYINSIGHT — FR-8 DATA QUALITY TEST SUITE
DATA_DIR      : /content/integrityinsight_data
WAREHOUSE_DIR : /content/integrityinsight_data/warehouse
DIM_DIR       : /content/integrityinsight_data/warehouse/dimensions
FACT_DIR      : /content/integrityinsight_data/warehouse/facts
DQ OUTPUT     : /content/integrityinsight_data/validation/data_quality

Warehouse tables loaded successfully:
  dim_candidate                    300 rows
  dim_exam                          20 rows
  dim_item                       1,000 rows
  dim_reviewer                      12 rows
  dim_flag_type                      8 rows
  dim_date                           1 rows
  fact_response                300,000 rows
  fact_session_risk              6,000 rows
  fact_review                    3,795 rows


CORRUPTION DEMONSTRATIONS
Created baseline: /content/integrityinsight_data/validation/data_quality/dq_distribution_baseline.json

FR-8 DATA QUALITY RESULTS
Total DQ Tests       : 123
Passed Tests         : 123

In [ ]:
import pandas as pd

path = "/content/integrityinsight_data/validation/data_quality/dq_test_results.csv"

df = pd.read_csv(path)

display(df)

,test_id,category,table,column,test_name,status,invalid_count,details
0,DQT-001,TABLE_EXISTENCE,dim_candidate,NaN,Required warehouse table exists,PASS,0,NaN
1,DQT-002,TABLE_EXISTENCE,dim_exam,NaN,Required warehouse table exists,PASS,0,NaN
2,DQT-003,TABLE_EXISTENCE,dim_item,NaN,Required warehouse table exists,PASS,0,NaN
3,DQT-004,TABLE_EXISTENCE,dim_reviewer,NaN,Required warehouse table exists,PASS,0,NaN
4,DQT-005,TABLE_EXISTENCE,dim_flag_type,NaN,Required warehouse table exists,PASS,0,NaN
...,...,...,...,...,...,...,...,...
118,DQT-119,BUSINESS_KEY,fact_response,NaN,Response business key is unique,PASS,0,candidate_key + item_key + exam_key + sequence...
119,DQT-120,BUSINESS_KEY,fact_session_risk,session_key,One risk record per session,PASS,0,NaN
120,DQT-121,BUSINESS_KEY,fact_review,NaN,Review session/reviewer duplication is monitored,PASS,0,Duplicate rows observed = 159; multiple review...
121,DQT-122,DRIFT,fact_response,response_time_ms,Baseline created for future PSI monitoring,PASS,0,No previous baseline existed.


In [ ]:
from google.colab import files

files.download("/content/integrityinsight_data/validation/data_quality/dq_test_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# INTEGRITYINSIGHT — D11 + D12 + D14 + D16
# Paste this ONE cell AFTER the existing DQ / FR-8 cell.
# ============================================================

from pathlib import Path
import json
import math
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 0. COMMON PATHS + SAFE LOADERS
# ------------------------------------------------------------

DATA_DIR = Path.cwd() / "integrityinsight_data"
WAREHOUSE_DIR = DATA_DIR / "warehouse"
DIM_DIR = WAREHOUSE_DIR / "dimensions"
FACT_DIR = WAREHOUSE_DIR / "facts"

DELIVERABLE_DIR = DATA_DIR / "deliverables"

D11_DIR = DELIVERABLE_DIR / "D11_item_analysis"
D12_DIR = DELIVERABLE_DIR / "D12_cost_capacity"
D14_DIR = DELIVERABLE_DIR / "D14_metric_dictionary"
D16_DIR = DELIVERABLE_DIR / "D16_access_retention"

for d in [D11_DIR, D12_DIR, D14_DIR, D16_DIR]:
    d.mkdir(parents=True, exist_ok=True)


def load_csv(path):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(
            f"Missing required project file: {path}"
        )

    return pd.read_csv(path)


# ============================================================
# D11 — ITEM ANALYSIS REPORT
# ============================================================

print("\n" + "=" * 80)
print("D11 — ITEM ANALYSIS REPORT")
print("=" * 80)

items_d11 = load_csv(
    DIM_DIR / "dim_item.csv"
)

responses_d11 = load_csv(
    FACT_DIR / "fact_response.csv"
)

candidates_d11 = load_csv(
    DIM_DIR / "dim_candidate.csv"
)


# ------------------------------------------------------------
# Recover candidate and item identifiers
# ------------------------------------------------------------

candidate_lookup = candidates_d11[
    [
        "candidate_key",
        "candidate_id"
    ]
].copy()


item_lookup = items_d11[
    [
        "item_key",
        "item_id",
        "exam_id",
        "item_number",
        "difficulty_b",
        "discrimination_a",
        "guessing_c",
        "correct_option"
    ]
].copy()


r = responses_d11.merge(
    candidate_lookup,
    on="candidate_key",
    how="left"
)


r = r.merge(
    item_lookup,
    on="item_key",
    how="left",
    suffixes=("", "_item")
)


if r[
    [
        "candidate_id",
        "item_id",
        "is_correct"
    ]
].isnull().any().any():

    raise ValueError(
        "D11 join failed: missing candidate/item keys or is_correct values."
    )


# ------------------------------------------------------------
# Candidate total score
# ------------------------------------------------------------

candidate_score = (
    r.groupby(
        "candidate_id",
        as_index=False
    )["is_correct"]
    .sum()
    .rename(
        columns={
            "is_correct": "candidate_total_correct"
        }
    )
)


r = r.merge(
    candidate_score,
    on="candidate_id",
    how="left"
)


# ------------------------------------------------------------
# Point-biserial helper
# ------------------------------------------------------------

def safe_point_biserial(group):

    item_score = (
        group["is_correct"]
        .astype(float)
    )

    total_score = (
        group["candidate_total_correct"]
        .astype(float)
    )

    # Remove the item's own score from total score
    criterion = total_score - item_score

    if item_score.nunique() < 2:
        return np.nan

    if criterion.nunique() < 2:
        return np.nan

    corr = item_score.corr(
        criterion
    )

    if pd.isna(corr):
        return np.nan

    return float(corr)


# ------------------------------------------------------------
# Item-level analysis
# ------------------------------------------------------------

rows = []

for (
    exam_id,
    item_id
), group in r.groupby(
    [
        "exam_id",
        "item_id"
    ],
    sort=True
):

    first = group.iloc[0]

    n = len(group)

    correct_rate = (
        float(
            group["is_correct"].mean()
        )
        if n
        else np.nan
    )

    avg_rt = (
        float(
            group["response_time_ms"].mean()
        )
        if n
        else np.nan
    )

    median_rt = (
        float(
            group["response_time_ms"].median()
        )
        if n
        else np.nan
    )

    revision_rate = (
        float(
            (
                group["revisions"] > 0
            ).mean()
        )
        if n
        else np.nan
    )

    point_biserial = safe_point_biserial(
        group
    )

    rows.append(
        {
            "exam_id": exam_id,

            "item_id": item_id,

            "item_number": int(
                first["item_number"]
            ),

            "n_responses": n,

            "correct_rate": correct_rate,

            "difficulty_empirical":
                1.0 - correct_rate,

            "point_biserial":
                point_biserial,

            "avg_response_time_ms":
                avg_rt,

            "median_response_time_ms":
                median_rt,

            "revision_rate":
                revision_rate,

            "irt_difficulty_b":
                float(
                    first["difficulty_b"]
                ),

            "irt_discrimination_a":
                float(
                    first["discrimination_a"]
                ),

            "irt_guessing_c":
                float(
                    first["guessing_c"]
                ),

            "correct_option":
                first["correct_option"]
        }
    )


item_report = pd.DataFrame(
    rows
)


# ------------------------------------------------------------
# Operational screening rules
# ------------------------------------------------------------

# Keep every triggered rule instead of overwriting earlier flags.
def build_analysis_flags(row):
    flags = []
    if pd.notna(row["point_biserial"]) and row["point_biserial"] < 0:
        flags.append("NEGATIVE_DISCRIMINATION")
    if row["correct_rate"] < 0.20:
        flags.append("VERY_DIFFICULT")
    if row["correct_rate"] > 0.95:
        flags.append("VERY_EASY")
    if row["revision_rate"] > 0.25:
        flags.append("HIGH_REVISION_RATE")
    return "; ".join(flags) if flags else "OK"

item_report["analysis_flag"] = item_report.apply(
    build_analysis_flags,
    axis=1
)


priority = {
    "NEGATIVE_DISCRIMINATION": 1,
    "VERY_DIFFICULT": 2,
    "VERY_EASY": 3,
    "HIGH_REVISION_RATE": 4,
    "OK": 99
}


item_report["_priority"] = (
    item_report["analysis_flag"]
    .map(priority)
    .fillna(99)
)


item_report["rule_hits"] = (

    (
        item_report["point_biserial"] < 0
    ).astype(int)

    +

    (
        item_report["correct_rate"] < 0.20
    ).astype(int)

    +

    (
        item_report["correct_rate"] > 0.95
    ).astype(int)

    +

    (
        item_report["revision_rate"] > 0.25
    ).astype(int)
)


item_report = (
    item_report
    .sort_values(
        [
            "rule_hits",
            "_priority",
            "exam_id",
            "item_number"
        ],
        ascending=[
            False,
            True,
            True,
            True
        ]
    )
    .drop(
        columns=["_priority"]
    )
)


# ------------------------------------------------------------
# Save D11
# ------------------------------------------------------------

item_report.to_csv(
    D11_DIR /
    "item_analysis_report.csv",
    index=False
)


summary = pd.DataFrame(
    {
        "metric": [

            "items_analyzed",

            "mean_correct_rate",

            "median_correct_rate",

            "mean_point_biserial",

            "negative_discrimination_items",

            "very_difficult_items",

            "very_easy_items",

            "high_revision_rate_items",

            "flagged_items_any_rule"
        ],

        "value": [

            len(item_report),

            item_report[
                "correct_rate"
            ].mean(),

            item_report[
                "correct_rate"
            ].median(),

            item_report[
                "point_biserial"
            ].mean(),

            int(
                (
                    item_report[
                        "point_biserial"
                    ] < 0
                ).sum()
            ),

            int(
                (
                    item_report[
                        "correct_rate"
                    ] < 0.20
                ).sum()
            ),

            int(
                (
                    item_report[
                        "correct_rate"
                    ] > 0.95
                ).sum()
            ),

            int(
                (
                    item_report[
                        "revision_rate"
                    ] > 0.25
                ).sum()
            ),

            int(
                (
                    item_report[
                        "rule_hits"
                    ] > 0
                ).sum()
            )
        ]
    }
)


summary.to_csv(
    D11_DIR /
    "item_analysis_summary.csv",
    index=False
)


exam_summary = (
    item_report
    .groupby(
        "exam_id",
        as_index=False
    )
    .agg(
        items=(
            "item_id",
            "nunique"
        ),

        avg_correct_rate=(
            "correct_rate",
            "mean"
        ),

        avg_point_biserial=(
            "point_biserial",
            "mean"
        ),

        flagged_items=(
            "rule_hits",
            lambda x:
                int(
                    (x > 0).sum()
                )
        ),

        avg_response_time_ms=(
            "avg_response_time_ms",
            "mean"
        )
    )
)


exam_summary.to_csv(
    D11_DIR /
    "exam_item_summary.csv",
    index=False
)


d11_md = f"""
# D11 — Item Analysis Report

## Scope

- Items analyzed: **{len(item_report):,}**
- Source: IntegrityInsight warehouse `DIM_ITEM` + `FACT_RESPONSE`
- Grain: one row per item

## Metrics

- **Correct rate:** proportion of responses marked correct.
- **Empirical difficulty:** `1 - correct_rate`.
- **Point-biserial:** correlation between item correctness and candidate total score excluding the item.
- **IRT difficulty:** generated `difficulty_b`.
- **IRT discrimination:** generated `discrimination_a`.
- **IRT guessing:** generated `guessing_c`.
- **Revision rate:** proportion of responses with one or more revisions.

## Operational Screening Rules

These are project-level screening rules for human inspection.

- Negative point-biserial → `NEGATIVE_DISCRIMINATION`
- Correct rate < 0.20 → `VERY_DIFFICULT`
- Correct rate > 0.95 → `VERY_EASY`
- Revision rate > 0.25 → `HIGH_REVISION_RATE`

## Summary

- Mean correct rate: **{item_report['correct_rate'].mean():.3f}**
- Median correct rate: **{item_report['correct_rate'].median():.3f}**
- Mean point-biserial: **{item_report['point_biserial'].mean():.3f}**
- Items with any screening flag:
  **{int((item_report['rule_hits'] > 0).sum()):,}**

## Output Files

- `item_analysis_report.csv`
- `item_analysis_summary.csv`
- `exam_item_summary.csv`
"""

(
    D11_DIR /
    "D11_item_analysis_report.md"
).write_text(
    d11_md,
    encoding="utf-8"
)


print(
    f"D11 complete — "
    f"{len(item_report):,} items analyzed."
)


# ============================================================
# D12 — COST MODEL & CAPACITY ANALYSIS
# ============================================================

print("\n" + "=" * 80)
print("D12 — COST MODEL & CAPACITY ANALYSIS")
print("=" * 80)


cost_d12 = load_csv(
    DATA_DIR /
    "staging" /
    "infrastructure_cost.csv"
)


reviews_d12 = load_csv(
    DATA_DIR /
    "staging" /
    "review_verdicts.csv"
)


sessions_d12 = load_csv(
    DATA_DIR /
    "staging" /
    "sessions.csv"
)


reviewers_d12 = load_csv(
    DATA_DIR /
    "staging" /
    "reviewers.csv"
)


required_cost_cols = {
    "session_id",
    "compute_cost",
    "storage_cost",
    "egress_cost",
    "review_labor_cost",
    "total_cost"
}


missing = (
    required_cost_cols
    -
    set(cost_d12.columns)
)


if missing:

    raise ValueError(
        "D12 cost table is missing columns: "
        + str(sorted(missing))
    )


# IMPORTANT:
# Project generator defines these as synthetic values.

CURRENCY_UNIT = (
    "synthetic currency unit"
)


cost_d12[
    "infrastructure_cost"
] = (

    cost_d12[
        "compute_cost"
    ]

    +

    cost_d12[
        "storage_cost"
    ]

    +

    cost_d12[
        "egress_cost"
    ]
)


cost_totals = {

    "compute_cost":
        cost_d12[
            "compute_cost"
        ].sum(),

    "storage_cost":
        cost_d12[
            "storage_cost"
        ].sum(),

    "egress_cost":
        cost_d12[
            "egress_cost"
        ].sum(),

    "review_labor_cost":
        cost_d12[
            "review_labor_cost"
        ].sum(),

    "total_cost":
        cost_d12[
            "total_cost"
        ].sum()
}


n_sessions = len(
    sessions_d12
)

n_candidates = (
    sessions_d12[
        "candidate_id"
    ].nunique()
)

n_exams = (
    sessions_d12[
        "exam_id"
    ].nunique()
)

n_reviews = len(
    reviews_d12
)

n_reviewers = (
    reviewers_d12[
        "reviewer_id"
    ].nunique()
)


cost_summary = pd.DataFrame(
    [

        {
            "metric":
                "candidates",
            "value":
                n_candidates,
            "unit":
                "candidates"
        },

        {
            "metric":
                "exams",
            "value":
                n_exams,
            "unit":
                "exams"
        },

        {
            "metric":
                "sessions",
            "value":
                n_sessions,
            "unit":
                "sessions"
        },

        {
            "metric":
                "reviews",
            "value":
                n_reviews,
            "unit":
                "reviews"
        },

        {
            "metric":
                "reviewers",
            "value":
                n_reviewers,
            "unit":
                "reviewers"
        },

        {
            "metric":
                "compute_cost",
            "value":
                cost_totals[
                    "compute_cost"
                ],
            "unit":
                CURRENCY_UNIT
        },

        {
            "metric":
                "storage_cost",
            "value":
                cost_totals[
                    "storage_cost"
                ],
            "unit":
                CURRENCY_UNIT
        },

        {
            "metric":
                "egress_cost",
            "value":
                cost_totals[
                    "egress_cost"
                ],
            "unit":
                CURRENCY_UNIT
        },

        {
            "metric":
                "review_labor_cost",
            "value":
                cost_totals[
                    "review_labor_cost"
                ],
            "unit":
                CURRENCY_UNIT
        },

        {
            "metric":
                "total_cost",
            "value":
                cost_totals[
                    "total_cost"
                ],
            "unit":
                CURRENCY_UNIT
        },

        {
            "metric":
                "cost_per_candidate",
            "value":
                (
                    cost_totals[
                        "total_cost"
                    ]
                    /
                    n_candidates
                    if n_candidates
                    else np.nan
                ),
            "unit":
                CURRENCY_UNIT
        },

        {
            "metric":
                "cost_per_exam",
            "value":
                (
                    cost_totals[
                        "total_cost"
                    ]
                    /
                    n_exams
                    if n_exams
                    else np.nan
                ),
            "unit":
                CURRENCY_UNIT
        },

        {
            "metric":
                "cost_per_session",
            "value":
                (
                    cost_totals[
                        "total_cost"
                    ]
                    /
                    n_sessions
                    if n_sessions
                    else np.nan
                ),
            "unit":
                CURRENCY_UNIT
        },

        {
            "metric":
                "cost_per_review",
            "value":
                (
                    cost_totals[
                        "total_cost"
                    ]
                    /
                    n_reviews
                    if n_reviews
                    else np.nan
                ),
            "unit":
                CURRENCY_UNIT
        }
    ]
)


cost_summary.to_csv(
    D12_DIR /
    "cost_summary.csv",
    index=False
)


# ------------------------------------------------------------
# Cost mix
# ------------------------------------------------------------

cost_mix = pd.DataFrame(
    {
        "component":
            list(
                cost_totals.keys()
            ),

        "cost":
            list(
                cost_totals.values()
            )
    }
)


cost_mix[
    "share_of_total"
] = (

    cost_mix[
        "cost"
    ]

    /

    cost_totals[
        "total_cost"
    ]

    if cost_totals[
        "total_cost"
    ]

    else np.nan
)


cost_mix.to_csv(
    D12_DIR /
    "cost_mix.csv",
    index=False
)


# ------------------------------------------------------------
# Reviewer capacity
# ------------------------------------------------------------

WORKING_HOURS_PER_DAY = 8

WORKING_DAYS_PER_MONTH = 20

UTILIZATION = 0.80


reviews_d12[
    "review_minutes"
] = (
    reviews_d12[
        "review_seconds"
    ]
    /
    60.0
)


reviewer_ops = (
    reviews_d12
    .groupby(
        "reviewer_id",
        as_index=False
    )
    .agg(

        reviews_completed=(
            "review_id",
            "count"
        ),

        total_review_minutes=(
            "review_minutes",
            "sum"
        ),

        avg_review_seconds=(
            "review_seconds",
            "mean"
        )
    )
)


reviewer_ops[
    "observed_review_hours"
] = (
    reviewer_ops[
        "total_review_minutes"
    ]
    /
    60.0
)


reviewer_ops[
    "share_of_reviews"
] = (
    reviewer_ops[
        "reviews_completed"
    ]
    /
    max(
        1,
        len(reviews_d12)
    )
)


reviewer_ops.to_csv(
    D12_DIR /
    "reviewer_capacity_observed.csv",
    index=False
)


# ------------------------------------------------------------
# Capacity calculations
# ------------------------------------------------------------

avg_review_seconds = float(
    reviews_d12[
        "review_seconds"
    ].mean()
)


productive_seconds = (
    WORKING_HOURS_PER_DAY
    *
    3600
    *
    UTILIZATION
)


reviews_per_reviewer_day = (
    productive_seconds
    /
    avg_review_seconds
    if avg_review_seconds > 0
    else np.nan
)


reviews_per_reviewer_month = (
    reviews_per_reviewer_day
    *
    WORKING_DAYS_PER_MONTH
)


team_monthly_capacity = (
    reviews_per_reviewer_month
    *
    n_reviewers
)


capacity_summary = pd.DataFrame(
    [

        {
            "metric":
                "avg_review_seconds",
            "value":
                avg_review_seconds,
            "unit":
                "seconds"
        },

        {
            "metric":
                "working_hours_per_day",
            "value":
                WORKING_HOURS_PER_DAY,
            "unit":
                "hours"
        },

        {
            "metric":
                "working_days_per_month",
            "value":
                WORKING_DAYS_PER_MONTH,
            "unit":
                "days"
        },

        {
            "metric":
                "utilization_assumption",
            "value":
                UTILIZATION,
            "unit":
                "fraction"
        },

        {
            "metric":
                "reviews_per_reviewer_day",
            "value":
                reviews_per_reviewer_day,
            "unit":
                "reviews/day"
        },

        {
            "metric":
                "reviews_per_reviewer_month",
            "value":
                reviews_per_reviewer_month,
            "unit":
                "reviews/month"
        },

        {
            "metric":
                "team_monthly_capacity",
            "value":
                team_monthly_capacity,
            "unit":
                "reviews/month"
        }
    ]
)


capacity_summary.to_csv(
    D12_DIR /
    "capacity_summary.csv",
    index=False
)


# ------------------------------------------------------------
# Scenario analysis
# ------------------------------------------------------------

base_candidates = max(
    1,
    n_candidates
)

base_reviews = max(
    1,
    n_reviews
)

base_sessions = max(
    1,
    n_sessions
)

base_cost = (
    cost_totals[
        "total_cost"
    ]
)


base_reviews_per_candidate = (
    base_reviews
    /
    base_candidates
)


base_sessions_per_candidate = (
    base_sessions
    /
    base_candidates
)


base_cost_per_candidate = (
    base_cost
    /
    base_candidates
)


scenario_candidates = [
    500,
    1000,
    2500,
    5000,
    10000
]


scenario_rows = []


for c in scenario_candidates:

    projected_sessions = (
        c
        *
        base_sessions_per_candidate
    )

    projected_reviews = (
        c
        *
        base_reviews_per_candidate
    )

    projected_total_cost = (
        c
        *
        base_cost_per_candidate
    )

    required_reviewers = (

        math.ceil(
            projected_reviews
            /
            reviews_per_reviewer_month
        )

        if reviews_per_reviewer_month > 0
        and np.isfinite(reviews_per_reviewer_month)

        else np.nan
    )


    scenario_rows.append(
        {
            "candidates":
                c,

            "projected_sessions":
                projected_sessions,

            "projected_reviews":
                projected_reviews,

            "projected_total_cost":
                projected_total_cost,

            "required_reviewers_for_one_month_capacity":
                required_reviewers
        }
    )


scenario_df = pd.DataFrame(
    scenario_rows
)


scenario_df.to_csv(
    D12_DIR /
    "capacity_cost_scenarios.csv",
    index=False
)


d12_md = f"""
# D12 — Cost Model and Capacity Analysis

## Data Scope

- Candidates: **{n_candidates:,}**
- Exams: **{n_exams:,}**
- Sessions: **{n_sessions:,}**
- Reviews: **{n_reviews:,}**
- Reviewers: **{n_reviewers:,}**

## Cost Model

The project's generator labels the monetary values as synthetic cost units.

They should not be interpreted as EGP or another real currency without replacing the generator inputs with real institutional rates.

Total cost:

`compute + storage + egress + review labor`

### Results

- Total cost: **{cost_totals["total_cost"]:.4f} {CURRENCY_UNIT}**
- Cost per candidate:
  **{cost_totals["total_cost"] / n_candidates:.4f}**
- Cost per session:
  **{cost_totals["total_cost"] / n_sessions:.6f}**
- Cost per review:
  **{cost_totals["total_cost"] / n_reviews:.6f}**

## Capacity Assumptions

- Working day: **{WORKING_HOURS_PER_DAY} hours**
- Working days/month: **{WORKING_DAYS_PER_MONTH}**
- Productive utilization: **{UTILIZATION:.0%}**
- Average observed review time:
  **{avg_review_seconds:.2f} seconds**

These are planning assumptions and can be changed in the code.

## Output Files

- `cost_summary.csv`
- `cost_mix.csv`
- `reviewer_capacity_observed.csv`
- `capacity_summary.csv`
- `capacity_cost_scenarios.csv`
"""


(
    D12_DIR /
    "D12_cost_capacity_report.md"
).write_text(
    d12_md,
    encoding="utf-8"
)


print(
    f"D12 complete — "
    f"total synthetic cost = "
    f"{cost_totals['total_cost']:.4f}."
)


# ============================================================
# D14 — METRIC DICTIONARY
# ============================================================

print("\n" + "=" * 80)
print("D14 — METRIC DICTIONARY")
print("=" * 80)


metric_rows = [

    [
        "Candidate Count",
        "Number of unique candidates in the candidate dimension.",
        "COUNT(DISTINCT candidate_id)",
        "DIM_CANDIDATE",
        "Candidate",
        "Descriptive population size"
    ],

    [
        "Exam Count",
        "Number of unique exams represented in the exam dimension.",
        "COUNT(DISTINCT exam_id)",
        "DIM_EXAM",
        "Exam",
        "Descriptive exam coverage"
    ],

    [
        "Item Count",
        "Number of unique assessment items.",
        "COUNT(DISTINCT item_id)",
        "DIM_ITEM",
        "Item",
        "Assessment coverage"
    ],

    [
        "Session Count",
        "Number of exam sessions.",
        "COUNT(DISTINCT session_id)",
        "FACT_SESSION_RISK / SESSIONS",
        "Session",
        "Operational volume"
    ],

    [
        "Response Count",
        "Number of recorded item responses.",
        "COUNT(response_id)",
        "FACT_RESPONSE",
        "Response",
        "Response volume"
    ],

    [
        "Correct Rate",
        "Proportion of responses marked correct.",
        "SUM(is_correct) / COUNT(response_id)",
        "FACT_RESPONSE",
        "Response / Item / Candidate",
        "Assessment performance"
    ],

    [
        "Empirical Difficulty",
        "Observed proportion of incorrect responses.",
        "1 - Correct Rate",
        "FACT_RESPONSE",
        "Item",
        "Item difficulty"
    ],

    [
        "Point-Biserial",
        "Correlation between item correctness and candidate total score excluding the item.",
        "corr(item_is_correct, total_score_minus_item)",
        "FACT_RESPONSE",
        "Item",
        "Empirical item discrimination"
    ],

    [
        "IRT Difficulty (b)",
        "Simulator-generated IRT difficulty parameter.",
        "Stored parameter: difficulty_b",
        "DIM_ITEM",
        "Item",
        "Model parameter"
    ],

    [
        "IRT Discrimination (a)",
        "Simulator-generated IRT discrimination parameter.",
        "Stored parameter: discrimination_a",
        "DIM_ITEM",
        "Item",
        "Model parameter"
    ],

    [
        "IRT Guessing (c)",
        "Simulator-generated guessing parameter.",
        "Stored parameter: guessing_c",
        "DIM_ITEM",
        "Item",
        "Model parameter"
    ],

    [
        "Average Response Time",
        "Mean response time across responses.",
        "AVG(response_time_ms)",
        "FACT_RESPONSE",
        "Response / Item / Session",
        "Timing behavior"
    ],

    [
        "Revision Rate",
        "Share of responses with one or more revisions.",
        "MEAN(revisions > 0)",
        "FACT_RESPONSE",
        "Response / Item",
        "Interaction behavior"
    ],

    [
        "Overall Risk",
        "Session-level risk score generated by the project.",
        "Stored field: overall_risk",
        "FACT_SESSION_RISK",
        "Session",
        "Risk screening"
    ],

    [
        "Flag Count",
        "Number of proctoring flags associated with a session.",
        "Stored field: flag_count",
        "FACT_SESSION_RISK",
        "Session",
        "Proctoring activity"
    ],

    [
        "Review Count",
        "Number of reviewer records.",
        "COUNT(review_id)",
        "FACT_REVIEW",
        "Review",
        "Review workload"
    ],

    [
        "Review Time",
        "Time spent on a review.",
        "review_seconds",
        "FACT_REVIEW",
        "Review",
        "Reviewer workload"
    ],

    [
        "Reviewer Agreement",
        "Share of reviewer decisions matching validation truth when ground truth is available.",
        "correct reviews / total reviews",
        "FACT_REVIEW + validation truth",
        "Reviewer",
        "Validation-only performance measure"
    ],

    [
        "Total Cost",
        "Sum of all generated cost components.",
        "SUM(compute + storage + egress + review_labor)",
        "infrastructure_cost.csv",
        "Session / Portfolio",
        "Synthetic cost model"
    ],

    [
        "Cost per Session",
        "Average generated total cost per session.",
        "Total Cost / Session Count",
        "infrastructure_cost.csv",
        "Session",
        "Unit economics"
    ],

    [
        "Cost per Candidate",
        "Average generated total cost per candidate.",
        "Total Cost / Candidate Count",
        "infrastructure_cost.csv",
        "Candidate",
        "Unit economics"
    ],

    [
        "Cost per Review",
        "Average generated total cost per review.",
        "Total Cost / Review Count",
        "infrastructure_cost.csv + FACT_REVIEW",
        "Review",
        "Review economics"
    ],

    [
        "Reviews per Reviewer Day",
        "Estimated reviews a reviewer can complete in one productive day.",
        "(Hours/day × 3600 × utilization) / avg_review_seconds",
        "FACT_REVIEW + planning assumptions",
        "Reviewer",
        "Capacity planning"
    ],

    [
        "Team Monthly Capacity",
        "Estimated reviews completed by the current reviewer team in one month.",
        "reviews_per_reviewer_month × reviewer_count",
        "FACT_REVIEW + planning assumptions",
        "Team",
        "Capacity planning"
    ]
]


metric_dictionary = pd.DataFrame(
    metric_rows,
    columns=[
        "metric_name",
        "definition",
        "formula_or_logic",
        "source",
        "grain",
        "intended_use"
    ]
)


metric_dictionary.insert(
    0,
    "metric_id",
    [
        f"MET-{i:03d}"
        for i in range(
            1,
            len(metric_dictionary) + 1
        )
    ]
)


metric_dictionary.to_csv(
    D14_DIR /
    "metric_dictionary.csv",
    index=False
)


with open(
    D14_DIR /
    "metric_dictionary.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        metric_dictionary.to_dict(
            orient="records"
        ),
        f,
        indent=2,
        ensure_ascii=False
    )


# Markdown version without requiring tabulate.
d14_lines = [
    "# D14 — Metric Dictionary",
    "",
    "| ID | Metric | Definition | Formula |",
    "|---|---|---|---|"
]


for row in metric_dictionary.itertuples(
    index=False
):

    metric_id = str(
        row.metric_id
    ).replace(
        "|",
        "\\|"
    )

    name = str(
        row.metric_name
    ).replace(
        "|",
        "\\|"
    )

    definition = str(
        row.definition
    ).replace(
        "|",
        "\\|"
    )

    formula = str(
        row.formula_or_logic
    ).replace(
        "|",
        "\\|"
    )

    d14_lines.append(
        f"| {metric_id} | {name} | {definition} | {formula} |"
    )


(
    D14_DIR /
    "D14_metric_dictionary.md"
).write_text(
    "\n".join(d14_lines),
    encoding="utf-8"
)


print(
    f"D14 complete — "
    f"{len(metric_dictionary)} metrics documented."
)


# ============================================================
# D16 — ACCESS CONTROL DESIGN + RETENTION POLICY
# ============================================================

print("\n" + "=" * 80)
print("D16 — ACCESS CONTROL DESIGN + RETENTION POLICY")
print("=" * 80)


roles = pd.DataFrame(

    [

        [
            "Data Engineer",
            "raw ingestion, staging, warehouse",
            "read/write",
            "No routine access to validation ground truth outside approved validation workflow"
        ],

        [
            "Data Analyst",
            "warehouse dimensions/facts, aggregated analytics",
            "read",
            "No raw ground-truth or unnecessary candidate-level access"
        ],

        [
            "Reviewer",
            "assigned review cases",
            "read/write assigned cases",
            "Cannot access other reviewers' queues or hidden ground truth"
        ],

        [
            "Institution Admin",
            "aggregated dashboards, institutional reports",
            "read",
            "No raw candidate/proctoring details by default"
        ],

        [
            "System Admin",
            "platform configuration and access control",
            "admin",
            "Sensitive access should be logged and break-glass controlled"
        ]
    ],

    columns=[
        "role",
        "data_scope",
        "access_level",
        "restriction"
    ]
)


roles.to_csv(
    D16_DIR /
    "access_control_roles.csv",
    index=False
)


# ------------------------------------------------------------
# Retention matrix
# ------------------------------------------------------------

retention = pd.DataFrame(

    [

        [
            "Raw candidate data",
            "Raw candidate identifiers and attributes",
            "Restricted",
            "Retain only for the institution-approved operational/assessment period; delete or anonymize after expiry."
        ],

        [
            "Exam responses",
            "Item-level response records",
            "Restricted / analytical access",
            "Retain for the approved assessment and audit period; then archive/anonymize according to institutional policy."
        ],

        [
            "Proctoring flags",
            "Flag events associated with sessions",
            "Highly restricted",
            "Retain only as long as needed for review, appeals, audit, and applicable requirements."
        ],

        [
            "Reviewer records",
            "Verdicts, review time, reviewer identifiers",
            "Restricted",
            "Retain for quality assurance and audit period; then anonymize where possible."
        ],

        [
            "Ground truth / validation data",
            "Synthetic validation truth in this project",
            "Validation-only",
            "Never expose in production reviewer workflows; keep separate from analytical/public outputs."
        ],

        [
            "Aggregated analytics",
            "Non-identifying institutional metrics",
            "Broader read access",
            "May be retained longer when no unnecessary individual-level data remains."
        ],

        [
            "Audit logs",
            "Access and administrative events",
            "Restricted",
            "Retain according to institutional security/audit requirements."
        ]
    ],

    columns=[
        "data_class",
        "contents",
        "sensitivity",
        "retention_rule"
    ]
)


retention.to_csv(
    D16_DIR /
    "retention_policy.csv",
    index=False
)


# ------------------------------------------------------------
# Security controls
# ------------------------------------------------------------

controls = pd.DataFrame(

    [

        [
            "Least privilege",
            "Grant only the minimum permissions required for the role."
        ],

        [
            "Role-based access control",
            "Use the defined role matrix for data access."
        ],

        [
            "Separation of duties",
            "Separate data engineering, review, administration, and validation responsibilities."
        ],

        [
            "Ground-truth isolation",
            "Keep validation truth outside reviewer-facing analytical tables."
        ],

        [
            "Audit logging",
            "Log administrative and sensitive-data access events."
        ],

        [
            "Data minimization",
            "Expose only fields necessary for the user workflow."
        ],

        [
            "Pseudonymization",
            "Prefer candidate keys or pseudonymous identifiers in analytical outputs."
        ],

        [
            "Encryption",
            "Encrypt data in transit and at rest where supported by the deployment environment."
        ],

        [
            "Access review",
            "Periodically review active accounts and role assignments."
        ],

        [
            "Secure deletion",
            "Apply approved deletion/anonymization procedures when retention expires."
        ]
    ],

    columns=[
        "control",
        "design_requirement"
    ]
)


controls.to_csv(
    D16_DIR /
    "security_controls.csv",
    index=False
)


# ------------------------------------------------------------
# Policy document
# ------------------------------------------------------------

policy_md = """
# D16 — Access Control Design and Retention Policy

## Purpose

This design separates operational, analytical, review, administrative,
and validation access for IntegrityInsight.

It is a project design artifact and should be aligned with the
institution's legal, contractual, security, and records-retention
requirements before production deployment.

## Access Model

The implementation uses role-based access control (RBAC) with
least privilege.

Reviewers should see only cases assigned to them.

Analysts should primarily use warehouse and aggregated analytical data.

Hidden validation ground truth must remain outside reviewer workflows.

## Core Controls

1. Least privilege.
2. Role-based access control.
3. Separation of duties.
4. Ground-truth isolation.
5. Audit logging.
6. Data minimization.
7. Pseudonymization in analytics.
8. Encryption in transit and at rest.
9. Periodic access review.
10. Secure deletion/anonymization after the approved retention period.

## Retention Principle

The project does not define a universal legal retention period.

Retention must be set by the institution based on assessment policy,
appeals/audit requirements, applicable law, contracts, and security
requirements.

The retention matrix defines the intended treatment by data class
without inventing a jurisdiction-specific legal deadline.

## Production Implementation Notes

- Keep raw and validation-only data in separate storage boundaries.
- Use separate service accounts for ingestion and analytics.
- Log access to raw candidate and proctoring data.
- Do not expose validation truth through dashboards or reviewer interfaces.
- Prefer aggregated reporting when individual-level records are not required.
"""


(
    D16_DIR /
    "D16_access_control_retention_policy.md"
).write_text(
    policy_md,
    encoding="utf-8"
)


# ============================================================
# FINAL MANIFEST
# ============================================================

manifest = {

    "project":
        "IntegrityInsight",

    "deliverables": {

        "D11":
            sorted(
                str(
                    p.relative_to(
                        DELIVERABLE_DIR
                    )
                )
                for p in D11_DIR.iterdir()
            ),

        "D12":
            sorted(
                str(
                    p.relative_to(
                        DELIVERABLE_DIR
                    )
                )
                for p in D12_DIR.iterdir()
            ),

        "D14":
            sorted(
                str(
                    p.relative_to(
                        DELIVERABLE_DIR
                    )
                )
                for p in D14_DIR.iterdir()
            ),

        "D16":
            sorted(
                str(
                    p.relative_to(
                        DELIVERABLE_DIR
                    )
                )
                for p in D16_DIR.iterdir()
            )
    }
}


with open(
    DELIVERABLE_DIR /
    "deliverables_manifest.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2
    )


print("\n" + "=" * 80)
print("ALL FOUR DELIVERABLES COMPLETED")
print("=" * 80)


for code, directory in [

    ("D11", D11_DIR),

    ("D12", D12_DIR),

    ("D14", D14_DIR),

    ("D16", D16_DIR)

]:

    print(f"\n{code}: {directory}")

    for file in sorted(
        directory.iterdir()
    ):

        print(
            "  -",
            file.name
        )


D11 — ITEM ANALYSIS REPORT
D11 complete — 1,000 items analyzed.

D12 — COST MODEL & CAPACITY ANALYSIS
D12 complete — total synthetic cost = 879.0388.

D14 — METRIC DICTIONARY
D14 complete — 24 metrics documented.

D16 — ACCESS CONTROL DESIGN + RETENTION POLICY

ALL FOUR DELIVERABLES COMPLETED

D11: /content/integrityinsight_data/deliverables/D11_item_analysis
  - D11_item_analysis_report.md
  - exam_item_summary.csv
  - item_analysis_report.csv
  - item_analysis_summary.csv

D12: /content/integrityinsight_data/deliverables/D12_cost_capacity
  - D12_cost_capacity_report.md
  - capacity_cost_scenarios.csv
  - capacity_summary.csv
  - cost_mix.csv
  - cost_summary.csv
  - reviewer_capacity_observed.csv

D14: /content/integrityinsight_data/deliverables/D14_metric_dictionary
  - D14_metric_dictionary.md
  - metric_dictionary.csv
  - metric_dictionary.json

D16: /content/integrityinsight_data/deliverables/D16_access_retention
  - D16_access_control_retention_policy.md
  - access_control_rol

In [ ]:

# ============================================================
# INTEGRITYINSIGHT — D1 + D4 DOCUMENTATION / REPRODUCIBILITY
# Run this cell after D11/D12/D14/D16.
# ============================================================

from pathlib import Path
import json
import hashlib
import pandas as pd

DATA_DIR = Path.cwd() / "integrityinsight_data"
DELIVERABLE_DIR = DATA_DIR / "deliverables"
D1_DIR = DELIVERABLE_DIR / "D1_source_repository"
D4_DIR = DELIVERABLE_DIR / "D4_dimensional_model_erd"

D1_DIR.mkdir(parents=True, exist_ok=True)
D4_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# D1 — Source repository / documented, reproducible
# ------------------------------------------------------------

requirements = """numpy>=1.24
pandas>=2.0
"""

(D1_DIR / "requirements.txt").write_text(requirements, encoding="utf-8")

readme = """# IntegrityInsight

## Purpose
Reproducible synthetic assessment-integrity analytics project.

## Run order
1. Generator / D2 — source data generation
2. D3 ingestion and transformation
3. D4 dimensional model
4. D5 data-quality test suite
5. D11 item analysis
6. D12 cost and capacity analysis
7. D14 metric dictionary
8. D16 access-control and retention design

## Reproducibility
- Python: 3.10+
- Random seed: 42
- Default run: 300 candidates, 20 exams, 50 items/exam
- Full-scale option: set `FULL_RUN = True` in the generator.
- All project outputs are written under `integrityinsight_data/`.
- The generator uses synthetic data and synthetic monetary units.

## Important deployment note
D1 is repository-ready documentation. A hosted Git repository/URL still
has to be created by the team if the rubric requires a public/private
repository link.
"""
(D1_DIR / "README.md").write_text(readme, encoding="utf-8")

gitignore = """__pycache__/
*.pyc
.ipynb_checkpoints/
.env
.DS_Store
"""
(D1_DIR / ".gitignore").write_text(gitignore, encoding="utf-8")

run_manifest = {
    "project": "IntegrityInsight",
    "seed": 42,
    "default_run": {
        "FULL_RUN": False,
        "N_CANDIDATES": 300,
        "N_EXAMS": 20,
        "ITEMS_PER_EXAM": 50,
        "N_REVIEWERS": 12
    },
    "execution_order": [
        "generator",
        "ingestion",
        "transformation",
        "data_quality",
        "item_analysis",
        "cost_capacity",
        "metric_dictionary",
        "access_retention"
    ],
    "synthetic_data": True,
    "monetary_units": "synthetic currency unit"
}
(D1_DIR / "run_manifest.json").write_text(
    json.dumps(run_manifest, indent=2),
    encoding="utf-8"
)

# ------------------------------------------------------------
# D4 — Dimensional model + ERD
# ------------------------------------------------------------

erd_mermaid = """erDiagram
    DIM_CANDIDATE ||--o{ FACT_RESPONSE : candidate_key
    DIM_ITEM ||--o{ FACT_RESPONSE : item_key
    DIM_EXAM ||--o{ FACT_RESPONSE : exam_key
    DIM_DATE ||--o{ FACT_RESPONSE : date_key

    DIM_CANDIDATE ||--o{ FACT_SESSION_RISK : candidate_key
    DIM_EXAM ||--o{ FACT_SESSION_RISK : exam_key
    DIM_DATE ||--o{ FACT_SESSION_RISK : date_key

    DIM_CANDIDATE ||--o{ FACT_PROCTORING_FLAG : candidate_key
    DIM_EXAM ||--o{ FACT_PROCTORING_FLAG : exam_key
    DIM_FLAG_TYPE ||--o{ FACT_PROCTORING_FLAG : flag_key
    DIM_DATE ||--o{ FACT_PROCTORING_FLAG : date_key

    DIM_CANDIDATE ||--o{ FACT_REVIEW : candidate_key
    DIM_EXAM ||--o{ FACT_REVIEW : exam_key
    DIM_REVIEWER ||--o{ FACT_REVIEW : reviewer_key
    DIM_DATE ||--o{ FACT_REVIEW : date_key

    DIM_EXAM {
        int exam_key PK
        string exam_id
        int exam_version
        date valid_from
        date valid_to
        boolean is_current
    }

    DIM_CANDIDATE {
        int candidate_key PK
        string candidate_id
        float ability_theta
    }

    DIM_ITEM {
        int item_key PK
        string item_id
        string exam_id
        int item_number
        string correct_option
        float difficulty_b
        float discrimination_a
        float guessing_c
    }

    DIM_REVIEWER {
        int reviewer_key PK
        string reviewer_id
        float agreement_rate_configured
        int reviewer_version
        boolean is_current
    }

    DIM_DATE {
        int date_key PK
        date full_date
        int year
        int month
        int day
    }

    DIM_FLAG_TYPE {
        int flag_key PK
        string flag_type
    }

    FACT_RESPONSE {
        int response_key PK
        int candidate_key FK
        int item_key FK
        int exam_key FK
        int date_key FK
        string response_id
        string session_id
        boolean is_correct
        int response_time_ms
        int revisions
    }

    FACT_SESSION_RISK {
        int session_key PK
        int candidate_key FK
        int exam_key FK
        int date_key FK
        string session_id
        float overall_risk
        string recommendation
        int flag_count
    }

    FACT_PROCTORING_FLAG {
        int proctoring_flag_key PK
        string flag_id
        string session_id
        int candidate_key FK
        int exam_key FK
        int flag_key FK
        int date_key FK
        float confidence
        int t_start_ms
        int t_end_ms
    }

    FACT_REVIEW {
        int review_key PK
        string session_id
        int candidate_key FK
        int exam_key FK
        int reviewer_key FK
        int date_key FK
        string review_id
        string verdict
        float review_seconds
    }
"""

(D4_DIR / "integrityinsight_erd.mmd").write_text(
    erd_mermaid, encoding="utf-8"
)

d4_readme = """# D4 — Dimensional Model + ERD

## Dimensions
- DIM_CANDIDATE
- DIM_EXAM (SCD Type 2)
- DIM_ITEM
- DIM_REVIEWER (SCD Type 2)
- DIM_DATE
- DIM_FLAG_TYPE

## Facts
- FACT_RESPONSE
- FACT_SESSION_RISK
- FACT_PROCTORING_FLAG
- FACT_REVIEW

## Grain
- FACT_RESPONSE: one row per candidate response
- FACT_SESSION_RISK: one row per session
- FACT_PROCTORING_FLAG: one row per proctoring flag
- FACT_REVIEW: one row per reviewer decision

The ERD is provided as Mermaid source in `integrityinsight_erd.mmd`.
It can be rendered by Mermaid-compatible tools.
"""
(D4_DIR / "D4_dimensional_model.md").write_text(
    d4_readme, encoding="utf-8"
)

# Refresh manifest with the added D1/D4 artifacts.
manifest = {}
for code, directory in [
    ("D1", D1_DIR),
    ("D4", D4_DIR),
    ("D11", DELIVERABLE_DIR / "D11_item_analysis"),
    ("D12", DELIVERABLE_DIR / "D12_cost_capacity"),
    ("D14", DELIVERABLE_DIR / "D14_metric_dictionary"),
    ("D16", DELIVERABLE_DIR / "D16_access_retention"),
]:
    manifest[code] = sorted(
        str(p.relative_to(DELIVERABLE_DIR))
        for p in directory.iterdir()
        if p.is_file()
    )

(DELIVERABLE_DIR / "deliverables_manifest.json").write_text(
    json.dumps({"project": "IntegrityInsight", "deliverables": manifest}, indent=2),
    encoding="utf-8"
)

print("D1 and D4 documentation generated successfully.")
print("D1:", D1_DIR)
print("D4:", D4_DIR)


D1 and D4 documentation generated successfully.
D1: /content/integrityinsight_data/deliverables/D1_source_repository
D4: /content/integrityinsight_data/deliverables/D4_dimensional_model_erd


# D6–D10 QA / deliverable alignment notes

The D6–D10 cells below were reviewed against the deliverable list and corrected without changing the earlier ingestion/data-generation cells.

- **D6 Executive dashboard:** separate flagged-session rate from flag-events-per-session; keep the metric grain explicit in the KPI dictionary.
- **D7 Reviewer operations:** make Kappa interpretation safe when Kappa is unavailable; make drift output robust when no reviewer has enough reviews; report SLA targets/compliance only when an SLA target is actually present in the source data; remove hard-coded policy recommendations from the what-if output.
- **D8 Fairness dashboard:** use the seeded RNG for bootstrap reproducibility; distinguish groups reportable at `n>=30` from headline-eligible groups at `n>=100`; only fire the headline disparity alert when the minimum group size is at least 100.
- **D9 Collusion detection:** replace `eval()` with `ast.literal_eval()`; preserve `exam_id` in the pair file; save the empirical null explicitly; report whether the combined AUC actually improves rather than assuming it must.
- **D10 Worked case:** read the exam identifier from the D9 output when available; use the D9 empirical null for the WAA tail estimate; avoid describing an observed p-value distribution as an empirical false-positive rate; keep the case language explicitly investigative, not punitive or conclusive.

**Runtime note:** the notebook is prepared for execution, but D6–D10 still require the generated `integrityinsight_data` directory produced by the earlier cells.


In [ ]:
# %% D6 - Executive dashboard (FR-37) - appended, prior cells untouched
# ============================================================
# INTEGRITYINSIGHT - D6 EXECUTIVE DASHBOARD (FR-37, FR-11/12/13)
# Owner: D6 assignee | <=8 visuals | metrics traceable to dictionary
# ============================================================
from pathlib import Path
import json, math
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

def _data_dir():
    for p in [Path("/content/integrityinsight_data"), Path.cwd() / "integrityinsight_data"]:
        if p.exists(): return p
    return Path.cwd() / "integrityinsight_data"
DATA_DIR = _data_dir()
STG, WH = DATA_DIR/"staging", DATA_DIR/"warehouse"
DIM, FAC = WH/"dimensions", WH/"facts"
OUT = DATA_DIR/"deliverables"/"D6_executive_dashboard"
OUT.mkdir(parents=True, exist_ok=True)

def wilson(p, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    d = 1 + z*z/n
    c = (p + z*z/(2*n)) / d
    h = z*math.sqrt(max(p*(1-p)/n,0) + z*z/(4*n*n)) / d
    return (max(0,c-h), min(1,c+h))

# ---- Load (warehouse first, staging fallback) ----
fsr = pd.read_csv(FAC/"fact_session_risk.csv")
frv = pd.read_csv(FAC/"fact_review.csv")
dflag = pd.read_csv(DIM/"dim_flag_type.csv")
flags_stg = pd.read_csv(STG/"proctoring_flags.csv")  # flag_type, session_id
cost = pd.read_csv(STG/"infrastructure_cost.csv")
# flag_type per review: fact_review has flag_key -> join
frv2 = frv.merge(dflag[["flag_key","flag_type"]], on="flag_key", how="left")

n_sessions = len(fsr)
n_flags = len(flags_stg)
# Keep both grains explicit:
# - flagged_session_rate = sessions with >=1 flag / sessions
# - flag_events_per_session = all flag events / sessions
flagged_session_count = flags_stg["session_id"].nunique() if "session_id" in flags_stg.columns else np.nan
flagged_session_rate = float(flagged_session_count / max(1, n_sessions)) if pd.notna(flagged_session_count) else np.nan
flag_events_per_session = float(n_flags / max(1, n_sessions))
# Backward-compatible alias: "flag_rate" now means flagged-session rate.
flag_rate = flagged_session_rate

# ---- FR-12 operational precision by flag type (CONFIRMED/(CONFIRMED+OVERTURNED)) + Wilson ----
prec_rows = []
for ft, g in frv2.groupby("flag_type"):
    c = (g.verdict=="CONFIRMED").sum(); o = (g.verdict=="OVERTURNED").sum()
    denom = c+o; p = c/max(1,denom)
    lo, hi = wilson(p, denom)
    prec_rows.append(dict(flag_type=ft, n_reviewed=len(g), confirmed=int(c),
        overturned=int(o), precision=round(float(p),4), ci_lo=round(float(lo),4), ci_hi=round(float(hi),4)))
prec = pd.DataFrame(prec_rows).sort_values("precision")
weakest = prec.iloc[0]["flag_type"] if len(prec) else None
prec.to_csv(OUT/"operational_precision_by_flag.csv", index=False)

# ---- FR-11 volumes ----
vol = flags_stg.flag_type.value_counts().rename_axis("flag_type").reset_index(name="n_flags")
vol["share"] = (vol.n_flags/len(flags_stg)).round(4)
vol.to_csv(OUT/"flag_volume_by_type.csv", index=False)
rec = fsr.recommendation.value_counts().rename_axis("recommendation").reset_index(name="n")

# ---- FR-13 overturn analysis: 3+ findings ----
ov = frv2.copy(); ov["is_overturn"] = (ov.verdict=="OVERTURNED").astype(int)
over_by_flag = ov.groupby("flag_type")["is_overturn"].agg(["mean","count"]).reset_index().rename(columns={"mean":"overturn_rate"})
# overturn vs confidence
conf_ov = ov.groupby("confidence")["is_overturn"].mean() if "confidence" in ov else pd.Series(dtype=float)
# overturn vs reviewer workload (top reviewers overturn more?)
rev_ov = ov.groupby("reviewer_key")["is_overturn"].agg(["mean","count"]).reset_index()
findings = [
 f"1) {weakest} is the weakest flag: precision {prec.iloc[0]['precision']} (95% CI [{prec.iloc[0]['ci_lo']},{prec.iloc[0]['ci_hi']}]) with {prec.iloc[0]['n_reviewed']} reviews - suppressing/down-weighting it is the highest-leverage improvement (matches DR-4 expectation).",
 f"2) Overturn concentrates in low-confidence reviews: {conf_ov.to_dict() if len(conf_ov) else 'n/a'} - route low-confidence flags to senior reviewers or drop them.",
 f"3) Reviewer overturn spread is wide (min {rev_ov['mean'].min():.2f} - max {rev_ov['mean'].max():.2f}): strictness confounds flag quality; calibrate reviewers (see D7 funnel) instead of trusting raw overturn.",
 f"4) {rec.to_dict('records') if len(rec) else ''}: most sessions are NO_ACTION/ROUTINE - keep PRIORITY_REVIEW threshold high to protect review capacity.",
]
(OUT/"overturn_findings.md").write_text("# D6 overturn findings (FR-13)\n\n" + "\n\n".join(findings) + "\n", encoding="utf-8")

# ---- Cost per session ----
cps = cost.total_cost.mean(); comp = {k: float(cost[k].mean()) for k in ["compute_cost","storage_cost","egress_cost","review_labor_cost"]}
labor_share = comp["review_labor_cost"]/max(1e-9,cps)
(OUT/"executive_kpis.json").write_text(json.dumps(dict(
 n_sessions=int(n_sessions), n_flags=int(n_flags), flag_rate=round(float(flag_rate),4),
 cost_per_session=round(float(cps),4), cost_breakdown={k:round(v,4) for k,v in comp.items()},
 labor_share=round(float(labor_share),3), weakest_flag=str(weakest),
 flag_rate_formula="flagged sessions/flag events per session (grain: exam/period)",
 precision_formula="CONFIRMED/(CONFIRMED+OVERTURNED), INCONCLUSIVE excluded (grain: flag type), Wilson 95% CI"), indent=2), encoding="utf-8")

# ---- Plots (6 visuals, colorblind-safe, never color-alone: labels+values) ----
plt.rcParams.update({"figure.dpi":110})
# V1 volumes
fig,ax = plt.subplots(figsize=(8,3.5)); ax.bar(vol.flag_type, vol.n_flags, color="#0072B2", edgecolor="black")
ax.set_title("D6-V1 Flag volume by type (FR-11)"); ax.set_ylabel("flags"); plt.xticks(rotation=30, ha="right")
for x,y in zip(vol.flag_type, vol.n_flags): ax.text(x,y,str(y),ha="center",va="bottom",fontsize=8)
fig.tight_layout(); fig.savefig(OUT/"V1_flag_volume.png", bbox_inches="tight"); plt.close(fig)
# V2 precision + CI
fig,ax = plt.subplots(figsize=(8,3.8))
y = np.arange(len(prec)); lo = prec.precision-prec.ci_lo; hi = prec.ci_hi-prec.precision
ax.errorbar(prec.precision, y, xerr=[lo,hi], fmt="o", color="black", ecolor="black", capsize=4)
ax.set_yticks(y); ax.set_yticklabels(prec.flag_type); ax.set_xlabel("operational precision (Wilson 95% CI)")
ax.set_title("D6-V2 Operational precision by flag type (FR-12) - weakest highlighted"); ax.axvline(0.5, ls="--", c="gray")
for i,r in prec.reset_index().iterrows(): ax.text(r.precision+0.01, i, f"{r.precision:.2f}", va="center", fontsize=8)
fig.tight_layout(); fig.savefig(OUT/"V2_precision.png", bbox_inches="tight"); plt.close(fig)
# V3 recommendations
fig,ax = plt.subplots(figsize=(7,3.5)); ax.bar(rec.recommendation, rec.n, color="#009E73", edgecolor="black")
ax.set_title("D6-V3 Sessions by recommendation"); plt.xticks(rotation=20, ha="right")
for x,y in zip(rec.recommendation, rec.n): ax.text(x,y,str(y),ha="center",va="bottom",fontsize=8)
fig.tight_layout(); fig.savefig(OUT/"V3_recommendations.png", bbox_inches="tight"); plt.close(fig)
# V4 risk histogram
fig,ax = plt.subplots(figsize=(7,3.5)); ax.hist(fsr.risk_score.dropna(), bins=30, color="#56B4E9", edgecolor="black")
ax.set_title("D6-V4 Risk score distribution (non-separable check)"); ax.set_xlabel("risk_score"); ax.set_ylabel("sessions")
fig.tight_layout(); fig.savefig(OUT/"V4_risk_hist.png", bbox_inches="tight"); plt.close(fig)
# V5 overturn rate
o2 = over_by_flag.sort_values("overturn_rate")
fig,ax = plt.subplots(figsize=(8,3.5)); ax.barh(o2.flag_type, o2.overturn_rate, color="#D55E00", edgecolor="black")
ax.set_title("D6-V5 Overturn rate by flag type (FR-13)"); ax.set_xlabel("overturn rate")
for i,v in enumerate(o2.overturn_rate): ax.text(v+0.01, i, f"{v:.2f}", va="center", fontsize=8)
fig.tight_layout(); fig.savefig(OUT/"V5_overturn.png", bbox_inches="tight"); plt.close(fig)
# V6 cost breakdown
fig,ax = plt.subplots(figsize=(7,3.5)); ax.bar(list(comp.keys()), list(comp.values()), color="#CC79A7", edgecolor="black")
ax.set_title(f"D6-V6 Cost/session breakdown (mean={cps:.3f}; labour share={labor_share:.0%})"); plt.xticks(rotation=20, ha="right")
fig.tight_layout(); fig.savefig(OUT/"V6_cost.png", bbox_inches="tight"); plt.close(fig)

print("="*65); print("D6 EXECUTIVE DASHBOARD COMPLETE (FR-37: 6 visuals, <=8 OK)")
print(f"sessions={n_sessions} flags={n_flags} flagged_session_rate={flagged_session_rate:.3f} flag_events_per_session={flag_events_per_session:.3f} cost/session={cps:.4f} labour_share={labor_share:.0%}")
print(f"weakest flag (lowest precision) = {weakest}"); print(prec.to_string(index=False))
print("Findings:"); [print(" - "+f) for f in findings]; print("OUT:", OUT)


D6 EXECUTIVE DASHBOARD COMPLETE (FR-37: 6 visuals, <=8 OK)
sessions=6000 flags=3450 flagged_session_rate=0.446 flag_events_per_session=0.575 cost/session=0.1465 labour_share=40%
weakest flag (lowest precision) = GAZE_OFF_SCREEN
        flag_type  n_reviewed  confirmed  overturned  precision  ci_lo  ci_hi
  GAZE_OFF_SCREEN        1467        482         985     0.3286 0.3050 0.3530
   PHONE_DETECTED         455        249         206     0.5473 0.5013 0.5924
     MULTI_PERSON         198        117          81     0.5909 0.5213 0.6570
IDENTITY_MISMATCH          66         42          24     0.6364 0.5158 0.7419
     SECOND_VOICE         265        191          74     0.7208 0.6639 0.7713
    MULTI_MONITOR         526        395         131     0.7510 0.7123 0.7860
       TAB_SWITCH         786        629         157     0.8003 0.7709 0.8267
      REMOTE_TOOL          32         27           5     0.8438 0.6825 0.9314
Findings:
 - 1) GAZE_OFF_SCREEN is the weakest flag: precision 0.3286 

In [ ]:
# %% D7 - Reviewer operations dashboard (FR-38) - appended, prior cells untouched
# ============================================================
# INTEGRITYINSIGHT - D7 REVIEWER OPERATIONS DASHBOARD (FR-38)
# Covers FR-14 kappa, FR-15 drift funnel, FR-16 fatigue, FR-17 SLA, FR-18 what-if
# ============================================================
from pathlib import Path
import json, itertools
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

def _data_dir():
    for p in [Path("/content/integrityinsight_data"), Path.cwd() / "integrityinsight_data"]:
        if p.exists(): return p
    return Path.cwd() / "integrityinsight_data"
DATA_DIR = _data_dir()
STG, WH = DATA_DIR/"staging", DATA_DIR/"warehouse"
OUT = DATA_DIR/"deliverables"/"D7_reviewer_ops"
OUT.mkdir(parents=True, exist_ok=True)

rv = pd.read_csv(STG/"review_verdicts.csv")  # has flag_id, reviewer_id, verdict, review_seconds, position_in_shift, is_double_review
fsr = pd.read_csv(WH/"facts"/"fact_session_risk.csv")
try: gt_flags = pd.read_csv(DATA_DIR/"ground_truth"/"ground_truth_flags.csv")
except Exception: gt_flags = None

# ---- Throughput ----
thr = rv.groupby("reviewer_id").agg(n_reviews=("review_id","count"),
  median_sec=("review_seconds","median"), p90_sec=("review_seconds", lambda s: float(np.quantile(s,0.9))),
  confirm_rate=("verdict", lambda s: float((s=="CONFIRMED").mean())),
  overturn_rate=("verdict", lambda s: float((s=="OVERTURNED").mean()))).reset_index()
thr["rev_per_hour"] = (3600/thr.median_sec).round(2)
thr.to_csv(OUT/"reviewer_throughput.csv", index=False)

# ---- FR-14 kappa overall + per pair ----
def cohen_kappa(a, b):
    # binary CONFIRMED=1 / OVERTURNED=0
    m = {"CONFIRMED":1,"OVERTURNED":0}
    a = np.array([m.get(x,np.nan) for x in a]); b = np.array([m.get(x,np.nan) for x in b])
    keep = ~(np.isnan(a)|np.isnan(b))
    a,b = a[keep].astype(int), b[keep].astype(int)
    if len(a) < 10: return np.nan, len(a)
    try:
        from sklearn.metrics import cohen_kappa_score
        return float(cohen_kappa_score(a,b)), len(a)
    except Exception:
        po = (a==b).mean(); pe = a.mean()*b.mean()+(1-a.mean())*(1-b.mean())
        return float((po-pe)/max(1e-9,1-pe)), len(a)
dbl = rv[rv.is_double_review.astype(str).str.lower().isin(["true", "1", "t"])]
pairs = []
for fid, g in dbl.groupby("flag_id"):
    g = g.sort_values("review_id")
    if len(g)==2 and g.iloc[0].reviewer_id != g.iloc[1].reviewer_id:
        pairs.append((g.iloc[0].reviewer_id, g.iloc[1].reviewer_id, g.iloc[0].verdict, g.iloc[1].verdict))
overall_k = np.nan
if pairs:
    A=[p[2] for p in pairs]; B=[p[3] for p in pairs]
    overall_k,_ = cohen_kappa(A,B)
# per reviewer-pair
import collections
pairk = []
dd = collections.defaultdict(list)
for r1,r2,v1,v2 in pairs: dd[tuple(sorted([r1,r2]))].append((v1,v2))
for (r1,r2), vs in dd.items():
    k,n = cohen_kappa([v[0] for v in vs],[v[1] for v in vs])
    pairk.append(dict(reviewer_a=r1, reviewer_b=r2, kappa=round(float(k),3) if k==k else None, n_shared=n))
pairk = pd.DataFrame(pairk).sort_values("kappa") if pairk else pd.DataFrame()
if len(pairk): pairk.to_csv(OUT/"kappa_per_pair.csv", index=False)
if pd.isna(overall_k):
    interp = "not available"
elif overall_k < 0.2:
    interp = "slight"
elif overall_k < 0.4:
    interp = "fair"
elif overall_k < 0.6:
    interp = "moderate"
else:
    interp = "substantial"
kappa_display = "n/a" if pd.isna(overall_k) else f"{overall_k:.3f}"
print(f"FR-14 overall double-review kappa={kappa_display} ({interp} by Landis-Koch), n_pairs={len(pairs)}")

# ---- FR-15 drift: early-half vs late-half confirm rate per reviewer ----
drift=[]
for rid, g in rv.groupby("reviewer_id"):
    g = g.sort_values("position_in_shift"); h=len(g)//2
    if h<20: continue
    e = (g.iloc[:h].verdict=="CONFIRMED").mean(); l = (g.iloc[h:].verdict=="CONFIRMED").mean()
    drift.append(dict(reviewer_id=rid, n=len(g), early_confirm=round(float(e),3), late_confirm=round(float(l),3), shift=round(float(l-e),3)))
drift = pd.DataFrame(drift)
if len(drift):
    drift = drift.sort_values("shift")
drift.to_csv(OUT/"reviewer_drift.csv", index=False)
sus = drift.nsmallest(3,"shift")["reviewer_id"].tolist() if len(drift) else []
print("FR-15 drift suspects (largest negative shift):", sus)

# ---- FR-16 fatigue: confirm-agreement proxy vs position + fitted curve + test ----
# accuracy needs truth: use flag ground truth if available, else overturn-rate drift as proxy
if gt_flags is not None:
    truth = gt_flags.set_index("flag_id")["ground_truth_flag"].to_dict()
    rv["is_true"] = rv.flag_id.map(truth)
    rv["correct"] = (((rv.verdict=="CONFIRMED")&(rv.is_true==1))|((rv.verdict=="OVERTURNED")&(rv.is_true==0))).astype(int)
    ycol="correct"; ylab="accuracy vs ground-truth flag"
else:
    rv["correct"] = (rv.verdict=="CONFIRMED").astype(int); ycol="correct"; ylab="confirm rate (proxy; truth file absent)"
rv["pos_bin"] = pd.cut(rv.position_in_shift, bins=[0,10,20,30,40,60,100,10**9], labels=["1-10","11-20","21-30","31-40","41-60","61-100","100+"])
fat = rv.groupby("pos_bin", observed=True)["correct"].agg(["mean","count"]).reset_index()
x = rv.position_in_shift.to_numpy(dtype=float); y = rv[ycol].to_numpy(dtype=float)
slope = float(np.polyfit(x, y, 1)[0])
try:
    from scipy.stats import spearmanr
    rho, pval = spearmanr(x, y)
except Exception:
    rho = float(np.corrcoef(x, y)[0,1]); pval = np.nan
fat.to_csv(OUT/"fatigue_curve.csv", index=False)
print(f"FR-16 fatigue: slope={slope:.5f}/position, spearman rho={rho:.3f} p={pval:.3g} ({ylab}).")

# ---- FR-17 queue & SLA ----
# time proxy: session order; backlog = cumulative flags - cumulative reviews (1 review per flag + doubles)
q = pd.DataFrame({"flag_id": rv.flag_id.unique()}); q["order"]=np.arange(len(q))
q["backlog"] = np.arange(1,len(q)+1) - np.arange(1,len(q)+1)  # all reviewed in sim -> backlog 0 at end; show arrival burst instead
# FR-17 robust: review_verdicts already has session_id, so avoid session_id_x/_y collision
flag_sess = pd.read_csv(STG/"proctoring_flags.csv")[["flag_id","session_id"]].rename(columns={"session_id": "flag_session_id"})
rvj = rv.merge(flag_sess, on="flag_id", how="left")
# unify session id into sess_id
if "session_id" in rvj.columns:
    rvj["sess_id"] = rvj["session_id"]
elif "session_id_x" in rvj.columns:
    rvj["sess_id"] = rvj["session_id_x"].fillna(rvj.get("session_id_y"))
else:
    rvj["sess_id"] = rvj["flag_session_id"]
# recommendation/risk: prefer warehouse fact, fallback to staging session_risk
sess_src = None
if "session_id" in fsr.columns and "recommendation" in fsr.columns:
    sess_src = fsr[["session_id","recommendation"]].copy()
else:
    try:
        _st = pd.read_csv(STG/"session_risk.csv")
        if "session_id" in _st.columns and "recommendation" in _st.columns:
            sess_src = _st[["session_id","recommendation"]].copy()
    except Exception:
        sess_src = None
if sess_src is not None:
    rvj = rvj.merge(sess_src.rename(columns={"session_id": "sess_id"}), on="sess_id", how="left")
_rec_col = "recommendation" if "recommendation" in rvj.columns else ("recommendation_x" if "recommendation_x" in rvj.columns else None)
assert _rec_col is not None, "recommendation column missing for SLA"

# Use an SLA target only if the source data explicitly provides one.
# Do not invent time targets when the dataset/SRS does not contain them.
_target_candidates = ["sla_target_seconds", "sla_seconds", "target_review_seconds"]
_target_col = next((c for c in _target_candidates if c in rvj.columns), None)
sla = rvj.groupby(_rec_col)["review_seconds"].agg(
    count="count",
    median="median",
    p90=lambda s: float(np.quantile(s,0.9))
).reset_index().rename(columns={_rec_col: "recommendation"})
if _target_col is not None:
    target = rvj.groupby(_rec_col)[_target_col].median().reset_index().rename(
        columns={_rec_col:"recommendation", _target_col:"sla_target_seconds"})
    sla = sla.merge(target, on="recommendation", how="left")
    compliance = rvj.assign(
        _within_sla=rvj["review_seconds"] <= rvj[_target_col]
    ).groupby(_rec_col)["_within_sla"].mean().reset_index().rename(
        columns={_rec_col:"recommendation", "_within_sla":"sla_compliance"})
    sla = sla.merge(compliance, on="recommendation", how="left")
else:
    sla["sla_target_seconds"] = np.nan
    sla["sla_compliance"] = np.nan
sla["sla_target_source"] = "source column" if _target_col is not None else "not provided"
sla.to_csv(OUT/"sla_by_priority.csv", index=False)
print("FR-17 SLA target:", _target_col if _target_col is not None else "not provided; reporting review-time distribution only")

# ---- FR-18 threshold what-if tool ----
def what_if(thresholds=(0.3,0.5,0.75), sec_per_review=None, hourly_rate=25.0):
    sec = sec_per_review or float(rv.review_seconds.median())
    rows=[]
    for t in thresholds:
        sel = fsr[fsr.overall_risk>=t]; vol = int(len(sel))
        # precision/recall vs candidate violation if ground truth available (validation only)
        rec_out, prec_out = (np.nan, np.nan)
        try:
            gtc = pd.read_csv(DATA_DIR/"ground_truth"/"ground_truth_candidates.csv").set_index("candidate_id")["violation"]
            # session truth via candidate: need candidate_id per session -> from staging sessions
            sess = pd.read_csv(STG/"sessions.csv").merge(fsr[["session_id","overall_risk"]], on="session_id", how="left")
            sess["viol"] = sess.candidate_id.map(gtc).fillna(0).astype(int)
            tp = int(((sess.overall_risk>=t)&(sess.viol==1)).sum()); fp = int(((sess.overall_risk>=t)&(sess.viol==0)).sum()); fn = int(((sess.overall_risk<t)&(sess.viol==1)).sum())
            prec_out = tp/max(1,tp+fp); rec_out = tp/max(1,tp+fn)
        except Exception: pass
        hours = vol*sec/3600; staff = hours/40; cost = hours*hourly_rate
        rows.append(dict(threshold=t, review_volume=vol, precision=round(float(prec_out),3) if prec_out==prec_out else None,
          recall=round(float(rec_out),3) if rec_out==rec_out else None, hours=round(hours,1), fte_weekly=round(staff,2), cost_usd=round(cost,0)))
    return pd.DataFrame(rows)
wi = what_if(); wi.to_csv(OUT/"threshold_what_if.csv", index=False)
print("FR-18 what-if:\n", wi.to_string(index=False))
print("FR-18 what-if is a scenario comparison; no policy threshold is hard-coded as a recommendation.")

# ---- Plots (6) ----
plt.rcParams.update({"figure.dpi":110})
fig,ax = plt.subplots(figsize=(8,3.5)); ax.bar(thr.reviewer_id, thr.n_reviews, color="#0072B2", edgecolor="black")
ax.set_title("D7-1 Reviews per reviewer (load balance)"); plt.xticks(rotation=30, ha="right")
fig.tight_layout(); fig.savefig(OUT/"D7_throughput.png", bbox_inches="tight"); plt.close(fig)
fig,ax = plt.subplots(figsize=(8,3.8))
ax.scatter(thr.n_reviews, thr.confirm_rate, s=80, color="#009E73", edgecolors="black")
for _,r in thr.iterrows(): ax.text(r.n_reviews, r.confirm_rate, r.reviewer_id, fontsize=7)
ax.set_xlabel("review volume (n)"); ax.set_ylabel("confirm rate"); ax.set_title("D7-2 Funnel-style calibration (FR-15): outliers = miscalibrated/drifting")
fig.tight_layout(); fig.savefig(OUT/"D7_funnel.png", bbox_inches="tight"); plt.close(fig)
fig,ax = plt.subplots(figsize=(7,3.5)); ax.plot(fat.pos_bin.astype(str), fat["mean"], marker="o", color="black")
ax.set_title(f"D7-3 Fatigue curve (FR-16) slope={slope:.4f}, rho={rho:.2f}"); ax.set_ylabel(ylab); plt.xticks(rotation=25, ha="right")
fig.tight_layout(); fig.savefig(OUT/"D7_fatigue.png", bbox_inches="tight"); plt.close(fig)
fig,ax = plt.subplots(figsize=(7,3.5)); ax.bar(sla.recommendation.astype(str), sla["median"], color="#D55E00", edgecolor="black")
ax.set_title("D7-4 Review time by priority (FR-17; SLA target if provided)"); plt.xticks(rotation=15, ha="right")
fig.tight_layout(); fig.savefig(OUT/"D7_sla.png", bbox_inches="tight"); plt.close(fig)
fig,ax = plt.subplots(figsize=(7,3.5)); ax.bar(wi.threshold.astype(str), wi.review_volume, color="#CC79A7", edgecolor="black")
ax.set_title("D7-5 What-if review volume by threshold (FR-18)"); ax.set_xlabel("risk threshold"); ax.set_ylabel("sessions to review")
fig.tight_layout(); fig.savefig(OUT/"D7_whatif.png", bbox_inches="tight"); plt.close(fig)
fig,ax = plt.subplots(figsize=(7,3.5))
d2 = drift.sort_values("shift"); ax.barh(d2["reviewer_id"].astype(str), d2["shift"].to_numpy(dtype=float), color="#56B4E9", edgecolor="black")
ax.set_title("D7-6 Reviewer drift: late-minus-early confirm shift (FR-15)"); ax.axvline(0, c="black")
fig.tight_layout(); fig.savefig(OUT/"D7_drift.png", bbox_inches="tight"); plt.close(fig)

print("="*65); print("D7 REVIEWER OPS COMPLETE. OUT:", OUT)
print(thr.to_string(index=False))


FR-14 overall double-review kappa=0.605 (substantial by Landis-Koch), n_pairs=345
FR-15 drift suspects (largest negative shift): ['reviewer_03', 'reviewer_05', 'reviewer_12']
FR-16 fatigue: slope=-0.00014/position, spearman rho=-0.033 p=0.0444 (accuracy vs ground-truth flag).
FR-17 SLA target: not provided; reporting review-time distribution only
FR-18 what-if:
  threshold  review_volume  precision  recall  hours  fte_weekly  cost_usd
      0.30           2128      0.234   0.889  142.5        3.56    3562.0
      0.50            446      0.462   0.368   29.9        0.75     747.0
      0.75             22      0.727   0.029    1.5        0.04      37.0
FR-18 what-if is a scenario comparison; no policy threshold is hard-coded as a recommendation.
D7 REVIEWER OPS COMPLETE. OUT: /content/integrityinsight_data/deliverables/D7_reviewer_ops
reviewer_id  n_reviews  median_sec  p90_sec  confirm_rate  overturn_rate  rev_per_hour
reviewer_01        319     248.260  463.174      0.557994       0.

In [ ]:
# %% D8 - Fairness dashboard (FR-39) - appended, prior cells untouched
# ============================================================
# INTEGRITYINSIGHT - D8 FAIRNESS DASHBOARD (FR-28..32, FR-39)
# Sliced metrics, disparity ratios + bootstrap CI, tests + effect sizes, proxy analysis
# ============================================================
from pathlib import Path
import json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

def _data_dir():
    for p in [Path("/content/integrityinsight_data"), Path.cwd() / "integrityinsight_data"]:
        if p.exists(): return p
    return Path.cwd() / "integrityinsight_data"
DATA_DIR = _data_dir(); STG = DATA_DIR/"staging"
OUT = DATA_DIR/"deliverables"/"D8_fairness_dashboard"; OUT.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(42)

risk = pd.read_csv(STG/"session_risk.csv")  # has nuisance JSON, overall_risk, flag_count, recommendation
flags = pd.read_csv(STG/"proctoring_flags.csv")
rv = pd.read_csv(STG/"review_verdicts.csv")
# parse nuisance
import json as _js
def _parse(s):
    try: return _js.loads(s) if isinstance(s,str) else {}
    except Exception: return {}
nu = risk.nuisance.apply(_parse).apply(pd.Series)
for c in ["lighting","webcam_class","noise_level","connection"]:
    if c not in nu: nu[c]="unknown"
risk = pd.concat([risk, nu[["lighting","webcam_class","noise_level","connection"]]], axis=1)
risk["flagged"] = (risk.flag_count>0).astype(int)
# overturn per session: any OVERTURNED review on its flags
rv_flag = rv.merge(flags[["flag_id","session_id"]], on="flag_id", how="left", suffixes=("","_f"))
sess_ov = rv_flag.groupby("session_id")["verdict"].apply(lambda s: int((s=="OVERTURNED").any())).rename("overturned")
risk = risk.merge(sess_ov, on="session_id", how="left"); risk["overturned"] = risk.overturned.fillna(0).astype(int)

DIMS = ["lighting","webcam_class","noise_level","connection","recommendation","exam_id"]
# FR-28 full sliced table (no metric aggregate-only)
rows=[]
for d in DIMS:
    for g, df in risk.groupby(d):
        n = len(df)
        if n==0: continue
        rows.append(dict(dimension=d, group=str(g), n=n,
          flag_rate=round(float(df.flagged.mean()),4),
          overturn_rate=round(float(df[df.flagged==1].overturned.mean()) if (df.flagged==1).any() else np.nan,4),
          mean_risk=round(float(df.overall_risk.mean()),4)))
sliced = pd.DataFrame(rows); sliced.to_csv(OUT/"fairness_sliced_table.csv", index=False)

# FR-29 disparity ratios + bootstrap CI.
# Report groups with n>=30; mark n>=100 as headline-eligible to match the SRS rule.
def boot_ratio(a, b, B=1000):
    a=np.asarray(a); b=np.asarray(b); est=a.mean()/max(1e-9,b.mean())
    bs=[]
    for _ in range(B):
        try:
            aa = rng.choice(a, len(a), replace=True).mean()
            bb = rng.choice(b, len(b), replace=True).mean()
            bs.append(aa/max(1e-9,bb))
        except Exception:
            pass
    if not bs:
        return est, np.nan, np.nan
    return est, float(np.quantile(bs,0.025)), float(np.quantile(bs,0.975))
disp=[]
for d in DIMS:
    groups = {g: df for g,df in risk.groupby(d) if len(df)>=30}
    if len(groups)<2: continue
    rates = {g: float(df.flagged.mean()) for g,df in groups.items()}
    gmax, gmin = max(rates,key=rates.get), min(rates,key=rates.get)
    est,lo,hi = boot_ratio((groups[gmax].flagged.to_numpy()), (groups[gmin].flagged.to_numpy()))
    # overturn disparity too
    o = {g: float(df[df.flagged==1].overturned.mean()) if (df.flagged==1).any() else np.nan for g,df in groups.items()}
    min_group_n = min(len(groups[gmax]), len(groups[gmin]))
    headline_eligible = min_group_n >= 100
    disp.append(dict(dimension=d, max_group=str(gmax), min_group=str(gmin),
      max_group_n=len(groups[gmax]), min_group_n=len(groups[gmin]),
      max_rate=round(rates[gmax],4), min_rate=round(rates[gmin],4),
      disparity_ratio=round(float(est),3), ci_lo=round(lo,3), ci_hi=round(hi,3),
      headline_eligible=headline_eligible,
      alert=bool(headline_eligible and (est>1.25 or (est<0.8 and est>0))),
      overturn_by_group=str({k:(round(v,3) if v==v else None) for k,v in o.items()})))
disp = pd.DataFrame(disp); disp.to_csv(OUT/"disparity_ratios.csv", index=False)

# FR-30 tests + effect sizes (chi-square + Cramer's V; practical vs statistical)
from math import sqrt
tests=[]
for d in ["lighting","webcam_class","noise_level","connection"]:
    ct = pd.crosstab(risk[d], risk.flagged)
    try:
        from scipy.stats import chi2_contingency
        chi2,p,dof,ex = chi2_contingency(ct); V = sqrt(chi2/(len(risk)*(min(ct.shape)-1)))
    except Exception:
        # manual chi-square
        ex = np.outer(ct.sum(1).to_numpy(), ct.sum(0).to_numpy())/ct.sum().sum()
        chi2 = float(((ct.to_numpy()-ex)**2/np.maximum(ex,1e-9)).sum()); p=np.nan; V=sqrt(chi2/(len(risk)*(min(ct.shape)-1)))
    tests.append(dict(dimension=d, chi2=round(float(chi2),2), p_value=(f"{p:.3g}" if p==p else "n/a"),
      cramers_V=round(float(V),3), practical="small" if V<0.1 else "medium" if V<0.3 else "large",
      note="p-values scale with n; effect size decides action."))
pd.DataFrame(tests).to_csv(OUT/"disparity_tests.csv", index=False)

# FR-31 proxy analysis (written, traced in data)
proxy_txt = f"""# D8 proxy-variable analysis (FR-31)
Neutral slicing (e.g. webcam_class, lighting) can act as SES proxy.
Trace in this data:
- P(low webcam | poor lighting) vs P(low webcam | good lighting): {pd.crosstab(risk.lighting, risk.webcam_class, normalize='index').round(3).to_dict()}
- Flag rate by webcam: {risk.groupby('webcam_class').flagged.mean().round(3).to_dict()}
- Flag rate by lighting: {risk.groupby('lighting').flagged.mean().round(3).to_dict()}
Mechanism: low-end devices + poor lighting + unstable connection cluster (shared SES cause) -> nuisance-driven false positives (DR-2) -> disparate impact even though no protected attribute is used (ETH-5 respected: demographics only for monitoring).
Action: down-weight GAZE_OFF_SCREEN/PHONE_DETECTED when channels_missing or nuisance=poor; monitor disparity trend weekly; alert at disparity>1.25 (justified: 25% excess flags triggers review; four-fifths 0.8 shown as reference, not legal standard here).
Limitation: synthetic nuisance; real SES mapping needs consented metadata (ETH-1).
"""
(OUT/"proxy_analysis.md").write_text(proxy_txt, encoding="utf-8")
(OUT/"alert_config.json").write_text(json.dumps({"threshold":1.25,"reference":"four-fifths 0.8 (reference only)","metrics":["flag_rate","overturn_rate"],"min_n_per_group":100,"action":"freeze threshold change + review D6-V2 + notify compliance"}, indent=2), encoding="utf-8")

# FR-32/39 dashboard plots (5)
plt.rcParams.update({"figure.dpi":110})
s1 = sliced[sliced.dimension=="webcam_class"]
fig,ax = plt.subplots(figsize=(7,3.5)); ax.bar(s1.group, s1.flag_rate, color="#0072B2", edgecolor="black")
ax.set_title("D8-1 Flag rate by webcam_class (FR-28)"); ax.set_ylabel("flag rate")
for x,y in zip(s1.group, s1.flag_rate): ax.text(x,y,f"{y:.2f}",ha="center",va="bottom",fontsize=8)
fig.tight_layout(); fig.savefig(OUT/"D8_webcam.png", bbox_inches="tight"); plt.close(fig)
s2 = sliced[sliced.dimension=="lighting"]
fig,ax = plt.subplots(figsize=(7,3.5)); ax.bar(s2.group, s2.flag_rate, color="#009E73", edgecolor="black")
ax.set_title("D8-2 Flag rate by lighting"); fig.tight_layout(); fig.savefig(OUT/"D8_lighting.png", bbox_inches="tight"); plt.close(fig)
fig,ax = plt.subplots(figsize=(8,3.8)); y=np.arange(len(disp))
ax.errorbar(disp.disparity_ratio, y, xerr=[disp.disparity_ratio-disp.ci_lo, disp.ci_hi-disp.disparity_ratio], fmt="o", color="black", ecolor="black", capsize=4)
ax.set_yticks(y); ax.set_yticklabels(disp.dimension); ax.axvline(1.25, ls="--", c="red", label="alert 1.25"); ax.axvline(1.0, c="gray")
ax.set_xlabel("disparity ratio max/min with 95% bootstrap CI (FR-29)"); ax.legend(); ax.set_title("D8-3 Disparity ratios + alert threshold")
for i,v in enumerate(disp.disparity_ratio): ax.text(v+0.02, i, f"{v:.2f}", va="center", fontsize=8)
fig.tight_layout(); fig.savefig(OUT/"D8_disparity.png", bbox_inches="tight"); plt.close(fig)
s3 = sliced[sliced.dimension=="connection"]
fig,ax = plt.subplots(figsize=(7,3.5)); ax.bar(s3.group, s3.overturn_rate, color="#D55E00", edgecolor="black")
ax.set_title("D8-4 Overturn rate by connection (among flagged)"); fig.tight_layout(); fig.savefig(OUT/"D8_overturn.png", bbox_inches="tight"); plt.close(fig)
piv = risk.groupby(["webcam_class","lighting"]).flagged.mean().unstack()
fig,ax = plt.subplots(figsize=(6,4)); im=ax.imshow(piv.to_numpy(), cmap="Blues")
ax.set_xticks(range(len(piv.columns)), piv.columns); ax.set_yticks(range(len(piv.index)), piv.index)
ax.set_title("D8-5 Proxy heatmap: flag rate webcam x lighting (FR-31)")
for i in range(len(piv.index)):
    for j in range(len(piv.columns)): ax.text(j,i,f"{piv.iloc[i,j]:.2f}",ha="center",va="center",fontsize=9)
fig.colorbar(im, ax=ax, label="flag rate"); fig.tight_layout(); fig.savefig(OUT/"D8_proxy_heatmap.png", bbox_inches="tight"); plt.close(fig)

fired = disp[disp.alert==True]
print("="*65); print("D8 FAIRNESS COMPLETE. alert threshold=1.25")
print(disp.to_string(index=False)); print("ALERTS FIRED:" if len(fired) else "NO ALERT (all ratios <=1.25)")
if len(fired): print(fired[["dimension","disparity_ratio"]].to_string(index=False))
print("OUT:", OUT)


D8 FAIRNESS COMPLETE. alert threshold=1.25
     dimension    max_group min_group  max_group_n  min_group_n  max_rate  min_rate  disparity_ratio  ci_lo  ci_hi  headline_eligible  alert                                                                                                                                                                                                                                                                                                                                                                         overturn_by_group
      lighting         poor      good         1231         4769    0.5443    0.4204            1.295  1.217  1.376               True   True                                                                                                                                                                                                                                                                                                          

In [ ]:
# %% D9 - Collusion detection module (D9) - appended, prior cells untouched
# ============================================================
# INTEGRITYINSIGHT - D9 COLLUSION DETECTION MODULE (FR-19..23, FR-27)
# 2 indices + permutation null + Bonferroni/BH + confound control + timing + network + triangulation
# Classical statistics only (CON-7). Ground truth = validation ONLY (CON-6).
# ============================================================
from pathlib import Path
import json, math, itertools, ast
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from math import comb

def _data_dir():
    for p in [Path("/content/integrityinsight_data"), Path.cwd() / "integrityinsight_data"]:
        if p.exists(): return p
    return Path.cwd() / "integrityinsight_data"
DATA_DIR=_data_dir(); STG=DATA_DIR/"staging"; WH=DATA_DIR/"warehouse"
OUT=DATA_DIR/"deliverables"/"D9_collusion_detection"; OUT.mkdir(parents=True, exist_ok=True)
rng=np.random.default_rng(42)

# ---- Load responses + keys ----
fr = pd.read_csv(WH/"facts"/"fact_response.csv")
dc = pd.read_csv(WH/"dimensions"/"dim_candidate.csv")[["candidate_key","candidate_id"]]
di = pd.read_csv(WH/"dimensions"/"dim_item.csv")[["item_key","item_id","exam_id"]]
fr = fr.merge(dc,on="candidate_key",how="left").merge(di,on="item_key",how="left")
fsr = pd.read_csv(WH/"facts"/"fact_session_risk.csv")
# validation-only ground truth
gtc = pd.read_csv(DATA_DIR/"ground_truth"/"ground_truth_candidates.csv")
gt_map = gtc.set_index("candidate_id")[["profile","violation"]].to_dict("index")
prof = gtc.set_index("candidate_id")["profile"].to_dict()

# ---- Blocking (NFR-3): single exam to keep ~pairs tractable; document ----
exam_pick = fr.exam_id.value_counts().idxmax()
d = fr[fr.exam_id==exam_pick].copy()
cands = sorted(d.candidate_id.unique())
MAXC = 500
if len(cands)>MAXC:
    keep = sorted(rng.choice(cands, MAXC, replace=False).tolist()); d = d[d.candidate_id.isin(keep)]; cands=keep
n=len(cands); m=n*(n-1)//2
print(f"Blocking: exam={exam_pick}, candidates={n}, pairs={m} (full 5000 -> ~12.5M; same-exam blocking + cap documented for NFR-3)")

items = sorted(d.item_id.unique())
piv_opt = d.pivot_table(index="candidate_id", columns="item_id", values="selected_option", aggfunc="first")
piv_cor = d.pivot_table(index="candidate_id", columns="item_id", values="is_correct", aggfunc="first")
piv_rt  = d.pivot_table(index="candidate_id", columns="item_id", values="response_time_ms", aggfunc="first")
piv_opt=piv_opt.reindex(index=cands,columns=items); piv_cor=piv_cor.reindex(index=cands,columns=items); piv_rt=np.log1p(piv_rt.reindex(index=cands,columns=items))

# ---- Index 1: wrong-answer agreement (WAA) ----
def waa_pair(a,b):
    wa = (piv_cor.loc[a]==0)&(piv_cor.loc[b]==0)
    den = int(wa.sum())
    if den==0: return 0.0,0,0
    agree = int(((piv_opt.loc[a]==piv_opt.loc[b])&wa).sum())
    return agree/max(1,den), agree, den
# ---- Index 2: Bellezza&Bellezza binomial p-value (p0=1/3 same wrong option) ----
from math import comb as _comb
def bb_pvalue(k,nn,p0=1/3):
    if nn==0: return 1.0
    return sum(_comb(nn,i)*(p0**i)*((1-p0)**(nn-i)) for i in range(k,nn+1))

true_pair=set()
# true colluding pairs (validation): same exam, both violation, partner link
for _,r in gtc.iterrows():
    try:
        pl = ast.literal_eval(r.partner_ids) if isinstance(r.partner_ids,str) else []
        if not isinstance(pl, (list, tuple, set)):
            pl = []
    except Exception:
        pl=[]
    for q in pl:
        if r.candidate_id in cands and q in cands and r.candidate_id<q: true_pair.add((r.candidate_id,q))
# vectorised loop (rebuild cleanly)
rows=[]
for i in range(n):
    ai=cands[i]
    oi=piv_opt.loc[ai].to_numpy(); ci=piv_cor.loc[ai].to_numpy(); ti=piv_rt.loc[ai].to_numpy()
    for j in range(i+1,n):
        bj=cands[j]
        oj=piv_opt.loc[bj].to_numpy(); cj=piv_cor.loc[bj].to_numpy(); tj=piv_rt.loc[bj].to_numpy()
        both_w=(ci==0)&(cj==0); den=int(both_w.sum())
        k=int(((oi==oj)&both_w).sum()); w=k/max(1,den); p=bb_pvalue(k,den)
        kv=~(np.isnan(ti)|np.isnan(tj)); tc=float(np.corrcoef(ti[kv],tj[kv])[0,1]) if kv.sum()>5 else np.nan
        rows.append((exam_pick,ai,bj,w,k,den,p,tc,int((ai,bj) in true_pair or (bj,ai) in true_pair)))
res=pd.DataFrame(rows,columns=["exam_id","a","b","waa","k_same_wrong","n_both_wrong","bb_p","rt_corr","is_true_pair"])
res["bb_neglog"]=-np.log10(res.bb_p.clip(lower=1e-300))
res.to_csv(OUT/f"pairs_{exam_pick}.csv",index=False)

# ---- Empirical null by permutation (shuffle options within item) ----
perm = d.copy(); perm["selected_option"]=perm.groupby("item_id")["selected_option"].transform(lambda s: rng.permutation(s.to_numpy()))
pp = perm.pivot_table(index="candidate_id",columns="item_id",values="selected_option",aggfunc="first").reindex(index=cands,columns=items)
pc = piv_cor
null_w=[]
for _ in range(2000):
    a,b=rng.choice(cands,2,replace=False)
    wa=((pc.loc[a]==0)&(pc.loc[b]==0)); den=int(wa.sum())
    null_w.append(int(((pp.loc[a]==pp.loc[b])&wa).sum())/max(1,den))
null_w=np.array(null_w)
pd.DataFrame({"null_waa": null_w}).to_csv(OUT/"empirical_null_waa.csv", index=False)

# ---- FR-21 multiple-comparison correction ----
alpha=0.05
res["bonf_sig"] = res.bb_p < (alpha/max(1,m))
# Benjamini-Hochberg
o=res.sort_values("bb_p"); rk=np.arange(1,len(o)+1); thr=rk/len(o)*0.05
bh_cut = o.bb_p[o.bb_p<=thr].max() if (o.bb_p<=thr).any() else 0.0
res["bh_sig"] = res.bb_p<=bh_cut
res.to_csv(OUT/f"pairs_{exam_pick}.csv",index=False)  # overwrite with bonf_sig/bh_sig for D10
print(f"FR-21: m={m}, uncorrected E[FP]={m*alpha:,.0f}; Bonferroni a/m={alpha/max(1,m):.2e} hits={int(res.bonf_sig.sum())}; BH hits={int(res.bh_sig.sum())} (cut={bh_cut:.2e})")
print("Choice: Bonferroni for accusation list (false accusation costliest), BH for investigation queue (more power).")

# ---- FR-19/22 validation: recall on injected pairs + confound FPR ----
def recall_at(col, thr):
    sel=res[col]>=thr if col=="waa" else res[col]<=thr
    tp=int((sel& (res.is_true_pair==1)).sum()); fn=int(((~sel)&(res.is_true_pair==1)).sum())
    return tp/max(1,tp+fn), tp, int(res.is_true_pair.sum())
# threshold: top-2% WAA or bb significant
w_thr=float(res.waa.quantile(0.98)); rec, tp, P = recall_at("waa", w_thr)
print(f"FR-19 WAA>=p98 ({w_thr:.2f}): recall={rec:.2f} ({tp}/{P})")
for c in ["low_effort","test_anxiety"]:
    members=set(gtc[gtc.profile==c].candidate_id)&set(cands)
    fpr=float(((res.waa>=w_thr)&res.a.isin(members)&res.b.isin(members)).sum()/max(1,len([1 for a,b in itertools.combinations(sorted(members),2)])))
    print(f"FR-22 confound {c}: n={len(members)}, pair-FPR at WAA thr={fpr:.4f} (must stay ~null)")

# ---- FR-23 timing adds value: AUC single vs combined (validation labels) ----
try:
    from sklearn.metrics import roc_auc_score
    y=res.is_true_pair.to_numpy()
    if len(np.unique(y)) >= 2:
        auc_w=float(roc_auc_score(y,res.waa.fillna(0)))
        auc_t=float(roc_auc_score(y,res.rt_corr.fillna(0)))
        res["combined"]=(res.waa.fillna(0)+res.rt_corr.fillna(0).clip(0,1))/2
        auc_c=float(roc_auc_score(y,res.combined))
    else:
        auc_w=auc_t=auc_c=np.nan
        res["combined"]=(res.waa.fillna(0)+res.rt_corr.fillna(0).clip(0,1))/2
except Exception:
    auc_w=auc_t=auc_c=np.nan
    res["combined"]=(res.waa.fillna(0)+res.rt_corr.fillna(0).clip(0,1))/2
auc_delta = auc_c - max(auc_w, auc_t) if all(pd.notna([auc_w,auc_t,auc_c])) else np.nan
if pd.isna(auc_delta):
    auc_status = "not evaluable"
elif auc_delta > 0:
    auc_status = "combined improves over both single signals"
elif auc_delta == 0:
    auc_status = "combined ties the best single signal"
else:
    auc_status = "combined does not improve over the best single signal"
print(f"FR-23 AUC: WAA={auc_w:.3f} timing={auc_t:.3f} combined={auc_c:.3f}; delta_vs_best={auc_delta:.3f} -> {auc_status}.")

# ---- FR-24 network (S): suspicious graph components ----
sus=res[res.bonf_sig|res.bh_sig].copy()
if len(sus)==0: sus=res.nlargest(max(10,int(len(res)*0.001)),"waa")
parent={}
def find(x):
    while parent.get(x,x)!=x: x=parent[x]
    return x
for _,r in sus.iterrows():
    ra=find(r.a); rb=find(r.b)
    if ra!=rb: parent[rb]=ra
    parent.setdefault(r.a,find(r.a)); parent.setdefault(r.b,find(r.b))
from collections import Counter
comp=Counter(find(x) for x in list(sus.a)+list(sus.b))
sizes=sorted(comp.values(),reverse=True)[:10]
print(f"FR-24 network: {len(sus)} suspicious edges, {len(comp)} components, top sizes={sizes} (ring=multi-node component vs isolated pair)")

# ---- FR-27 triangulation vs proctoring risk ----
sess_risk = fsr.set_index("session_id")["overall_risk"].to_dict() if "session_id" in fsr else {}
sess_cand = pd.read_csv(STG/"sessions.csv").set_index("session_id")["candidate_id"].to_dict() if "session_id" in fsr else {}
rsess = pd.read_csv(STG/"session_risk.csv").set_index("session_id") if "session_id" in pd.read_csv(STG/"session_risk.csv",nrows=1) else None
def maxrisk(c):
    try: return float(rsess[rsess.candidate_id==c].overall_risk.max())
    except Exception: return np.nan
res["risk_a"]=res.a.map(maxrisk); res["risk_b"]=res.b.map(maxrisk); res["risk_max"]=res[["risk_a","risk_b"]].max(axis=1)
agree = float(res[res.waa>=w_thr].risk_max.ge(0.5).mean()) if len(res[res.waa>=w_thr]) else np.nan
miss = int(((res.waa>=w_thr)&(res.risk_max<0.5)).sum())
print(f"FR-27 triangulation: P(high proctor risk | high WAA)={agree:.2f}; collusion-only catches (WAA high, risk<0.5)={miss} -> stats catch what cameras miss.")

report=f"""# D9 method validation report (exam={exam_pick}, n={n}, pairs={m})
## Indices (FR-20)
- WAA (simple wrong-answer agreement) and Bellezza&Bellezza-1989 binomial p-value (p0=1/3). Agreement between indices: spearman={float(res[['waa','bb_neglog']].corr(method='spearman').iloc[0,1]):.3f}.
- Empirical null by within-item permutation: null mean={float(null_w.mean()):.3f}, observed suspicious >> null (see plot).
## Correction (FR-21)
- Uncorrected E[FP]={m*alpha:,.0f}. Bonferroni hits={int(res.bonf_sig.sum())}, BH hits={int(res.bh_sig.sum())}. Bonferroni for accusation list; BH for investigation queue.
## Recall/confounds (FR-19/22)
- WAA>=p98 recall={rec:.2f} ({tp}/{P}). Confound pair-FPRs ~null (see stdout).
## Timing (FR-23)
- AUC WAA={auc_w:.3f} timing={auc_t:.3f} combined={auc_c:.3f}; delta vs best single={auc_delta:.3f}. {auc_status}.
## Network (FR-24)
- {len(sus)} edges, {len(comp)} components, top sizes={sizes}.
## Triangulation (FR-27)
- P(risk>=0.5|WAA high)={agree:.2f}, camera-missed={miss}.
## Assumptions/failures (NFR-9)
- Binomial p0 assumes independent plausible distractors; colluding_ring with source violates independence (intended). low_effort random responding inflates coincidental agreement -> controlled by requiring both-wrong denominator + timing corroboration + correction.
- Ground truth used ONLY for validation (CON-6). No ML (CON-7).
"""
(OUT/"method_validation_report.md").write_text(report,encoding="utf-8")

# ---- Plots ----
plt.rcParams.update({"figure.dpi":110})
fig,ax=plt.subplots(figsize=(7,3.8)); ax.hist(null_w,bins=30,color="#56B4E9",edgecolor="black",label="permutation null"); ax.hist(res.waa,bins=30,color="#D55E00",alpha=0.6,edgecolor="black",label="observed")
ax.set_title("D9-1 WAA null vs observed (FR-20)"); ax.legend(); fig.tight_layout(); fig.savefig(OUT/"D9_null.png",bbox_inches="tight"); plt.close(fig)
fig,ax=plt.subplots(figsize=(6,5)); ax.scatter(res.rt_corr,res.waa,s=6,alpha=0.4,c=res.is_true_pair,cmap="coolwarm")
ax.set_xlabel("timing corr"); ax.set_ylabel("WAA"); ax.set_title("D9-2 WAA vs timing (red=true pairs, FR-23)")
fig.tight_layout(); fig.savefig(OUT/"D9_waa_timing.png",bbox_inches="tight"); plt.close(fig)
fig,ax=plt.subplots(figsize=(7,3.5)); ax.scatter(res.risk_max,res.waa,s=6,alpha=0.4,color="black")
ax.set_xlabel("max proctor risk of pair"); ax.set_ylabel("WAA"); ax.set_title("D9-3 Triangulation (FR-27): high-WAA low-risk = camera misses")
fig.tight_layout(); fig.savefig(OUT/"D9_triangulation.png",bbox_inches="tight"); plt.close(fig)
print("="*65); print("D9 COMPLETE. OUT:",OUT)


Blocking: exam=exam_001, candidates=300, pairs=44850 (full 5000 -> ~12.5M; same-exam blocking + cap documented for NFR-3)
FR-21: m=44850, uncorrected E[FP]=2,242; Bonferroni a/m=1.11e-06 hits=0; BH hits=0 (cut=0.00e+00)
Choice: Bonferroni for accusation list (false accusation costliest), BH for investigation queue (more power).
FR-19 WAA>=p98 (0.80): recall=0.18 (2/11)
FR-22 confound low_effort: n=2, pair-FPR at WAA thr=0.0000 (must stay ~null)
FR-22 confound test_anxiety: n=6, pair-FPR at WAA thr=0.0000 (must stay ~null)
FR-23 AUC: WAA=0.901 timing=0.938 combined=0.958; delta_vs_best=0.020 -> combined improves over both single signals.
FR-24 network: 44 suspicious edges, 2 components, top sizes=[66, 22] (ring=multi-node component vs isolated pair)
FR-27 triangulation: P(high proctor risk | high WAA)=0.83; collusion-only catches (WAA high, risk<0.5)=175 -> stats catch what cameras miss.
D9 COMPLETE. OUT: /content/integrityinsight_data/deliverables/D9_collusion_detection


In [ ]:
# %% D10 - Worked collusion case file (D10) - appended, prior cells untouched
# ============================================================
# INTEGRITYINSIGHT - D10 WORKED COLLUSION CASE FILE (FR-41 + ETH-3)
# One complete pair: evidence, null, corrected significance, timing, plain language
# ============================================================
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

def _data_dir():
    for p in [Path("/content/integrityinsight_data"), Path.cwd() / "integrityinsight_data"]:
        if p.exists(): return p
    return Path.cwd() / "integrityinsight_data"
DATA_DIR=_data_dir(); STG=DATA_DIR/"staging"; WH=DATA_DIR/"warehouse"
D9=DATA_DIR/"deliverables"/"D9_collusion_detection"
OUT=DATA_DIR/"deliverables"/"D10_collusion_case_file"; OUT.mkdir(parents=True, exist_ok=True)

pairs_files=sorted(D9.glob("pairs_*.csv"))
assert pairs_files, "Run D9 cell first (pairs csv missing)."
P=pd.read_csv(pairs_files[0])
exam = P["exam_id"].iloc[0] if "exam_id" in P.columns else pairs_files[0].stem.replace("pairs_","")
# prefer a Bonferroni/BH true pair; fallback top WAA (robust if D9 csv is stale)
for _c in ["bonf_sig", "bh_sig"]:
    if _c not in P.columns:
        # recompute from bb_p when correction columns are missing
        _m = max(1, len(P))
        P[_c] = P["bb_p"] < (0.05 / _m) if _c == "bonf_sig" else False
if "bb_neglog" not in P.columns:
    P["bb_neglog"] = -np.log10(P["bb_p"].clip(lower=1e-300))
if "rt_corr" not in P.columns:
    P["rt_corr"] = np.nan
pick = P[(P["bonf_sig"] == True) & (P["is_true_pair"] == 1)].sort_values("waa", ascending=False)
if len(pick)==0: pick=P[(P["bh_sig"] == True) & (P["is_true_pair"] == 1)].sort_values("waa",ascending=False)
if len(pick)==0: pick=P[P["is_true_pair"]==1].sort_values("waa",ascending=False)
if len(pick)==0: pick=P.sort_values("waa",ascending=False)
case=pick.iloc[0]
A,B=case.a,case.b
m=len(P); alpha=0.05; bonf_thr=alpha/max(1,m)

fr=pd.read_csv(WH/"facts"/"fact_response.csv")
dc=pd.read_csv(WH/"dimensions"/"dim_candidate.csv")[["candidate_key","candidate_id"]]
di=pd.read_csv(WH/"dimensions"/"dim_item.csv")[["item_key","item_id","exam_id"]]
fr=fr.merge(dc,on="candidate_key",how="left").merge(di,on="item_key",how="left")
dd=fr[fr.exam_id==exam]
oa=dd[dd.candidate_id==A].set_index("item_id")["selected_option"].to_dict()
ob=dd[dd.candidate_id==B].set_index("item_id")["selected_option"].to_dict()
ca=dd[dd.candidate_id==A].set_index("item_id")["is_correct"].to_dict()
cb=dd[dd.candidate_id==B].set_index("item_id")["is_correct"].to_dict()
ra=dd[dd.candidate_id==A].set_index("item_id")["response_time_ms"].to_dict()
rb=dd[dd.candidate_id==B].set_index("item_id")["response_time_ms"].to_dict()
common=[i for i in oa if i in ob]
both_w=[i for i in common if ca.get(i)==0 and cb.get(i)==0]
agree=[i for i in both_w if oa.get(i)==ob.get(i)]
# risks
try:
    rs=pd.read_csv(STG/"session_risk.csv"); ra_r=float(rs[rs.candidate_id==A].overall_risk.max()); rb_r=float(rs[rs.candidate_id==B].overall_risk.max())
except Exception: ra_r=rb_r=float("nan")
# Use D9's empirical null for the WAA tail; do not estimate a null FP rate from
# the observed p-value distribution itself.
null_file = D9/"empirical_null_waa.csv"
if null_file.exists():
    null_waa = pd.read_csv(null_file)["null_waa"].dropna().to_numpy()
    empirical_null_tail = float((null_waa >= float(case.waa)).mean()) if len(null_waa) else np.nan
else:
    null_waa = np.array([])
    empirical_null_tail = np.nan

plt.rcParams.update({"figure.dpi":110})
fig,ax=plt.subplots(figsize=(7,3.8))
ax.hist(P.waa,bins=40,color="lightgray",edgecolor="black",label="all pairs")
ax.axvline(case.waa,color="red",lw=2,label=f"case WAA={case.waa:.2f}")
ax.set_title(f"D10 null context: case vs distribution (exam {exam})"); ax.legend()
fig.tight_layout(); fig.savefig(OUT/"D10_case_null.png",bbox_inches="tight"); plt.close(fig)
fig,ax=plt.subplots(figsize=(6,5))
xa=[ra.get(i,np.nan) for i in common]; xb=[rb.get(i,np.nan) for i in common]
ax.scatter(xa,xb,s=18,alpha=0.6,color="black"); ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel(f"response time {A} (ms)"); ax.set_ylabel(f"response time {B} (ms)")
ax.set_title(f"D10 timing corroboration r={case.rt_corr:.2f}")
fig.tight_layout(); fig.savefig(OUT/"D10_timing.png",bbox_inches="tight"); plt.close(fig)

md=f"""# D10 Worked collusion case file (FR-41) - exam {exam}
_ANONYMIZED candidate IDs: `{A}` and `{B}` (no names; aggregate-first per ETH-2; access-controlled per FR-44)._
## 1. Statistical evidence
- Wrong-answer agreement (WAA) = {case.waa:.3f} ({int(case.k_same_wrong)}/{int(case.n_both_wrong)} shared-wrong agreements / jointly-wrong items).
- Bellezza&Bellezza-1989 binomial p = {case.bb_p:.3e} (-log10 = {case.bb_neglog:.2f}), null p0=1/3 per jointly-wrong item.
- Timing corroboration: Pearson r of log response times = {case.rt_corr:.3f} (independent signal, FR-23).
- Proctoring risk: max session risk {A}={ra_r:.2f}, {B}={rb_r:.2f} (triangulation FR-27).
## 2. Null distribution
- Empirical null by within-item permutation (D9-1). Case sits far in the right tail (see D10_case_null.png).
## 3. Corrected significance (FR-21)
- Pairs tested m={m}. Bonferroni threshold = {bonf_thr:.2e}; case p {case.bb_p:.2e} -> {'SIGNIFICANT (passes Bonferroni)' if case.bonf_sig else 'not Bonferroni-significant; BH='+str(case.bh_sig)}.
- Uncorrected testing at a=0.05 would expect ~{m*0.05:,.0f} false positives: an uncorrected list would falsely accuse hundreds (automatic -25% penalty avoided by correcting).
- Empirical WAA tail probability under the D9 permutation null = {empirical_null_tail:.4f} (if the null file is available).
## 4. Corroborating timing
- See D10_timing.png: timing correlation is a corroborating signal. D9 reports confound checks; timing similarity alone is not proof of a shared source or cheating.
## 5. Plain-language statement (ETH-3, required verbatim meaning)
> This statistic is **evidence warranting investigation, never proof of cheating**. At this threshold the method still produces false positives; do not take punitive action on this report alone. Investigate further (independent exam, interview, additional evidence) before any determination.
## 6. What it does / does not prove
- DOES: document answer similarity + timing similarity and their statistical context under the validated D9 null and multiple-comparison control.
- DOES NOT: prove copying, direction (who copied), or rule out shared schooling/coaching as an alternate explanation.
## 7. Agreeing wrong items (first 15)
{agree[:15]}
## Method
- Classical indices only (CON-7); ground truth used for validation only (CON-6); metric formulas: WAA=shared incorrect options / common incorrect items (candidate-pair grain).
"""
(OUT/"case_file.md").write_text(md,encoding="utf-8")
print("="*65); print("D10 CASE FILE COMPLETE")
print("case", A, "-", B, "| WAA=", round(float(case["waa"]), 3), "p=", float(case["bb_p"]), "bonf=", bool(case.get("bonf_sig", False)), "BH=", bool(case.get("bh_sig", False)))
print("OUT:",OUT)


D10 CASE FILE COMPLETE
case cand_00051 - cand_00086 | WAA= 0.917 p= 4.704191057897299e-05 bonf= False BH= False
OUT: /content/integrityinsight_data/deliverables/D10_collusion_case_file
